# What the 10 Hz Combine tracking can and cannot tell you

The 2027 Big Data Bowl asks how Combine movement translates to NFL play. This notebook checks the Combine
tracking before anything is built on it. The headline is in the figure below. From the 10-yard mark to the
finish, the tracking's 40 split agrees with the official one (r = 0.95 within position group). Over the
first 10 yards, it does not (r = 0.28).

**Companion notebook (October 2026):** [Who Beat Their Combine? 2023-25 Rookies](https://www.kaggle.com/code/dariushafshar/who-beat-their-combine-2023-25-rookies) uses the tracking metric this audit found trustworthy (top speed, not the start) and asks how much the Combine adds to draft position when predicting rookie playing time. Out of sample, it finds no detectable gain from the Combine once draft position is known.

In [ ]:
import os, json, time, warnings
from collections import deque

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
T0 = time.time()
SEARCH_ROOT = os.environ.get("SMOKE_ROOT", "/kaggle/input")
WORK = os.environ.get("SMOKE_WORK", "/kaggle/working")
os.makedirs(WORK, exist_ok=True)


def find_file(name, root=SEARCH_ROOT, max_depth=6):
    queue, hits = deque([(root, 0)]), []
    while queue:
        d, depth = queue.popleft()
        try:
            entries = sorted(os.scandir(d), key=lambda e: e.name)
        except OSError:
            continue
        for e in entries:
            try:
                if e.is_file() and e.name == name:
                    hits.append(e.path)
                elif e.is_dir() and depth < max_depth:
                    queue.append((e.path, depth + 1))
            except OSError:
                continue
    if not hits:
        raise FileNotFoundError(f"{name!r} not found under {root}")
    return sorted(hits, key=len)[0]


# one palette for every chart: a colour-blind-checked blue / orange pair, grey for context
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK, INK2, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#8a8984", "#e6e5e1", "#fcfcfb"
SEQ = LinearSegmentedColormap.from_list("seq", ["#f4f8fd", "#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "figure.dpi": 110, "font.size": 10, "font.family": "DejaVu Sans",
    "axes.edgecolor": MUTED, "axes.linewidth": 0.6, "axes.labelcolor": INK2,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.titlecolor": INK,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "xtick.color": INK2, "ytick.color": INK2, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "legend.frameon": False, "legend.fontsize": 9,
})

POS_ORDER = ["WR", "TE", "DB", "DL", "OL"]
POS_NAME = {"WR": "Wide receivers", "TE": "Tight ends", "DB": "Defensive backs",
            "DL": "Defensive line", "OL": "Offensive line"}
GROUP_NOUN = {"WR": "wide receivers", "TE": "tight ends", "DB": "defensive backs", "DL": "the defensive line",
              "OL": "the offensive line"}
RNG = np.random.default_rng(2027)


def fisher_ci(r, n, level=0.95):
    z, se = np.arctanh(r), 1 / np.sqrt(n - 3)
    q = stats.norm.ppf(0.5 + level / 2)
    return np.tanh(z - q * se), np.tanh(z + q * se)


def r_min(n, alpha=0.05, power=0.80):
    # smallest |r| a two-sided test detects with the given power (Fisher z approximation)
    return np.tanh((stats.norm.ppf(1 - alpha / 2) + stats.norm.ppf(power)) / np.sqrt(n - 3))


def r_crit(n, alpha=0.05, k=0):
    # |r| needed for p < alpha, two-sided, with k covariates partialled out
    df = np.asarray(n) - 2 - k
    t = stats.t.ppf(1 - alpha / 2, df)
    return t / np.sqrt(df + t * t)


def boot_r(x, y, B=4000, rng=RNG):
    x, y = np.asarray(x, float), np.asarray(y, float)
    idx = rng.integers(0, len(x), (B, len(x)))
    xs, ys = x[idx], y[idx]
    xs = xs - xs.mean(1, keepdims=True)
    ys = ys - ys.mean(1, keepdims=True)
    r = (xs * ys).sum(1) / np.sqrt((xs ** 2).sum(1) * (ys ** 2).sum(1))
    return np.quantile(r, [0.025, 0.975])


def note(ax, text, xy=(0.02, 0.97), **kw):
    ax.text(*xy, text, transform=ax.transAxes, va="top", ha="left", fontsize=9, color=INK2, **kw)

print(f"pandas {pd.__version__}, numpy {np.__version__}, scipy {stats.__name__.split('.')[0]}")

In [ ]:
# The figure: the title's two numbers, computed here from the Combine files alone. Section 2 rebuilds them with
# the full pipeline, and the last cell checks that the two agree.
c0 = pd.read_csv(find_file("combine_results.csv")).set_index("nfl_id")
f0 = pd.read_csv(find_file("combine_tracking.csv"))
f0["t"] = pd.to_datetime(f0["time"])
f0 = f0.sort_values(["event_id", "t"], kind="mergesort")
f0 = f0[(f0.entity_type == "PLAYER") & (f0.drill_type == "FORTY_YARD_DASH")]


def cross0(tt, v, level):
    # the time v first reaches level, interpolated between frames (section 2's xcross)
    i = np.flatnonzero(v >= level)
    if len(i) == 0 or i[0] == 0:
        return np.nan
    i = i[0]
    return tt[i - 1] + (level - v[i - 1]) / (v[i] - v[i - 1]) * (tt[i] - tt[i - 1])


rows0 = []
for e0, d0 in f0.groupby("event_id", sort=False):
    tt0 = (d0.t - d0.t.iloc[0]).dt.total_seconds().to_numpy()
    x0, s0 = d0.x.to_numpy(), d0.s.to_numpy()
    rows0.append((d0.nfl_id.iloc[0], cross0(tt0, x0, 50.5) - cross0(tt0, x0, 20.5),     # 10-yard to 40-yard mark
                  cross0(tt0, x0, 20.5) - cross0(tt0, s0, 2.0)))                         # 2 yd/s to the 10-yard mark
best0 = pd.DataFrame(rows0, columns=["nfl_id", "fly30", "first10"]).groupby("nfl_id").min()
o0 = c0[["combine_position", "forty", "ten_yd_split"]].join(best0)
o0["fly_off"] = o0.forty - o0.ten_yd_split
FIG0_R, cen0 = {}, {}
for key0, trk0, ref0 in (("fly", "fly30", "fly_off"), ("f10", "first10", "ten_yd_split")):
    d0 = o0[[trk0, ref0, "combine_position"]].dropna()
    c_ = d0[[trk0, ref0]] - d0.groupby("combine_position")[[trk0, ref0]].transform("mean")   # within position group
    FIG0_R[key0], cen0[key0] = np.corrcoef(c_[trk0], c_[ref0])[0, 1], c_
FIG0_N = (len(cen0["fly"]), len(cen0["f10"]))

fig, axs = plt.subplots(1, 2, figsize=(13, 5.4))
lim0 = max(np.abs(c_.to_numpy()).max() for c_ in cen0.values()) * 1.06
for ax, (key0, trk0, ref0, col0, ttl0) in zip(axs, (
        ("fly", "fly30", "fly_off", BLUE, "Speed: the 10-to-40-yard split"),
        ("f10", "first10", "ten_yd_split", ORANGE, "Start: the first 10 yards"))):
    c_ = cen0[key0]
    ax.plot([-lim0, lim0], [-lim0, lim0], color=MUTED, lw=1, zorder=1)
    ax.scatter(c_[ref0], c_[trk0], s=14, color=col0, alpha=0.65, edgecolor=SURFACE, linewidth=0.3, zorder=2)
    ax.set_xlim(-lim0, lim0); ax.set_ylim(-lim0, lim0); ax.set_aspect("equal")
    ax.text(0.04, 0.96, f"r = {FIG0_R[key0]:.2f}", transform=ax.transAxes, fontsize=22, fontweight="bold",
            va="top", color=INK)
    ax.text(0.04, 0.84, f"{len(c_)} prospects", transform=ax.transAxes, fontsize=10, va="top", color=INK2)
    ax.set_title(ttl0, fontsize=12)
    ax.set_xlabel("official time (s), minus his position group's mean")
axs[0].set_ylabel("tracking time (s), minus his position group's mean")
fig.suptitle("Combine tracking against the official 40 splits, within position group "
             "(one dot per prospect, best attempt; grey line: tracking = official)", x=0.01, ha="left", fontsize=11.5,
             fontweight="bold", color=INK)
fig.tight_layout()
plt.show()
print(f"within position group: 10-to-40-yard split r = {FIG0_R['fly']:.3f} (n {FIG0_N[0]}); "
      f"first 10 yards r = {FIG0_R['f10']:.3f} (n {FIG0_N[1]})")

## Test your own metric

Click **Copy & Edit**, change the settings and `my_metric` in the next cell, then **Run All**. Section 10
prints your metric's scorecard: whether it repeats, how many players it can be tested on, what correlation it
would need, and whether it says anything about NFL play beyond size and draft position. The default outcome is
rookie snap share from my public dataset, [BDB 2027 Public Enrichment](https://www.kaggle.com/datasets/dariushafshar/bdb-2027-public-enrichment), which adds each player's
season-by-season NFL snap counts, keyed on the competition's `nfl_id`.

In [ ]:
# ============ TEST YOUR OWN METRIC: Copy & Edit, change this cell, then Run All ============
# Section 10 prints the scorecard. This cell runs before every section, so keep your code inside my_metric
# (define any helper inside it too): a name set out here is seen by every later cell, and an error here stops the
# whole run, not just section 10.
MY_LABEL = "Tracking first 10 yards, from 2 yd/s"
MY_DRILLS = ["FORTY_YARD_DASH"]   # drill_name values; a name that does not exist stops section 10 with the full
                                  # list. A group that ran two or more of them gets a drill-to-drill repeat check
MY_BEST = "min"                   # one value per player from his attempts: "min", "max", "mean" or "median"
MY_OUTCOME = "share"              # "share": rookie snap share (section 9, from my public dataset); "top_speed":
                                  # in-game top speed; "main": each group's main outcome (section 4)
MY_TRIES = 1                      # metric-and-outcome pairs you have run through this card, this one included


def my_metric(f):
    # f is ONE drill attempt: the player's frames in time order, with sec = seconds from its first frame,
    # x, y, s (yd/s), a (yd/s^2), dir, drill_name, attempt and nfl_id. Do not sum dis (section 1).
    # Return one number, or np.nan when this attempt does not allow it. xcross comes from section 2.
    sec, x, s = f.sec.to_numpy(), f.x.to_numpy(), f.s.to_numpy()
    return xcross(sec, x, 20.5) - xcross(sec, s, 2.0)     # 2 yd/s to the 10-yard mark, as in section 2


# Or start from the per-player table of section 8: set MY_DRILLS = None and write my_metric(t) for a table t
# with one row per 40 attempt: that table's columns (the seven 40 metrics hold this attempt's value) and the
# official Combine results. Columns that do not change between attempts count as measured without error.
# def my_metric(t):
#     return t.combine_weight * 200 / t.full40 ** 4          # a speed score from the tracking 40

## What I asked, and what I found

**What I asked.** Before building a metric on the Combine tracking I wanted to know four things:

1. **Does the tracking reproduce the official times?** If it does not, a tracking metric is
   measuring something other than what scouts already have.
2. **Which tracking metrics repeat from one attempt to the next?** A metric that changes
   between two runs of the same drill can predict only weakly what a player does a season
   later.
3. **How many prospects have both Combine tracking and enough NFL snaps?** That is the real
   n behind every analysis.
4. **At that n, how large must a correlation be before it can be told from noise?**

**What I found.** Every comparison is within a Combine position group, because receivers
and linemen differ on almost everything.

1. **The tracking matches the official 40 after the start, not at the start.** The
   tracking's 10-to-40-yard split agrees with the official one (40 time minus 10-yard
   split) at r = 0.85 to 0.99 across the five groups (0.948 with
   each group centred on its own mean; 418 players). Its first 10 yards agree with
   the official 10-yard split at r = -0.12 to 0.49 (0.28 centred), and for
   wide receivers, tight ends and defensive backs the interval includes zero. Among wide receivers the official splits
   have an SD of 0.026 s; the tracking differs from them by an SD of
   0.096 s.
2. **Speed repeats; burst repeats mainly for linemen.** Comparing each player's first two
   40 attempts, top speed repeats at r = 0.89 to 0.98 in every
   group. Peak acceleration off the start repeats at r = 0.02 for wide receivers
   and 0.26 for defensive backs, against 0.65 on the defensive line.
3. **The real n is small.** Of 510 prospects, 243 have a tracked 40
   and at least 100 regular-season plays measuring their position's main outcome, from
   18 (tight ends) to 62 (defensive backs).
4. **The noise floor is high.** One pre-chosen test needs |r| of 0.25 to
   0.47 for p < 0.05. Searching all 18 Combine metrics against each
   group's outcomes, with every pair judged at its own n, a pair measured on the whole group
   needs |r| of 0.36 to 0.68 to beat the best that chance produces (95th
   percentile of the largest |z|, outcomes shuffled).
   28 real pairs clear that bar, in 3 of 5 groups, and
   8 of them involve a burst or agility metric, all for the defensive line.
5. **A worked example: defensive line burst and get-off.** A defensive lineman's 10-yard
   split correlates with his median NFL get-off time at r = 0.67 (95% CI
   0.50 to 0.81, n = 54). With body weight held fixed it falls to
   0.35 (0.09 to 0.56), and holding where he lines up fixed as
   well gives 0.34 (0.09 to 0.54): a lead, not a finding.
6. **The data page and the files disagree in places.** Preseason games are in the data
   (16% of the cohort's tracked plays), the game files hold 2.6 times the
   documented rows, and the Combine `dis` column is close to the squared step, not the step.
7. **New in version 2: braking in the position drills does not repeat from drill to drill.**
   Only 33 of 107 receivers have an official 3-cone, but 103 have a
   hard stop in at least two position drills. A receiver's peak braking into the stop in one
   drill barely predicts it in another (mean r = 0.07, 95% CI -0.07 to
   0.20), and the same holds for defensive backs (0.00, -0.12 to
   0.13). The official 3-cone and shuttle agree with each other at r = 0.62 to
   0.83. Pooling the three Combine years would have shown receivers' braking repeating at
   0.23, because the same drill reads up to 3.3 SDs differently from one
   Combine to the next. Against NFL separation, with size held fixed, the receivers' braking
   score reads r = 0.01 (-0.30 to 0.29, n = 59), which rules out only a
   large link.
8. **Speed shows up in rookie playing time, but most of it cannot be told apart from draft
   position.** Section 9 joins my public dataset, [BDB 2027 Public Enrichment](https://www.kaggle.com/datasets/dariushafshar/bdb-2027-public-enrichment), which adds
   each player's season-by-season NFL snaps. Faster prospects played more as rookies
   (r = 0.25, 95% CI 0.16 to 0.34, size and draft class held
   fixed). With draft position held fixed as well, it is 0.04 (-0.06 to
   0.14). Only the defensive line keeps a link, and it does not hold up with ranks.
9. **New in version 5: `player_play` holds 72% of these players' regular-season snaps, and
   beyond draft position, top speed does not show up in the special-teams snaps it leaves out.** The shares
   are measured against the public snap counts in my dataset (section 11). No `player_play` row is a punt, kickoff, field goal
   or extra point, yet special teams are 17% of these players' snaps. Receivers and tight ends appear
   only when they run a route, so their rows cover 58% and 40% of their snaps on offence.
   I predicted that top speed would show up in rookie special-teams snaps beyond draft position (r =
   +0.20 for receivers, tight ends and defensive backs). It does not: r = 0.01
   (-0.13 to 0.13), which rules out my prediction but not a small link.

Everything runs on CPU. Every number in this text is printed by the cell that computes
it. The notebook reads the competition files from the input mount and writes no raw rows
out: its output files are a table of per-player values I derived (section 8) and `headline.json`, the
numbers this text quotes. Sections 9 to 11
also read my public dataset of NFL snaps, [BDB 2027 Public Enrichment](https://www.kaggle.com/datasets/dariushafshar/bdb-2027-public-enrichment), which holds no
competition data.

**What changed in this version.** Section 11 is new. It counts the snaps in the competition's play
files against public snap counts. Then it tests, with definitions and a prediction written down in advance,
whether top speed shows up in the special-teams snaps the files do not hold. The figure under the title is new:
it shows the title's two numbers. The cell where you write your own metric moved from section 10 to the top, and
section 10 still prints its scorecard. Sections 1 to 10 print the same numbers as before. The intro gains finding
9, the limits list and the checklist each gain one line, and every place that uses my dataset,
[BDB 2027 Public Enrichment](https://www.kaggle.com/datasets/dariushafshar/bdb-2027-public-enrichment), links to it.

**What changed in version 4.** Section 10 was added. You write your own Combine metric in one cell,
and the notebook runs this audit's checks on it and prints a scorecard. It ships with the tracking's first 10
yards as a worked example. Sections 1 to 9 were unchanged. The intro gained a pointer to section 10, and the
limits list and the checklist each gained one line.

**What changed in version 3.** Section 9 was added. It joins my public dataset of NFL snaps,
[BDB 2027 Public Enrichment](https://www.kaggle.com/datasets/dariushafshar/bdb-2027-public-enrichment), to the Combine tracking and asks whether top speed shows up in rookie playing
time, before and after draft position is held fixed. Sections 1 to 8 were unchanged. The intro gained finding 8,
and the limits list and the checklist each gained one item.

**What changed in version 2.** Section 7 was added. It asks whether change of direction can be
measured in the Combine tracking, whether it repeats, and whether it shows up as NFL
separation. It also rechecks version 1's numbers with each Combine year centred. Section 8
was added too: a per-player CSV of the tracking metrics, each with a reliability flag, that you
can join to your own work. Sections 1 to 6 were unchanged and printed the same numbers as
version 1. The limits list and the checklist at the end each gained one item.

## Loading

The game-tracking files are about 2 GB. I read four of their twelve columns and keep one
row per player-play (that play's maximum speed); section 1 reads three more columns of the
2023 file to check `dis`.

In [ ]:
players = pd.read_csv(find_file("players.csv"))
combine = pd.read_csv(find_file("combine_results.csv")).set_index("nfl_id")
career = pd.read_csv(find_file("player_career_successes.csv")).set_index("nfl_id")
games = pd.read_csv(find_file("games.csv"))
ct = pd.read_csv(find_file("combine_tracking.csv"))
ct["t"] = pd.to_datetime(ct["time"])
ct = ct.sort_values(["event_id", "t"], kind="mergesort").reset_index(drop=True)
POS = combine["combine_position"]

GT_ROWS, per_play = {}, []
for yr in (2023, 2024, 2025):
    g = pd.read_csv(find_file(f"game_tracking_{yr}.csv"), usecols=["game_id", "play_id", "nfl_id", "s"],
                    engine="pyarrow")
    GT_ROWS[yr] = len(g)
    per_play.append(g.groupby(["nfl_id", "game_id", "play_id"], sort=False)["s"].max()
                    .rename("smax").reset_index())
    del g
plays = pd.concat(per_play, ignore_index=True).merge(
    games[["game_id", "season", "season_type"]], on="game_id", how="left")

pp = pd.read_csv(find_file("player_play.csv"), low_memory=False,
                 usecols=["game_id", "play_id", "nfl_id", "separation_at_pass_forward",
                          "player_get_off", "pressure_allowed", "lined_up_position", "dropback_duration"])
pp = pp.merge(games[["game_id", "season_type"]], on="game_id", how="left")
pp["pressure_allowed"] = pp["pressure_allowed"].astype("string").str.upper().map({"TRUE": 1.0, "FALSE": 0.0})

print(f"prospects {len(players)} | combine_results {len(combine)} | combine_tracking {len(ct):,} rows, "
      f"{ct.event_id.nunique():,} drill attempts")
print(f"game tracking: {sum(GT_ROWS.values()):,} frames -> {len(plays):,} player-plays | "
      f"player_play {len(pp):,} rows | games {len(games):,}")
print(f"games without a season match: {plays.season.isna().sum()}")
print(f"loaded in {time.time() - T0:.0f}s")

## 1. The data page and the files

These are worth knowing before any analysis. Each "files" value is computed below; the
"data page" column summarizes the competition's Data tab as I read it on 2026-10-07.

In [ ]:
DOC_GT_ROWS = {2023: 1_421_805, 2024: 3_042_118, 2025: 4_339_021}   # Data tab, read 2026-10-07
DOC_PP_ROWS = 316_338
rows = []
for yr in (2023, 2024, 2025):
    rows.append([f"game_tracking_{yr}.csv rows", f"{DOC_GT_ROWS[yr]:,}", f"{GT_ROWS[yr]:,}"])
GT_RATIO = sum(GT_ROWS.values()) / sum(DOC_GT_ROWS.values())
rows.append(["player_play.csv rows", f"{DOC_PP_ROWS:,}", f"{len(pp):,}"])

st = games.season_type.value_counts()
PRE_SHARE = (plays.season_type == "PRE").mean()
rows.append(["games.csv season_type", "REG, POST", ", ".join(f"{k} {st.get(k, 0)}" for k in ("REG", "PRE", "POST"))])
rows.append(["tracked rookie player-plays in preseason", "(not mentioned)", f"{PRE_SHARE:.1%}"])

ball = ct[ct.entity_type == "BALL"]
rows.append(["combine_tracking BALL rows with an nfl_id", "null for BALL", f"{ball.nfl_id.notna().sum():,} of {len(ball):,}"])

pl = ct[ct.entity_type == "PLAYER"].copy()
gb = pl.groupby("event_id", sort=False)
pl["step"] = np.hypot(gb.x.diff(), gb.y.diff())
ok = pl.step > 0.05
DIS_STEP = (pl.dis[ok] / pl.step[ok]).median()
DIS_STEP2 = (pl.dis[ok] / pl.step[ok] ** 2).median()
CORR_STEP = np.corrcoef(pl.dis[ok], pl.step[ok])[0, 1]
CORR_STEP2 = np.corrcoef(pl.dis[ok], pl.step[ok] ** 2)[0, 1]
rows.append(["Combine dis / frame step (median)", "1 (distance per frame)", f"{DIS_STEP:.2f}  (dis / step^2: {DIS_STEP2:.2f})"])

g23 = pd.read_csv(find_file("game_tracking_2023.csv"), engine="pyarrow",
                  usecols=["game_id", "play_id", "nfl_id", "time", "x", "y", "dis"])
g23 = g23.sort_values(["game_id", "play_id", "nfl_id", "time"], kind="mergesort")
k = g23.groupby(["game_id", "play_id", "nfl_id"], sort=False)
gstep = np.hypot(k.x.diff(), k.y.diff())
okg = gstep > 0.05
GAME_DIS_STEP = (g23.dis[okg] / gstep[okg]).median()
del g23, k, gstep
rows.append(["Game (2023) dis / frame step (median)", "1 (distance per frame)", f"{GAME_DIS_STEP:.2f}"])

forty = pl[pl.drill_type == "FORTY_YARD_DASH"]
f_ev = forty.groupby("event_id").agg(dis_sum=("dis", "sum"), x0=("x", "first"), x1=("x", "last"))
FORTY_DIS_SUM = f_ev.dis_sum.median()
FORTY_DX = (f_ev.x1 - f_ev.x0).median()
rows.append(["40-yard dash: sum of dis vs x moved (medians)", "equal", f"{FORTY_DIS_SUM:.1f} yd vs {FORTY_DX:.1f} yd"])

grid = pl.assign(on=(pl.x * 30 - np.round(pl.x * 30)).abs() < 1e-4).groupby("draft_year").on.mean()
rows.append(["Combine x on a 1/30-yard grid (2023 / 2024 / 2025 Combine)", "(not mentioned)",
             " / ".join(f"{grid[y]:.0%}" for y in (2023, 2024, 2025))])

gs = career.join(players.set_index("nfl_id").draft_year).groupby("draft_year").career_games_started.mean()
rows.append(["career_games_started, mean by draft class (2023 / 2024 / 2025)", "career totals",
             " / ".join(f"{gs[y]:.1f}" for y in (2023, 2024, 2025))])

audit = pd.DataFrame(rows, columns=["item", "data page", "files"])
print(audit.to_string(index=False))
print(f"\ngame rows vs documented: x{GT_RATIO:.2f} | corr(dis, step) {CORR_STEP:.3f}, corr(dis, step^2) {CORR_STEP2:.3f}")

What this means in practice:

* **Filter `season_type == "REG"`.** Preseason plays are 16% of the cohort's tracked
  plays, and the theme is regular-season performance.
* **In `combine_tracking`, filter `entity_type == "PLAYER"`, not just `nfl_id`.** The ball
  frames in receiver and tight-end drills carry an `nfl_id`.
* **Do not sum `dis` in the Combine file.** It behaves like the squared step: over a 40-yard
  dash it adds to 36.7 yards. Recompute distance from `x` and `y`. The game
  files' `dis` is the ordinary step (median ratio 1.00).
* **Career totals are exposure.** A 2023 pick has had three seasons to start games, a 2025
  pick one. Divide by seasons available, or compare within a draft class.

## 2. Does the tracking reproduce the official times?

The Combine file is clipped to each drill attempt, but it has no starting signal: no hand
lift, no gun. For the 40 I place the 10- and 40-yard marks at `x = 20.5` and `x = 50.5`
(attempts start at a median `x` of 10.67, and 98.9% of them reach 50.5)
and interpolate the crossing times between frames. The split between the two marks needs
no start at all. The first 10 yards do: I start the clock when speed first reaches
2 yd/s, and show below how much that choice matters.

For the shuttle and 3-cone I find the turns (speed minima) and the moment the player
recrosses the start line, with the same 2 yd/s start. Each player's best attempt is
compared with his official time, within his Combine position group.

In [ ]:
def xcross(tt, v, level):
    idx = np.flatnonzero(v >= level)
    if len(idx) == 0 or idx[0] == 0:
        return np.nan
    i = idx[0]
    return tt[i - 1] + (level - v[i - 1]) / (v[i] - v[i - 1]) * (tt[i] - tt[i - 1])


def onset(tt, s, v):
    return 0.0 if v == 0 else xcross(tt, s, v)


forty_frames = pl[pl.drill_type == "FORTY_YARD_DASH"].copy()
g40 = forty_frames.groupby("event_id", sort=False)
dsdt = (g40.s.shift(-1) - g40.s.shift(1)) / 0.2
okc = dsdt.notna() & forty_frames.step.notna()
print(f"sanity: corr(s, frame step / 0.1 s) {np.corrcoef(forty_frames.s[okc], forty_frames.step[okc] / 0.1)[0, 1]:.3f}, "
      f"corr(a, centred ds/dt) {np.corrcoef(forty_frames.a[okc], dsdt[okc])[0, 1]:.3f}")

ONSET = 2.0
ev40 = []
for e, d in g40:
    tt = (d.t - d.t.iloc[0]).dt.total_seconds().to_numpy()
    x, s, a = d.x.to_numpy(), d.s.to_numpy(), d.a.to_numpy()
    r = {"event_id": e, "nfl_id": d.nfl_id.iloc[0], "attempt": d.attempt.iloc[0], "x_moved": x[-1] - x[0],
         "x_start": x[0], "s_first": s[0]}
    for lo in (19.5, 20.0, 20.5):
        r[f"fly_{lo}"] = xcross(tt, x, lo + 30) - xcross(tt, x, lo)
    for v in (0, 1.0, 2.0, 3.0):
        r[f"first10_{v}"] = xcross(tt, x, 20.5) - onset(tt, s, v)
        r[f"full40_{v}"] = xcross(tt, x, 50.5) - onset(tt, s, v)
    r["split5_10"] = xcross(tt, x, 20.5) - xcross(tt, x, 15.5)
    r["acc2_8"] = xcross(tt, s, 8.0) - xcross(tt, s, 2.0)
    r["vmax"] = s.max()
    # peak acceleration off the start: 3-frame mean of a, from 0.3 s before speed first reaches 1 yd/s
    # to 1.5 s after; attempts whose clip starts at 2 yd/s or faster are left out
    i1 = np.flatnonzero(s >= 1.0)
    if s[0] < 2.0 and len(i1):
        a3 = pd.Series(a).rolling(3, center=True, min_periods=1).mean().to_numpy()
        w = (tt >= tt[i1[0]] - 0.3) & (tt <= tt[i1[0]] + 1.5)
        r["amax"] = a3[w].max()
    else:
        r["amax"] = np.nan
    ev40.append(r)
ev40 = pd.DataFrame(ev40)
ev40["fly30"], ev40["first10"], ev40["full40"] = ev40["fly_20.5"], ev40["first10_2.0"], ev40["full40_2.0"]
assert (ev40.x_moved > 0).all(), "a 40 attempt runs toward -x"
print(f"40-yard attempts {len(ev40)} by {ev40.nfl_id.nunique()} players; median start x {ev40.x_start.median():.2f}; "
      f"reaching x = 50.5: {ev40.fly30.notna().mean():.1%}; clips starting at 2 yd/s or faster: "
      f"{(ev40.s_first >= 2).sum()}")


def turns(s, i0, slow=1.5, fast=2.0):
    out, seen_fast = [], False
    for k in range(max(i0, 1), len(s) - 1):
        seen_fast |= s[k] > fast
        if seen_fast and s[k] < slow and s[k] <= s[k - 1] and s[k] <= s[k + 1]:
            out.append(k)
            seen_fast = False
    return out


def agility_time(d, drill):
    tt = (d.t - d.t.iloc[0]).dt.total_seconds().to_numpy()
    x, y, s = d.x.to_numpy(), d.y.to_numpy(), d.s.to_numpy()
    idx = np.flatnonzero(s >= ONSET)
    if len(idx) == 0 or idx[0] == 0:
        return np.nan, "no start"
    i0, t_on = idx[0], xcross(tt, s, ONSET)
    tk = turns(s, i0)
    P = np.c_[x, y]
    if drill == "SHORT_SHUTTLE":
        if len(tk) < 2:
            return np.nan, "turns"
        a_, b_ = P[tk[0]], P[tk[1]]
        L = np.linalg.norm(b_ - a_)
        if not 5 <= L <= 11:
            return np.nan, "geometry"
        u = (P - (a_ + b_) / 2) @ ((b_ - a_) / L)      # 0 = start line, between the two turns
        hit = np.flatnonzero(u[tk[1]:] <= 0)
        if len(hit) == 0:
            return np.nan, "no finish"
        i = tk[1] + hit[0]
    else:
        if len(tk) < 1:
            return np.nan, "turns"
        ab = P[tk[0]] - P[0]
        L = np.linalg.norm(ab)
        if not 1.5 <= L <= 6:
            return np.nan, "geometry"
        u = (P - P[0]) @ (ab / L)                       # 0 = start line
        path = np.r_[0, np.cumsum(np.hypot(np.diff(x), np.diff(y)))]
        path = path - path[i0]
        hit = np.flatnonzero((path > 20) & (u <= 0))
        if len(hit) == 0:
            return np.nan, "no finish"
        i = hit[0]
        if path[i] > 45:
            return np.nan, "geometry"
    if i == 0 or u[i - 1] <= 0:
        t_fin = tt[i]
    else:
        t_fin = tt[i - 1] + u[i - 1] / (u[i - 1] - u[i]) * (tt[i] - tt[i - 1])
    return t_fin - t_on, "ok"


agil = {}
for drill in ("SHORT_SHUTTLE", "THREE_CONE_DRILL"):
    rr = []
    for e, d in pl[pl.drill_type == drill].groupby("event_id", sort=False):
        t_, status = agility_time(d, drill)
        rr.append({"nfl_id": d.nfl_id.iloc[0], "secs": t_, "status": status})
    agil[drill] = pd.DataFrame(rr)
    print(f"{drill}: {len(rr)} attempts, timed {np.mean([q['status'] == 'ok' for q in rr]):.1%}, "
          f"not timed: {agil[drill].status.value_counts().drop('ok', errors='ignore').to_dict()}")

best40 = ev40.groupby("nfl_id").agg(
    {**{c: "min" for c in ev40.columns if c.startswith(("fly", "first10", "full40", "split", "acc"))},
     "vmax": "max", "amax": "max"})
off = combine[["forty", "ten_yd_split", "short_shuttle", "three_cone"]].copy()
off["fly_off"] = off.forty - off.ten_yd_split
off["shuttle_trk"] = agil["SHORT_SHUTTLE"].groupby("nfl_id")["secs"].min()
off["cone_trk"] = agil["THREE_CONE_DRILL"].groupby("nfl_id")["secs"].min()
off = off.join(best40).join(POS)


def agree(trk, ref, label=""):
    d = off[[trk, ref, "combine_position"]].dropna()
    c = d[[trk, ref]] - d.groupby("combine_position")[[trk, ref]].transform("mean")
    rw = np.corrcoef(c[trk], c[ref])[0, 1]
    lo, hi = fisher_ci(rw, len(d) - d.combine_position.nunique() + 1)
    out = {"comparison": label, "n": len(d), "r_within": rw, "lo": lo, "hi": hi,
           "r_pooled": np.corrcoef(d[trk], d[ref])[0, 1]}
    for p in POS_ORDER:
        g = d[d.combine_position == p]
        out[p] = np.corrcoef(g[trk], g[ref])[0, 1] if len(g) >= 10 else np.nan
        out[f"n_{p}"] = len(g)
        out[f"lo_{p}"], out[f"hi_{p}"] = fisher_ci(out[p], len(g)) if len(g) >= 10 else (np.nan, np.nan)
        out[f"sdref_{p}"], out[f"sddiff_{p}"] = g[ref].std(), (g[trk] - g[ref]).std()
    return out


AG = pd.DataFrame([
    agree("fly30", "fly_off", "40: 10 to 40 yd"),
    agree("full40", "forty", "40: full, from 2 yd/s"),
    agree("first10", "ten_yd_split", "40: first 10 yd, from 2 yd/s"),
    agree("shuttle_trk", "short_shuttle", "Short shuttle, from 2 yd/s"),
    agree("cone_trk", "three_cone", "3-cone, from 2 yd/s"),
]).set_index("comparison")
print("\nTracking vs official, each player's best attempt; r within position group")
print(AG[["n", "r_within", "lo", "hi", "r_pooled"] + POS_ORDER].round(3).to_string())
print("\nplayers per cell:")
print(AG[[f"n_{p}" for p in POS_ORDER]].to_string())
print("\nSD of the official time within group vs SD of (tracking - official), seconds:")
for lab in AG.index[[0, 2]]:
    print(f"  {lab}: " + " | ".join(f"{p} {AG.loc[lab, f'sdref_{p}']:.3f} vs {AG.loc[lab, f'sddiff_{p}']:.3f}"
                                   for p in POS_ORDER))
F10_ZERO = [p for p in POS_ORDER if AG.loc[AG.index[2], f"lo_{p}"] < 0 < AG.loc[AG.index[2], f"hi_{p}"]]
print(f"first 10 yd: groups whose 95% interval includes zero: {F10_ZERO}")
on01 = ((combine.ten_yd_split * 100 - np.round(combine.ten_yd_split * 100)).abs() < 1e-6).mean()
print(f"official 10-yard splits recorded to 0.01 s: {on01:.0%}")

print("\nWhere the 10- and 40-yard marks sit (10 to 40 yd split, r within group):")
for lo in (19.5, 20.0, 20.5):
    a_ = agree(f"fly_{lo}", "fly_off")
    print(f"  marks at x = {lo} / {lo + 30}: n {a_['n']}, r {a_['r_within']:.3f}")
print("\nWhen the clock starts (speed threshold; 0 = first frame of the attempt), r within group:")
ONS = []
for v in (0, 1.0, 2.0, 3.0):
    a10, a40 = agree(f"first10_{v}", "ten_yd_split"), agree(f"full40_{v}", "forty")
    ONS.append({"start": v, "n": a10["n"], "r_first10": a10["r_within"], "n40": a40["n"], "r_full40": a40["r_within"]})
    print(f"  start at {v:.1f} yd/s: first 10 yd r {a10['r_within']:.3f} (n {a10['n']}) | "
          f"full 40 r {a40['r_within']:.3f} (n {a40['n']})")
ONS = pd.DataFrame(ONS)

In [ ]:
fig, axs = plt.subplots(2, len(POS_ORDER), figsize=(15, 6.6))
for row, (trk, ref, lab, unit) in enumerate([("fly30", "fly_off", AG.index[0], "10 to 40 yd"),
                                              ("first10", "ten_yd_split", AG.index[2], "first 10 yd")]):
    for col, p in enumerate(POS_ORDER):
        ax = axs[row, col]
        d = off.loc[off.combine_position == p, [trk, ref]].dropna()
        lo_, hi_ = d.values.min() - 0.03, d.values.max() + 0.03
        ax.plot([lo_, hi_], [lo_, hi_], color=MUTED, lw=1, zorder=1)
        ax.scatter(d[ref], d[trk], s=14, color=BLUE, alpha=0.75, edgecolor=SURFACE, linewidth=0.4, zorder=2)
        ax.set_xlim(lo_, hi_); ax.set_ylim(lo_, hi_); ax.set_aspect("equal")
        ax.tick_params(labelsize=8)
        note(ax, f"r = {AG.loc[lab, p]:.2f}\nn = {AG.loc[lab, f'n_{p}']}")
        if row == 0:
            ax.set_title(POS_NAME[p])
        if col == 0:
            ax.set_ylabel(f"{unit}: tracking (s)")
        ax.set_xlabel("official (s)", fontsize=9)
fig.suptitle("Figure 1. Tracking against official times within each position group "
             "(top: 10 to 40 yd split; bottom: first 10 yd; grey line = same time)",
             x=0.01, ha="left", fontsize=12, fontweight="bold", color=INK)
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(11, 3.9))
cols = POS_ORDER + ["r_within", "r_pooled"]
vals = AG[cols].to_numpy(float)
ax.imshow(np.clip(vals, 0, 1), cmap=SEQ, vmin=0, vmax=1, aspect="auto")
for i in range(vals.shape[0]):
    for j in range(vals.shape[1]):
        n_ = AG.iloc[i][f"n_{cols[j]}"] if j < len(POS_ORDER) else AG.iloc[i]["n"]
        ax.text(j, i - 0.12, f"{vals[i, j]:.2f}", ha="center", va="center", fontsize=10,
                color="white" if vals[i, j] > 0.6 else INK)
        ax.text(j, i + 0.25, f"n {int(n_)}", ha="center", va="center", fontsize=7.5,
                color="white" if vals[i, j] > 0.6 else INK2)
ax.set_xticks(range(len(cols)))
ax.set_xticklabels(POS_ORDER + ["All groups,\ncentred", "All, pooled\n(contrast only)"])
ax.set_yticks(range(len(AG))); ax.set_yticklabels(AG.index)
ax.axvline(len(POS_ORDER) - 0.5, color=SURFACE, lw=4)
ax.grid(False); ax.tick_params(length=0)
for s_ in ax.spines.values():
    s_.set_visible(False)
ax.set_title("Figure 2. Agreement with the official time, Pearson r (negative values shown as 0 in colour)",
             fontsize=12)
fig.tight_layout()
plt.show()

After the first 10 yards the tracking and the stopwatch agree within every group (r =
0.85 to 0.99), and moving the marks to `x` = 19.5 or 20 changes nothing that
matters. The first 10 yards are another story. For wide receivers the official 10-yard
splits have an SD of 0.026 s across players, while the tracking's version differs from
them by an SD of 0.096 s, so the tracking cannot reproduce the official ranking of
receivers' starts. The start
rule does not rescue it: across the four rules the within-group r for the first 10 yards runs
from 0.19 to 0.28. Pooling all positions would have reported 0.62
for the same comparison, which is why every number here is within group.

The shuttle and 3-cone agree at r = 0.65 and 0.66 (groups centred), on the
79% and 76% of attempts my generic turn-finder could time; the direction of
any bias from the untimed attempts is unknown, and per-group n is small. The centred value
hides that the 3-cone shows no agreement for wide receivers and tight ends (r =
0.09 and -0.04, though on only 15 and 14 players). One caveat on the
40: if an official 10-yard split and 40 come from different runs, some of the first-10
disagreement is that.

## 3. Which tracking metrics repeat?

375 players ran at least two tracked 40s. If a metric is a trait of the player, his
first and second attempts should agree. I correlate attempt 1 with attempt 2 within each
Combine position group. The pooled column is shown for contrast only: pooling mixes the
differences between positions into the correlation.

In [ ]:
METRICS40 = {"vmax": "Top speed", "fly30": "10 to 40 yd split", "split5_10": "5 to 10 yd split",
             "acc2_8": "Time from 2 to 8 yd/s", "first10": "First 10 yd (from 2 yd/s)",
             "amax": "Peak acceleration off the start"}
ev40["pos"] = ev40.nfl_id.map(POS)
pairs = ev40.sort_values(["nfl_id", "attempt"]).groupby("nfl_id").head(2)
pairs = pairs[pairs.groupby("nfl_id").attempt.transform("size") == 2].copy()
pairs["k"] = pairs.groupby("nfl_id").cumcount()
N_RETEST = pairs.nfl_id.nunique()
REL, RELN, RELCI = {}, {}, {}
for m in METRICS40:
    w = pairs.pivot(index="nfl_id", columns="k", values=m).join(POS).dropna()
    REL[m] = {p: np.corrcoef(g[0], g[1])[0, 1] for p, g in w.groupby("combine_position")}
    REL[m]["All (pooled)"] = np.corrcoef(w[0], w[1])[0, 1]
    RELN[m] = w.groupby("combine_position").size().to_dict()
REL = pd.DataFrame(REL).T.loc[list(METRICS40), POS_ORDER + ["All (pooled)"]]
RELN = pd.DataFrame(RELN).T.loc[list(METRICS40), POS_ORDER]
print(f"players with two tracked 40 attempts: {N_RETEST}")
print("\nattempt 1 vs attempt 2, Pearson r")
print(REL.rename(index=METRICS40).round(3).to_string())
print("\npairs per cell:")
print(RELN.rename(index=METRICS40).to_string())
HW = [(lambda lo_hi: (lo_hi[1] - lo_hi[0]) / 2)(fisher_ci(REL.loc[m, p], RELN.loc[m, p]))
      for m in METRICS40 for p in POS_ORDER]
print(f"95% interval half-widths across the cells: {min(HW):.2f} to {max(HW):.2f}")
BURST = ["split5_10", "acc2_8", "first10", "amax"]
BR = REL.loc[BURST, POS_ORDER].agg(["min", "max"])
print("burst metrics (last four rows), range by group:\n" + BR.round(2).to_string())
w = pairs.pivot(index="nfl_id", columns="k", values="acc2_8").join(POS).dropna()
SPREAD = w.groupby("combine_position").apply(lambda g: pd.Series({
    "noise_sd": (g[0] - g[1]).std() / np.sqrt(2), "between_sd": ((g[0] + g[1]) / 2).std()})).reindex(POS_ORDER)
print("time from 2 to 8 yd/s: attempt-to-attempt noise SD vs between-player SD (s)")
print(SPREAD.round(3).to_string())

fig, ax = plt.subplots(figsize=(11, 4.3))
vals = REL.to_numpy(float)
ax.imshow(np.clip(vals, 0, 1), cmap=SEQ, vmin=0, vmax=1, aspect="auto")
for i in range(vals.shape[0]):
    for j in range(vals.shape[1]):
        ax.text(j, i, f"{vals[i, j]:.2f}", ha="center", va="center", fontsize=10,
                color="white" if vals[i, j] > 0.6 else INK)
ax.set_xticks(range(vals.shape[1]))
ax.set_xticklabels([f"{p}\n(n {RELN.loc[:, p].min()} to {RELN.loc[:, p].max()})" for p in POS_ORDER]
                   + ["All, pooled\n(contrast only)"])
ax.set_yticks(range(vals.shape[0])); ax.set_yticklabels([METRICS40[m] for m in REL.index])
ax.axvline(len(POS_ORDER) - 0.5, color=SURFACE, lw=4)
ax.grid(False); ax.tick_params(length=0)
for s_ in ax.spines.values():
    s_.set_visible(False)
ax.set_title("Figure 3. Attempt 1 vs attempt 2 of the 40, Pearson r within group", fontsize=12)
fig.tight_layout()
plt.show()

Top speed and the 10 to 40 yard split are traits: r = 0.89 or higher for top
speed in every group. The four burst metrics (the last four rows) split by group. For the
linemen they repeat best (defensive line 0.65 to 0.92, offensive line
0.51 to 0.84). For wide receivers they barely repeat
(0.02 to 0.34; peak acceleration 0.02), and for defensive
backs only partly (0.20 to 0.63). Part of the linemen's advantage is
spread, not precision: for the time from 2 to 8 yd/s, the attempt-to-attempt noise is
similar for receivers and offensive linemen (SD 0.12 vs 0.13 s), but the
offensive linemen differ from each other far more (SD 0.29 vs 0.10 s).
A metric that does not repeat limits what it can correlate with (section 5 puts a number
on it). With 31 to 93 pairs per cell, each r here is itself
uncertain (95% intervals of plus or minus 0.01 to 0.33), so read the pattern, not
the second decimal.

## 4. How many prospects can actually be analysed?

An analysis that links a Combine metric to NFL play needs both. I count regular-season
plays only, and require at least 100 plays on which the position's main outcome is
measured: separation at the throw for receivers and tight ends, get-off for the
defensive line, pressure allowed for the offensive line (recorded only on plays with a
dropback; the cell below prints the check), and (lacking a coverage outcome for defensive backs in `player_play`) any tracked
play for in-game speed.

In [ ]:
MIN_PLAYS = 100
reg_pp = pp[pp.season_type == "REG"]
reg_plays = plays[plays.season_type == "REG"]
OUTCOME_COL = {"WR": "separation_at_pass_forward", "TE": "separation_at_pass_forward",
               "DL": "player_get_off", "OL": "pressure_allowed"}
n_reg = reg_plays.groupby("nfl_id").size()
ol_pp = pp[pp.nfl_id.map(POS) == "OL"]
PA_DROP = ol_pp.pressure_allowed.notna()[ol_pp.dropback_duration.notna()].mean()
PA_NODROP = ol_pp.pressure_allowed.notna()[ol_pp.dropback_duration.isna()].mean()
print(f"offensive line: pressure_allowed recorded on {PA_DROP:.0%} of plays with a dropback time, "
      f"{PA_NODROP:.0%} of plays without")
n_out = {p: reg_pp.dropna(subset=[c]).groupby("nfl_id").size() for p, c in OUTCOME_COL.items()}
has40 = set(ev40.dropna(subset=["fly30"]).nfl_id)

cov = []
for p in POS_ORDER:
    ids = POS.index[POS == p]
    k_out = n_out[p].reindex(ids).fillna(0) if p in n_out else n_reg.reindex(ids).fillna(0)
    t40 = np.array([i in has40 for i in ids])
    anyreg = (n_reg.reindex(ids).fillna(0) > 0).to_numpy()
    enough_out = (k_out >= MIN_PLAYS).to_numpy()
    stage = {"Prospects": len(ids),
             "... with a tracked 40": int(t40.sum()),
             "... and a regular-season play": int((t40 & anyreg).sum()),
             f"... and {MIN_PLAYS}+ plays with the outcome": int((t40 & enough_out).sum())}
    cov.append({"pos": p, **stage})
COV = pd.DataFrame(cov).set_index("pos")
print(COV.to_string())
ANALYSIS_COL = COV.columns[-1]
N_ANALYSIS = int(COV[ANALYSIS_COL].sum())
print(f"\nanalysis n, all positions: {N_ANALYSIS} of {len(combine)} prospects")

dy = players.set_index("nfl_id").draft_year
share = (n_reg.reindex(players.nfl_id).fillna(0) >= MIN_PLAYS).groupby(dy.reindex(players.nfl_id).values).mean()
print("share with 100+ regular-season tracked plays, by draft class:",
      {int(k): f"{v:.0%}" for k, v in share.items()})

fig, axs = plt.subplots(1, len(POS_ORDER), figsize=(14, 3.6), sharey=True)
stages = list(COV.columns)
ramp = ["#86b6ef", "#5598e7", "#256abf", "#104281"]
for ax, p in zip(axs, POS_ORDER):
    v = COV.loc[p].to_numpy()
    yy = np.arange(len(stages))[::-1]
    ax.barh(yy, v, color=ramp, height=0.72, edgecolor=SURFACE, linewidth=2)
    for y_, val in zip(yy, v):
        ax.text(val + 2, y_, f"{val}", va="center", fontsize=9, color=INK)
    ax.set_xlim(0, COV.values.max() * 1.25)
    ax.set_title(f"{POS_NAME[p]}")
    ax.grid(axis="y", visible=False)
    ax.set_axisbelow(True)
axs[0].set_yticks(np.arange(len(stages))[::-1]); axs[0].set_yticklabels(stages)
fig.suptitle("Figure 4. From prospects to analysable players, by Combine position group", x=0.01, ha="left",
             fontsize=12, fontweight="bold", color=INK)
fig.tight_layout()
plt.show()

243 of 510 prospects survive both requirements. The 2025 class has had
one season, so only 50% of it reaches 100 regular-season tracked plays, against
70% of the 2023 class.

## 5. How big must a correlation be?

For each position group I take the analysis set from section 4 and the outcomes a typical
entry would try: the main outcome above plus in-game top speed (the 95th percentile of a
player's per-play maximum speed, regular season). The Combine side is 18
metrics: the official results (times, jumps, bench and body measurements) and the six
tracking metrics of section 3. Drills that many prospects skipped (shuttle, 3-cone, bench)
leave some pairs with far fewer players than others, so every pair is judged at its own n
through the Fisher statistic z = atanh(r) x sqrt(n - 3). Three thresholds:

* **One test:** the |r| needed for p < 0.05 when exactly one pair was chosen in advance.
* **Bonferroni:** the same after correcting for every metric-outcome pair tried.
* **Best pair by chance:** shuffle the outcomes across players 2,000 times and record the
  largest |z| among all pairs; its 95th percentile is the bar. This keeps the real
  correlations among the Combine metrics.

All three are shown in |r| at the n of the pair being judged. A fourth number matters for
planning: the true |r| a single test finds 80% of the time. The outcomes' own repeatability
(odd vs even games, Spearman-Brown corrected) is printed too.

In [ ]:
OFFICIAL = ["forty", "ten_yd_split", "vertical", "broad_jump", "three_cone", "short_shuttle", "bench_reps",
            "combine_height", "combine_weight", "arm_length", "hand_size", "wing_span"]
TRACK = list(METRICS40)
X = combine[OFFICIAL].join(best40[TRACK])
N_METRICS = X.shape[1]
BURST_AGILITY = {"ten_yd_split", "three_cone", "short_shuttle", "split5_10", "acc2_8", "first10", "amax"}
NICE = {**METRICS40, "forty": "40 time", "ten_yd_split": "10-yd split", "vertical": "vertical",
        "broad_jump": "broad jump", "three_cone": "3-cone", "short_shuttle": "shuttle", "bench_reps": "bench",
        "combine_height": "height", "combine_weight": "weight", "arm_length": "arm length",
        "hand_size": "hand size", "wing_span": "wingspan", "vmax": "Combine top speed",
        "top_speed": "in-game top speed", "separation": "separation", "get_off": "get-off",
        "pressure_allowed": "pressure allowed"}

speed = reg_plays[reg_plays.smax < 12.5]                     # drop glitch frames above 12.5 yd/s
N_GLITCH = int((reg_plays.smax >= 12.5).sum())
TOP = speed.groupby("nfl_id").smax.agg(lambda v: np.quantile(v, 0.95))
IDX = combine.index


def enough(counts):
    return counts.reindex(IDX).fillna(0) >= MIN_PLAYS


OUT = pd.DataFrame(index=IDX)
OUT["top_speed"] = TOP.reindex(IDX).where(enough(n_reg))
OUT["separation"] = reg_pp.groupby("nfl_id").separation_at_pass_forward.mean().reindex(IDX).where(enough(n_out["WR"]))
OUT["get_off"] = reg_pp.groupby("nfl_id").player_get_off.median().reindex(IDX).where(enough(n_out["DL"]))
OUT["pressure_allowed"] = reg_pp.groupby("nfl_id").pressure_allowed.mean().reindex(IDX).where(enough(n_out["OL"]))
OUTCOMES = {"WR": ["separation", "top_speed"], "TE": ["separation", "top_speed"], "DB": ["top_speed"],
            "DL": ["get_off", "top_speed"], "OL": ["pressure_allowed", "top_speed"]}
print(f"regular-season player-plays with a max speed of 12.5 yd/s or more, dropped as glitches: {N_GLITCH}")


def split_half(df, col, agg, ids):
    d = df[df.nfl_id.isin(ids)].dropna(subset=[col])
    gi = d[["nfl_id", "game_id"]].drop_duplicates().sort_values(["nfl_id", "game_id"])
    gi["half"] = gi.groupby("nfl_id").cumcount() % 2
    h = d.merge(gi, on=["nfl_id", "game_id"]).groupby(["nfl_id", "half"])[col].agg(agg).unstack().dropna()
    r = np.corrcoef(h[0], h[1])[0, 1]
    return 2 * r / (1 + r)


SRC = {"separation": (reg_pp, "separation_at_pass_forward", "mean"), "get_off": (reg_pp, "player_get_off", "median"),
       "pressure_allowed": (reg_pp, "pressure_allowed", "mean"),
       "top_speed": (speed, "smax", lambda v: np.quantile(v, 0.95))}


def pair_r(Xm, Ym, min_n=15):
    # pairwise-complete Pearson r and its n for every (metric, outcome) column pair
    out, nn_ = np.full((Xm.shape[1], Ym.shape[1]), np.nan), np.zeros((Xm.shape[1], Ym.shape[1]), int)
    for j in range(Ym.shape[1]):
        for i in range(Xm.shape[1]):
            m = ~np.isnan(Xm[:, i]) & ~np.isnan(Ym[:, j])
            nn_[i, j] = m.sum()
            if m.sum() >= min_n:
                out[i, j] = np.corrcoef(Xm[m, i], Ym[m, j])[0, 1]
    return out, nn_


def zstat(r, n):
    return np.arctanh(r) * np.sqrt(n - 3)


NF, REL_OUT, CLEARED, IDS = [], {}, [], {}
for p in POS_ORDER:
    outs = OUTCOMES[p]
    ids = [i for i in POS.index[POS == p] if i in has40 and np.isfinite(OUT.loc[i, outs[0]])]
    IDS[p] = ids
    n = len(ids)
    Xm, Ym = X.loc[ids].to_numpy(float), OUT.loc[ids, outs].to_numpy(float)
    obs, npair = pair_r(Xm, Ym)
    zobs = np.abs(zstat(obs, npair))
    m_pairs = int(np.isfinite(obs).sum())
    perm = np.empty(2000)
    for b in range(len(perm)):
        rb, nb_ = pair_r(Xm, Ym[RNG.permutation(n)])
        perm[b] = np.nanmax(np.abs(zstat(rb, nb_)))
    z95 = np.quantile(perm, 0.95)
    i_, j_ = np.unravel_index(np.nanargmax(zobs), zobs.shape)
    n_b = int(npair[i_, j_])
    for o in outs:
        df_, col, agg = SRC[o]
        REL_OUT[(p, o)] = split_half(df_, col, agg, ids)
    for i, j in zip(*np.nonzero(zobs > z95)):
        CLEARED.append({"pos": p, "metric": X.columns[i], "outcome": outs[j], "r": obs[i, j], "n": int(npair[i, j]),
                        "pair": f"{NICE[X.columns[i]]} vs {NICE[outs[j]]}"})
    NF.append({"pos": p, "n": n, "pairs": m_pairs, "pair_n_min": int(npair[np.isfinite(obs)].min()),
               "one_test": r_crit(n), "power80": r_min(n), "z95": z95,
               "best_pair": f"{NICE[X.columns[i_]]} vs {NICE[outs[j_]]}", "best_n": n_b, "best_r": obs[i_, j_],
               "best_bar_r": np.tanh(z95 / np.sqrt(n_b - 3)), "best_one_test": r_crit(n_b),
               "best_bonf": r_crit(n_b, 0.05 / m_pairs), "clears": zobs[i_, j_] > z95})
NF = pd.DataFrame(NF).set_index("pos")
CLEARED = pd.DataFrame(CLEARED)
print(NF.drop(columns=["z95"]).round(3).to_string())
print(f"\npairs that clear their group's shuffled bar: {len(CLEARED)}")
print(CLEARED.round(3).to_string(index=False))
CLEAR_GROUPS = [p for p in POS_ORDER if p in set(CLEARED.pos)]
N_CLEAR_BURST = int(CLEARED.metric.isin(BURST_AGILITY).sum())
BURST_GROUPS = [p for p in POS_ORDER if p in set(CLEARED.pos[CLEARED.metric.isin(BURST_AGILITY)])]
print(f"groups with at least one clearing pair: {CLEAR_GROUPS}; clearing pairs that use a burst or agility "
      f"metric: {N_CLEAR_BURST}, in {BURST_GROUPS}")
SIZE = {"combine_height", "combine_weight", "arm_length", "hand_size", "wing_span"}
SPEEDM = {"forty", "vmax", "fly30"}
CLEARED["kind"] = CLEARED.metric.map(lambda m: "size" if m in SIZE else "speed" if m in SPEEDM
                                     else "burst or agility" if m in BURST_AGILITY else "jump or strength")
KINDS = CLEARED.groupby(["pos", "kind"]).size().unstack(fill_value=0).reindex(CLEAR_GROUPS)
print(KINDS.to_string())
dlx = X.loc[IDS["DL"]]
DL_CLEAR_M = sorted(set(CLEARED.metric[CLEARED.pos == "DL"]) - {"combine_weight"})
DL_WCORR = {m: abs(dlx[[m, "combine_weight"]].dropna().corr().iloc[0, 1]) for m in DL_CLEAR_M}
print("defensive line, |r| of each other clearing metric with weight:",
      {NICE[m]: round(v, 2) for m, v in DL_WCORR.items()})
print("\noutcome repeatability (odd vs even games, Spearman-Brown):")
for (p, o), v in REL_OUT.items():
    print(f"  {p} {o:17s} {v:.3f}")

# expected ceiling from repeatability: sqrt(rel_x * rel_y), rel_x for the better of two attempts by
# Spearman-Brown; the interval carries the attempt-to-attempt r's own 95% interval through
CAP = {}
for p, o in [("WR", "separation")]:
    m = "amax"
    lo_, hi_ = fisher_ci(REL.loc[m, p], RELN.loc[m, p])
    sb = lambda r: 2 * max(r, 0) / (1 + max(r, 0))
    CAP[(p, m)] = (np.sqrt(sb(REL.loc[m, p]) * REL_OUT[(p, o)]), np.sqrt(sb(hi_) * REL_OUT[(p, o)]))
    print(f"expected ceiling on |r|: {p} {METRICS40[m]} vs {o}: {CAP[(p, m)][0]:.2f} "
          f"(up to {CAP[(p, m)][1]:.2f} at the top of the repeatability interval)")

from matplotlib.patches import Patch
from matplotlib.lines import Line2D
fig, axs = plt.subplots(1, 2, figsize=(14.5, 4.9), gridspec_kw={"width_ratios": [1, 1.3]})
ax = axs[0]
nn = np.arange(12, 521)
ax.axvspan(NF.n.min(), NF.n.max(), color=GRID, zorder=0)
ax.text(NF.n.max() + 6, 0.93, "the five analysis\nsets sit here", fontsize=8.5, color=INK2, va="top")
ax.plot(nn, r_min(nn), color=BLUE, lw=2)
ax.plot(nn, r_crit(nn), color=INK2, lw=2)
ax.text(505, r_min(505) + 0.02, "true |r| found 80% of the time", color=BLUE, ha="right", va="bottom", fontsize=9)
ax.text(505, r_crit(505) - 0.02, "observed |r| for p < 0.05", color=INK2, ha="right", va="top", fontsize=9)
ax.set_xlim(0, 525); ax.set_ylim(0, 1)
ax.set_xlabel("players in the analysis"); ax.set_ylabel("|r|")
ax.set_title("One pre-chosen test, by sample size")
ax = axs[1]
yy = np.arange(len(NF))[::-1]
ax.barh(yy, NF.best_bar_r, height=0.56, color=GRID, zorder=1)
ax.plot(NF.best_one_test, yy, "|", color=INK2, ms=22, mew=2, zorder=2)
ax.plot(NF.best_bonf, yy, "|", color=ORANGE, ms=22, mew=2, zorder=2)
for y_, (p, row) in zip(yy, NF.iterrows()):
    fc = BLUE if row.clears else SURFACE
    ax.plot(abs(row.best_r), y_, "D", color=BLUE, mfc=fc, ms=8, mew=1.8, zorder=3)
    ax.text(max(abs(row.best_r), row.best_bar_r, row.best_bonf) + 0.02, y_, f"{row.best_pair} (n {row.best_n})",
            va="center", fontsize=8.5, color=INK2)
ax.set_yticks(yy); ax.set_yticklabels(list(NF.index))
ax.set_xlim(0, 1.45); ax.set_ylim(-1.6, len(NF) - 0.3)
ax.set_xticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_xlabel("|r|, all bars at the n of the pair shown")
ax.grid(axis="y", visible=False)
handles = [Patch(color=GRID, label="best pair by chance (95th pct.)"),
           Line2D([], [], marker="|", color=INK2, ms=12, mew=2, ls="none", label="one test, p < 0.05"),
           Line2D([], [], marker="|", color=ORANGE, ms=12, mew=2, ls="none", label="Bonferroni, all pairs"),
           Line2D([], [], marker="D", color=BLUE, ms=7, ls="none", label="best real pair, clears chance"),
           Line2D([], [], marker="D", color=BLUE, mfc=SURFACE, mew=1.8, ms=7, ls="none", label="best real pair, does not")]
ax.legend(handles=handles, loc="lower center", ncol=3, fontsize=8, bbox_to_anchor=(0.5, -0.02))
ax.set_title(f"Each group's strongest pair among {N_METRICS} metrics x its outcomes")
fig.suptitle("Figure 5. The correlation needed before a Combine-to-NFL link can be told from noise", x=0.01,
             ha="left", fontsize=12, fontweight="bold", color=INK)
fig.tight_layout()
plt.show()

At these sample sizes a single pre-chosen pair needs |r| of about 0.25 to
0.47 for p < 0.05, and to find a true correlation 80% of the time it should be
about 0.35 to 0.62. Searching every pair, the best that chance produces sits
at |r| of 0.36 to 0.68 at the n of each group's strongest pair. 28
real pairs clear their group's bar (wide receivers 8: 5 size, 3 speed; defensive backs 1: 1 size; the defensive line 19: 8 burst or agility, 3 jump or strength, 2 size, 6 speed). 8 of them involve a burst or agility metric, all for the defensive line. On the defensive
line every clearing metric other than weight itself also correlates with weight (|r| from
0.37 to 0.84), so none of those pairs can be read without holding size
fixed; section 6 does that for one. For wide receivers the clearing pairs are size
(5) and straight-line speed (3). The repeatability of section 3 sets an expected ceiling too: a wide
receiver's peak acceleration off the start would be expected to correlate with his
separation at about 0.18 at most (up to 0.57 at the top of its
repeatability interval).

## 6. Worked example: does defensive-line burst show up as NFL get-off?

`player_get_off` is the time from the snap until a rusher crosses the line of scrimmage,
and it repeats well (split-half reliability 0.96, odd vs even games). The obvious
Combine counterpart is the official 10-yard split. The obvious confounders are size and
where a player lines up, and the data shows both: on regular-season rushes, plays from an
interior alignment have a median get-off of 1.025 s against 0.908 s from all
other alignments, and a lineman's share of get-off plays from an interior alignment
correlates 0.85 with his weight.
Intervals are bootstrap 95% (4,000 resamples of players).

In [ ]:
dl = pd.DataFrame({"get_off": OUT.get_off, "weight": combine.combine_weight}).join(
    combine[["ten_yd_split", "forty"]]).join(best40[["split5_10", "acc2_8", "first10", "fly30"]])
dl = dl[(POS.reindex(dl.index) == "DL") & dl.get_off.notna()]
rush = reg_pp[reg_pp.player_get_off.notna() & (reg_pp.nfl_id.map(POS) == "DL")]
inside = rush.lined_up_position.eq("INTERIOR_LINE")
print("DL regular-season get-off plays by alignment:", rush.lined_up_position.value_counts().to_dict())
GO_IN, GO_EDGE = rush.player_get_off[inside].median(), rush.player_get_off[~inside].median()
dl["interior"] = inside.groupby(rush.nfl_id).mean()
print(f"median get-off: interior {GO_IN:.3f} s, other alignments {GO_EDGE:.3f} s")


def partial(x, y, Z):
    Z = np.c_[np.ones(len(x)), Z]
    rx = x - Z @ np.linalg.lstsq(Z, x, rcond=None)[0]
    ry = y - Z @ np.linalg.lstsq(Z, y, rcond=None)[0]
    return np.corrcoef(rx, ry)[0, 1]


def partial_boot(x, y, Z, B=4000, rng=RNG):
    x, y, Z = np.asarray(x, float), np.asarray(y, float), np.asarray(Z, float).reshape(len(x), -1)
    idx = rng.integers(0, len(x), (B, len(x)))
    bs = np.array([partial(x[i], y[i], Z[i]) for i in idx])
    return partial(x, y, Z), np.quantile(bs, [0.025, 0.975])


DL_LABEL = {"ten_yd_split": "Official 10-yd split", "forty": "Official 40 time", "split5_10": "Tracking 5 to 10 yd split",
            "acc2_8": "Tracking 2 to 8 yd/s time", "first10": "Tracking first 10 yd", "fly30": "Tracking 10 to 40 yd"}
FOREST = []
for m in DL_LABEL:
    d = dl[[m, "get_off", "weight", "interior"]].dropna()
    r = np.corrcoef(d[m], d.get_off)[0, 1]
    lo, hi = boot_r(d[m], d.get_off)
    pr_, (plo, phi) = partial_boot(d[m], d.get_off, d[["weight"]])
    t_ = pr_ * np.sqrt((len(d) - 3) / (1 - pr_ ** 2))
    FOREST.append({"metric": m, "n": len(d), "raw": r, "raw_lo": lo, "raw_hi": hi,
                   "partial": pr_, "part_lo": plo, "part_hi": phi, "part_p": 2 * stats.t.sf(abs(t_), len(d) - 3),
                   "partial_w_align": partial(d[m].to_numpy(float), d.get_off.to_numpy(float),
                                              d[["weight", "interior"]].to_numpy(float))})
d = dl[["ten_yd_split", "get_off", "weight", "interior"]].dropna()
PW2, (PW2_LO, PW2_HI) = partial_boot(d.ten_yd_split, d.get_off, d[["weight", "interior"]])
print(f"10-yd split vs get-off, weight and interior share held fixed: {PW2:.3f} [{PW2_LO:.3f}, {PW2_HI:.3f}]")
FOREST = pd.DataFrame(FOREST).set_index("metric")
d = dl[["weight", "get_off", "ten_yd_split", "interior"]].dropna()
W_GO, W_TEN = np.corrcoef(d.weight, d.get_off)[0, 1], np.corrcoef(d.weight, d.ten_yd_split)[0, 1]
W_IN = np.corrcoef(d.weight, d.interior)[0, 1]
print(f"DL with {MIN_PLAYS}+ regular-season get-off plays: {dl.shape[0]}")
print(f"weight vs get-off r {W_GO:.3f} | weight vs 10-yd split r {W_TEN:.3f} | weight vs interior share r {W_IN:.3f}")
print(FOREST.round(4).to_string())
row = FOREST.loc["ten_yd_split"]
N_DL = int(row.n)
DL_CRIT = r_crit(N_DL, k=1)
DL_PAIRS = int(NF.loc["DL", "pairs"])
DL_BONF = r_crit(N_DL, 0.05 / DL_PAIRS, k=1)
SHARED = 1 - row.partial ** 2 * (1 - W_GO ** 2) / row.raw ** 2
TRK_PART = FOREST.loc[["split5_10", "acc2_8", "first10"], "partial"]
print(f"with weight fixed, |r| needed for p < 0.05 at n = {N_DL}: {DL_CRIT:.3f}; "
      f"after correcting for the {DL_PAIRS} DL pairs of section 5: {DL_BONF:.3f}")
print(f"share of the get-off variance the 10-yd split accounts for that weight also accounts for: {SHARED:.0%}")
print(f"tracking burst metrics with weight fixed: r {TRK_PART.min():.2f} to {TRK_PART.max():.2f}")

fig, axs = plt.subplots(1, 3, figsize=(14.5, 4.9), gridspec_kw={"width_ratios": [1, 1, 1.3]})
ax = axs[0]
sc = ax.scatter(d.ten_yd_split, d.get_off, c=d.weight, cmap=SEQ, vmin=d.weight.min() - 20, vmax=d.weight.max(),
                s=34, edgecolor=MUTED, linewidth=0.4)
cb = fig.colorbar(sc, ax=ax, fraction=0.05, pad=0.02); cb.set_label("Combine weight (lb)", color=INK2)
cb.outline.set_visible(False)
ax.set_xlabel("official 10-yard split (s)"); ax.set_ylabel("median NFL get-off (s)")
ax.set_title("Raw: slower at the Combine, slower off the snap")
ax.text(0.97, 0.03, f"r = {row.raw:.2f} [{row.raw_lo:.2f}, {row.raw_hi:.2f}], n = {int(row.n)}", transform=ax.transAxes,
        ha="right", va="bottom", fontsize=9, color=INK2)
ax = axs[1]
Z = np.c_[np.ones(len(d)), d.weight]
rx = d.ten_yd_split - Z @ np.linalg.lstsq(Z, d.ten_yd_split, rcond=None)[0]
ry = d.get_off - Z @ np.linalg.lstsq(Z, d.get_off, rcond=None)[0]
ax.scatter(rx, ry, s=34, color=BLUE, alpha=0.75, edgecolor=SURFACE, linewidth=0.5)
ax.axhline(0, color=MUTED, lw=0.8); ax.axvline(0, color=MUTED, lw=0.8)
ax.set_xlabel("10-yard split, beyond what weight predicts (s)"); ax.set_ylabel("get-off, beyond what weight predicts (s)")
ax.set_title("Same players, weight held fixed")
ax.set_ylim(ry.min() - 0.02, ry.max() + 0.07)
note(ax, f"partial r = {row.partial:.2f} [{row.part_lo:.2f}, {row.part_hi:.2f}]", xy=(0.03, 0.97))
ax = axs[2]
yy = np.arange(len(FOREST))[::-1]
ax.axvspan(-DL_CRIT, DL_CRIT, color=GRID, zorder=0)
ax.text(0, len(FOREST) - 0.35, f"p > 0.05 as one test\n(|r| < {DL_CRIT:.2f}, n = {N_DL})", ha="center",
        va="bottom", fontsize=8, color=INK2)
ax.axvline(DL_BONF, color=INK2, lw=1)
ax.text(DL_BONF + 0.02, len(FOREST) - 0.35, f"needed after\n{DL_PAIRS}-pair search", fontsize=8, color=INK2,
        va="bottom")
ax.hlines(yy + 0.15, FOREST.raw_lo, FOREST.raw_hi, color=MUTED, lw=2)
ax.plot(FOREST.raw, yy + 0.15, "o", color=MUTED, ms=6, mec=SURFACE, label="raw")
ax.hlines(yy - 0.15, FOREST.part_lo, FOREST.part_hi, color=ORANGE, lw=2)
ax.plot(FOREST.partial, yy - 0.15, "o", color=ORANGE, ms=7, mec=SURFACE, label="weight held fixed")
ax.axvline(0, color=MUTED, lw=0.8)
ax.set_yticks(yy); ax.set_yticklabels([DL_LABEL[m] for m in FOREST.index])
ax.set_xlim(-0.6, 1); ax.set_ylim(-0.6, len(FOREST) + 0.4)
ax.set_xlabel("correlation with median NFL get-off (95% bootstrap CI)")
ax.set_title("Every speed metric shrinks once weight is fixed")
ax.legend(loc="upper center", bbox_to_anchor=(0.45, -0.17), ncol=2)
ax.grid(axis="y", visible=False)
fig.suptitle("Figure 6. Defensive line: Combine burst against NFL get-off", x=0.01, ha="left", fontsize=12,
             fontweight="bold", color=INK)
fig.tight_layout()
plt.show()

The raw correlation looks like a finding (r = 0.67). But weight correlates
0.63 with get-off and 0.83 with the 10-yard split, and 84% of the get-off
variance the split accounts for is shared with weight. Hold weight fixed and the split's
correlation falls to 0.35 (0.09 to 0.56); hold his share of get-off
plays from an interior alignment fixed as well and it is 0.34 (0.09 to
0.54). As a single test the weight-adjusted value is
p = 0.011, but I chose this example after exploring the data, so even that p is
optimistic; inside the section 5 search over 36 pairs it would need 0.43.
The official 40 behaves the same way (0.68 raw, 0.36 with weight fixed),
and the tracking burst metrics fall further, to 0.04 to 0.22.

So most of the raw link cannot be told apart from size. Whatever burst adds beyond size and
alignment lies somewhere between trivial and moderate, and 54 linemen cannot say which.
That is a lead to test on more players, not a finding.

### A positive control: does speed transfer?

The same pipeline should find a link where one obviously exists. Combine top speed against
in-game top speed, within each position group:

In [ ]:
PC = []
d_all = best40[["vmax"]].join(OUT.top_speed).join(POS).dropna()
for p in POS_ORDER:
    d = d_all[d_all.combine_position == p]
    r = np.corrcoef(d.vmax, d.top_speed)[0, 1]
    lo, hi = boot_r(d.vmax, d.top_speed)
    PC.append({"pos": p, "n": len(d), "r": r, "lo": lo, "hi": hi})
PC = pd.DataFrame(PC).set_index("pos")
PC_POOLED = np.corrcoef(d_all.vmax, d_all.top_speed)[0, 1]
PC_CLEAR = [p for p in POS_ORDER if PC.loc[p, "lo"] > 0]
print(PC.round(3).to_string())
print(f"pooled across positions: r {PC_POOLED:.3f} (n {len(d_all)})")
print(f"groups whose interval excludes zero: {PC_CLEAR}")

fig, ax = plt.subplots(figsize=(9, 3.4))
yy = np.arange(len(PC))[::-1]
ax.hlines(yy, PC.lo, PC.hi, color=BLUE, lw=2)
ax.plot(PC.r, yy, "o", color=BLUE, ms=7, mec=SURFACE, mew=1.5)
ax.axvline(PC_POOLED, color=ORANGE, lw=1.5)
ax.text(PC_POOLED - 0.01, len(PC) - 0.45, f"pooled r = {PC_POOLED:.2f}\n(mixes positions)", color=ORANGE,
        ha="right", va="top", fontsize=9)
ax.axvline(0, color=MUTED, lw=0.8)
ax.set_yticks(yy); ax.set_yticklabels([f"{POS_NAME[p]} (n {PC.loc[p, 'n']})" for p in PC.index])
ax.set_xlim(-0.4, 1); ax.set_ylim(-0.6, len(PC) - 0.3)
ax.set_xlabel("correlation, Combine top speed vs NFL in-game top speed (95% bootstrap CI)")
ax.set_title(f"Figure 7. Positive control: Combine top speed carries over for {' and '.join(PC_CLEAR)}", fontsize=12)
ax.grid(axis="y", visible=False)
fig.tight_layout()
plt.show()

Within position, Combine top speed tracks in-game top speed for wide receivers (r = 0.59) and the defensive line (r = 0.62). For
the other three groups the interval includes zero, and this audit does not test why. Pooled
across positions the same correlation reads 0.87, which mostly says the position
groups differ in speed.

## 7. Change of direction (new in version 2)

Section 6 showed that top speed carries over for some groups. Agility is the other thing
scouts time, with the 3-cone and the short shuttle. I asked three questions:

1. **Where can the tracking see change of direction?**
2. **Does it repeat?** A trait should show up in more than one drill.
3. **Does it show up in the NFL outcome it should predict?** For receivers, that is
   separation at the throw.

Many position drills contain a designed hard stop: a comeback or a curl for receivers, the
corners of the box drill for defensive backs. In each attempt I find the first hard stop.
After the player first reaches 3 yd/s, it is the first frame where his speed falls below half
of his highest speed so far, followed down to its lowest point, and he must reach 3 yd/s again
afterwards. I measure four things: peak braking into the stop (yd/s², from speed smoothed
over three frames), his speed at the bottom, the time from 80% of his entry speed to the
bottom, and the speed he gains in the next second. A drill counts for a group if at least
70% of its attempts contain such a stop and at least 40 of the group's players ran it. The
tracked 3-cone and shuttle are left out, so the official times can serve as a check.

Each metric is standardised within each drill and each Combine year before players are
compared, because the drills read differently from one Combine to the next (the cell prints
how much). I fixed these definitions, and chose braking as the main metric, before looking at
any NFL number.

In [ ]:
import itertools

TRK_OK = {d_: set(agil[d_].loc[agil[d_].status == "ok", "nfl_id"]) for d_ in agil}


def stop_metrics(tt, s):
    # the first hard stop: after the player first reaches 3 yd/s, the first frame where his speed falls
    # below half of his running peak, followed down to its lowest point; he must reach 3 yd/s again after it
    i3 = np.flatnonzero(s >= 3.0)
    if len(i3) == 0:
        return None
    cand = np.flatnonzero((np.arange(len(s)) > i3[0]) & (s < 0.5 * np.maximum.accumulate(s)))
    if len(cand) == 0:
        return None
    i = cand[0]
    while i + 1 < len(s) and s[i + 1] <= s[i]:
        i += 1
    ie = int(np.argmax(s[:i + 1]))
    if i <= ie + 1 or i >= len(s) - 1 or s[i:].max() < 3.0:
        return None
    dsdt = np.gradient(pd.Series(s).rolling(3, center=True, min_periods=1).mean().to_numpy(), tt)
    k = ie + np.flatnonzero(s[ie:i + 1] < 0.8 * s[ie])[0]
    t80 = tt[k - 1] + (s[k - 1] - 0.8 * s[ie]) / (s[k - 1] - s[k]) * (tt[k] - tt[k - 1])
    later = np.flatnonzero(tt >= tt[i] + 1.0)
    return {"entry": s[ie], "brake": -dsdt[ie:i + 1].min(), "stop_speed": s[i], "stop_time": tt[i] - t80,
            "exit": s[later[0]] - s[i] if len(later) else np.nan}


rows = []
for e, d in pl[pl.drill_type != "FORTY_YARD_DASH"].groupby("event_id", sort=False):
    m = stop_metrics((d.t - d.t.iloc[0]).dt.total_seconds().to_numpy(), d.s.to_numpy())
    rows.append({"nfl_id": d.nfl_id.iloc[0], "drill": d.drill_name.iloc[0], "attempt": d.attempt.iloc[0],
                 "year": d.draft_year.iloc[0], "stop": m is not None, **(m or {})})
STOPS = pd.DataFrame(rows)
STOPS["pos"] = STOPS.nfl_id.map(POS)
COD_M = {"brake": "Peak braking into the stop", "stop_speed": "Speed at the bottom of the stop",
         "stop_time": "Time from 80% of entry speed to the bottom", "exit": "Speed gained in the next second"}
DR = STOPS.groupby(["pos", "drill"]).stop.agg(attempts="size", stop_share="mean").join(
    STOPS[STOPS.stop].groupby(["pos", "drill"]).nfl_id.nunique().rename("players"))
DR = DR[~DR.index.get_level_values("drill").isin(["SHORT_SHUTTLE", "THREE_CONE_DRILL"])]
ELIG = DR[(DR.stop_share >= 0.7) & (DR.players >= 40)]
DRILLS = {p: sorted(ELIG.loc[p].index) if p in ELIG.index.get_level_values("pos") else [] for p in POS_ORDER}
GROUPS7 = [p for p in POS_ORDER if len(DRILLS[p]) >= 2]

# one value per player and drill: his first attempt with a stop
in_drills = np.array([d_ in DRILLS.get(p_, []) for p_, d_ in zip(STOPS.pos, STOPS.drill)])
FIRST = STOPS[STOPS.stop & in_drills].sort_values(["nfl_id", "drill", "attempt"]).groupby(["nfl_id", "drill"]).head(1).copy()
for m in COD_M:
    FIRST[f"{m}_zy"] = FIRST.groupby(["pos", "drill", "year"])[m].transform(lambda v: (v - v.mean()) / v.std())
    FIRST[f"{m}_zd"] = FIRST.groupby(["pos", "drill"])[m].transform(lambda v: (v - v.mean()) / v.std())
N_DR = FIRST.groupby("nfl_id").drill.nunique()

C7 = []
for p in POS_ORDER:
    ids = POS.index[POS == p]
    C7.append({"pos": p, "prospects": len(ids),
               "official 3-cone": int(combine.loc[ids, "three_cone"].notna().sum()),
               "official shuttle": int(combine.loc[ids, "short_shuttle"].notna().sum()),
               "tracked 3-cone, timed": len(TRK_OK["THREE_CONE_DRILL"] & set(ids)),
               "tracked shuttle, timed": len(TRK_OK["SHORT_SHUTTLE"] & set(ids)),
               "stop in 2+ position drills": int((N_DR.reindex(ids).fillna(0) >= 2).sum()),
               "drills used": len(DRILLS[p])})
C7 = pd.DataFrame(C7).set_index("pos")
print("Where change of direction can be measured (players per group)")
print(C7.to_string())
print("\ndrills that count (a stop in 70%+ of attempts, 40+ players; 3-cone and shuttle left out):")
for p in POS_ORDER:
    print(f"  {p}: " + ", ".join(f"{d_} ({ELIG.loc[(p, d_), 'stop_share']:.0%} of {ELIG.loc[(p, d_), 'attempts']})"
                                 for d_ in DRILLS[p]))

YG = []
for (p, d_), g in FIRST.groupby(["pos", "drill"]):
    mu = g.groupby("year").brake.mean()
    YG.append({"pos": p, "drill": d_, **{int(y): mu.get(y, np.nan) for y in (2023, 2024, 2025)},
               "gap_sd": (mu.max() - mu.min()) / np.sqrt(g.groupby("year").brake.var().mean()),
               "high": int(mu.idxmax()), "low": int(mu.idxmin())})
YG = pd.DataFrame(YG).set_index(["pos", "drill"])
YG_TOP = YG.gap_sd.idxmax()
print("\npeak braking into the stop, mean by Combine year (yd/s^2); gap = largest minus smallest year, "
      "in within-year SDs")
print(YG.round(2).to_string())


def pair_rs(W):
    out = []
    for a_, b_ in itertools.combinations(range(W.shape[1]), 2):
        ok_ = ~np.isnan(W[:, a_]) & ~np.isnan(W[:, b_])
        out.append((np.corrcoef(W[ok_, a_], W[ok_, b_])[0, 1], int(ok_.sum())))
    return out


REP, PAIRS = [], []
for p in GROUPS7:
    k = len(DRILLS[p])
    f = FIRST[FIRST.pos == p]
    for m in COD_M:
        for cen, lab in (("zy", "drill and year"), ("zd", "drill only")):
            W = f.pivot(index="nfl_id", columns="drill", values=f"{m}_{cen}")[DRILLS[p]].to_numpy(float)
            prs = pair_rs(W)
            rbar = np.mean([q[0] for q in prs])
            bs = [np.mean([q[0] for q in pair_rs(W[RNG.integers(0, len(W), len(W))])]) for _ in range(1000)]
            lo, hi = np.quantile(bs, [0.025, 0.975])
            REP.append({"pos": p, "metric": m, "centred": lab, "drills": k, "mean_r": rbar, "lo": lo, "hi": hi,
                        "pair_r_min": min(q[0] for q in prs), "pair_r_max": max(q[0] for q in prs),
                        "pair_n_min": min(q[1] for q in prs), "pair_n_max": max(q[1] for q in prs),
                        "alpha": k * rbar / (1 + (k - 1) * rbar)})
            if m == "brake":
                names = list(itertools.combinations(DRILLS[p], 2))
                PAIRS += [{"pos": p, "centred": lab, "pair": f"{a_} / {b_}", "r": q[0], "n": q[1]}
                          for (a_, b_), q in zip(names, prs)]
REP = pd.DataFrame(REP).set_index(["pos", "metric", "centred"])
PAIRS = pd.DataFrame(PAIRS)
print("\none drill against another: mean r over the pairs of drills (bootstrap 95% CI, players resampled); "
      "alpha = reliability of the average over the drills")
print(REP.round(2).to_string())
print("\npeak braking, each pair of drills:")
print(PAIRS.round(2).to_string(index=False))

BENCH = {}
for p in POS_ORDER:
    b = combine.loc[POS == p, ["three_cone", "short_shuttle", "draft_year"]].dropna()
    b = b[["three_cone", "short_shuttle"]] - b.groupby("draft_year")[["three_cone", "short_shuttle"]].transform("mean")
    r = np.corrcoef(b.three_cone, b.short_shuttle)[0, 1]
    BENCH[p] = {"r": r, "n": len(b), **dict(zip(("lo", "hi"), fisher_ci(r, len(b))))}
BENCH = pd.DataFrame(BENCH).T
print("\nofficial 3-cone vs official shuttle, within group and Combine year (all prospects with both):")
print(BENCH.round(2).to_string())

# a braking score per player: the mean of his year-centred braking over the drills he ran (2 or more)
MIX = {p: np.corrcoef(*N_DR.reindex(POS.index[POS == p]).dropna().to_frame("n").join(
    combine.combine_weight).to_numpy(float).T)[0, 1] for p in GROUPS7}
print("\nr between weight and the number of these drills a player ran:", {p: round(v, 2) for p, v in MIX.items()})
SCORE_GROUPS = ["WR", "DB"]
COMP = FIRST.groupby("nfl_id").agg(brake_score=("brake_zy", "mean"), stop_drills=("drill", "nunique"))
COMP = COMP[(COMP.stop_drills >= 2)].join(POS)
COMP = COMP[COMP.combine_position.isin(SCORE_GROUPS)]
VAL = []
for p in SCORE_GROUPS:
    for ref in ("short_shuttle", "three_cone"):
        d = COMP[COMP.combine_position == p].join(combine[[ref, "draft_year"]]).dropna()
        d[ref] = d[ref] - d.groupby("draft_year")[ref].transform("mean")
        r = np.corrcoef(d.brake_score, d[ref])[0, 1]
        VAL.append({"pos": p, "official": ref, "n": len(d), "r": r, **dict(zip(("lo", "hi"), fisher_ci(r, len(d))))})
VAL = pd.DataFrame(VAL).set_index(["pos", "official"])
VAL_ZERO = bool(((VAL.lo < 0) & (VAL.hi > 0)).all())
print("\nbraking score vs the official agility times, both centred within Combine year "
      "(a harder brake should go with a faster time):")
print(VAL.round(2).to_string())
print(f"every interval includes zero: {VAL_ZERO}")

In [ ]:
fig, ax = plt.subplots(figsize=(11.5, 5.4))
ROWS7 = []
for p in GROUPS7:
    v = REL.loc["vmax", p]
    lo_, hi_ = fisher_ci(v, RELN.loc["vmax", p])
    ROWS7 += [(p, "Top speed, 40 attempt 1 vs 2 (section 3)", v, lo_, hi_, INK2, "s", True),
              (p, "Official 3-cone vs official shuttle", BENCH.loc[p, "r"], BENCH.loc[p, "lo"], BENCH.loc[p, "hi"],
               ORANGE, "D", True)]
    for lab, fill_ in (("drill only", False), ("drill and year", True)):
        q = REP.loc[(p, "brake", lab)]
        ROWS7.append((p, f"Braking, one drill vs another ({'years pooled' if lab == 'drill only' else 'year centred'})",
                      q.mean_r, q.lo, q.hi, BLUE, "o", fill_))
yy = []
y = 0
for i, (p, lab, v, lo_, hi_, col, mk, fill_) in enumerate(ROWS7):
    if i and ROWS7[i - 1][0] != p:
        y += 0.8
    yy.append(y)
    ax.hlines(y, lo_, hi_, color=col, lw=2)
    ax.plot(v, y, mk, color=col, mfc=col if fill_ else SURFACE, mew=1.8, ms=7.5)
    y += 1
ax.set_yticks(yy)
ax.set_yticklabels([f"{POS_NAME[p]}: {lab}" for p, lab, *_ in ROWS7], fontsize=8.5)
ax.invert_yaxis()
ax.axvline(0, color=MUTED, lw=0.8)
ax.set_xlim(-0.4, 1.02)
ax.set_xlabel("Pearson r between two measurements of the same players (95% CI)")
ax.grid(axis="y", visible=False)
ax.set_title("Figure 8. What repeats: top speed, the official agility times, position-drill braking",
             fontsize=12)
fig.tight_layout()
plt.show()

**Where change of direction can be measured.** The official agility times cover a minority of
receivers and defensive backs: 33 of 107 receivers have an official 3-cone and
38 an official shuttle, and 25 and 33 of 122 defensive
backs. The tracking timed fewer still. The position drills reach almost everyone:
103 receivers and 118 defensive backs have a stop in at least two of them.

**The drills read differently from year to year.** The average braking in the same drill moves
by up to 3.3 within-year SDs between Combines (comeback route right for wide receivers, 2023 vs 2025). So a player has to be
compared with the others from his own Combine year. If he is not, the year becomes a signal
shared by every drill and looks like a trait: pooled across years, receivers' braking repeats
at a mean r of 0.23 (0.12 to 0.35).

**Within a year, it barely repeats.** A receiver's braking in one stop route predicts his
braking in another at r = 0.02 to 0.17 (mean 0.07, 95% CI
-0.07 to 0.20, 92 to 98 players per pair). For
defensive backs it is -0.04 to 0.03 (mean 0.00, -0.12 to
0.13). The other three stop measures do no better. Across the two groups their mean r runs from
-0.01 to 0.17. The highest of those six values is the
speed gained after the stop for defensive backs, at 0.17 (0.02 to 0.34), and the highest of six
is biased upwards. The defensive line shows the same pattern (braking 0.04).

For contrast, the official 3-cone and shuttle are two different drills, yet they agree at
r = 0.62 to 0.83 within each group and Combine year, and top speed repeats at 0.89 or
more. Agility behaves like a trait in the official times. As I measured them, the stops in
the position drills do not rank players the same way twice.

The braking score also says little about the official times: r = -0.15 with the
shuttle for receivers (n = 36) and -0.03 for defensive backs
(n = 33); all four intervals, the 3-cone included, contain zero. I build the score only for receivers and defensive
backs. On the defensive line, how many of these drills a player ran tracks his weight
(r = -0.58), so an average over drills would mix in size.

### Does it show up as NFL separation?

This is the one test I chose in advance: a receiver's braking score against his mean
separation at the throw (regular season, 100+ plays), with height and weight held fixed,
because size alone correlates with separation (section 5). Defensive backs are not tested.
`player_play` has no coverage outcome for them, and the game tracking holds only this
cohort's players, so the separation they allow cannot be rebuilt.

In [ ]:
wr7 = COMP[COMP.combine_position == "WR"][["brake_score", "stop_drills"]].join(OUT.separation).join(
    combine[["combine_height", "combine_weight", "draft_year"]]).join(best40.vmax)
dt7 = wr7.dropna(subset=["brake_score", "separation", "combine_height", "combine_weight"])
N_T = len(dt7)
SIZE7 = ["combine_height", "combine_weight"]
T_RAW = np.corrcoef(dt7.brake_score, dt7.separation)[0, 1]
T_RAW_CI = boot_r(dt7.brake_score, dt7.separation)
T_P, T_P_CI = partial_boot(dt7.brake_score, dt7.separation, dt7[SIZE7])
dv7 = dt7.dropna(subset=["vmax"])
T_PV, T_PV_CI = partial_boot(dv7.brake_score, dv7.separation, dv7[SIZE7 + ["vmax"]])
yd7 = pd.get_dummies(dt7.draft_year, drop_first=True).astype(float).to_numpy()
T_PY, T_PY_CI = partial_boot(dt7.brake_score, dt7.separation, np.c_[dt7[SIZE7].to_numpy(float), yd7])
rk7 = dt7[["brake_score", "separation"] + SIZE7].rank()
T_PR, T_PR_CI = partial_boot(rk7.brake_score, rk7.separation, rk7[SIZE7])
TR = pd.DataFrame([
    {"held fixed": "nothing", "n": N_T, "r": T_RAW, "lo": T_RAW_CI[0], "hi": T_RAW_CI[1]},
    {"held fixed": "height, weight (chosen in advance)", "n": N_T, "r": T_P, "lo": T_P_CI[0], "hi": T_P_CI[1]},
    {"held fixed": "height, weight, Combine top speed", "n": len(dv7), "r": T_PV, "lo": T_PV_CI[0], "hi": T_PV_CI[1]},
    {"held fixed": "height, weight, draft class", "n": N_T, "r": T_PY, "lo": T_PY_CI[0], "hi": T_PY_CI[1]},
    {"held fixed": "height, weight, all four as ranks", "n": N_T, "r": T_PR, "lo": T_PR_CI[0], "hi": T_PR_CI[1]},
]).set_index("held fixed")
T_CRIT, T_POW = r_crit(N_T, k=2), r_min(N_T)
rep7 = REP.loc[("WR", "brake", "drill and year")]
k7 = int(rep7.drills)
T_CEIL = np.sqrt(max(rep7.alpha, 0) * REL_OUT[("WR", "separation")])
T_CEIL_HI = np.sqrt(max(k7 * rep7.hi / (1 + (k7 - 1) * rep7.hi), 0) * REL_OUT[("WR", "separation")])
print("wide receivers: braking score vs mean NFL separation at the throw (bootstrap 95% CI)")
print(TR.round(3).to_string())
print(f"|r| needed for p < 0.05 with two covariates at n = {N_T}: {T_CRIT:.3f}; "
      f"true |r| found 80% of the time: {T_POW:.3f}")
print(f"expected ceiling from repeatability: {T_CEIL:.2f} (up to {T_CEIL_HI:.2f} at the top of its interval)")

# selection on outcome availability: do the receivers who reached 100 separation plays brake differently?
allwr = COMP[COMP.combine_position == "WR"].brake_score
inn = allwr.index.isin(dt7.index)
SEL_D = allwr[inn].mean() - allwr[~inn].mean()
se7 = np.sqrt(allwr[inn].var() / inn.sum() + allwr[~inn].var() / (~inn).sum())
SEL_CI = (SEL_D - 1.96 * se7, SEL_D + 1.96 * se7)
SEL_SD = (allwr[inn].std(), allwr.std())
print(f"\nreceivers with a braking score: {len(allwr)}; in the test: {inn.sum()}; mean score in the test minus "
      f"out of it: {SEL_D:.2f} [{SEL_CI[0]:.2f}, {SEL_CI[1]:.2f}]; SD of the score {SEL_SD[0]:.2f} "
      f"in the test vs {SEL_SD[1]:.2f} for all")

fig, ax = plt.subplots(figsize=(6.6, 4.6))
Z7 = np.c_[np.ones(N_T), dt7[SIZE7].to_numpy(float)]
rx7 = dt7.brake_score - Z7 @ np.linalg.lstsq(Z7, dt7.brake_score, rcond=None)[0]
ry7 = dt7.separation - Z7 @ np.linalg.lstsq(Z7, dt7.separation, rcond=None)[0]
ax.scatter(rx7, ry7, s=34, color=BLUE, alpha=0.75, edgecolor=SURFACE, linewidth=0.5)
ax.axhline(0, color=MUTED, lw=0.8); ax.axvline(0, color=MUTED, lw=0.8)
ax.set_xlabel("braking score, beyond what size predicts"); ax.set_ylabel("separation (yd), beyond what size predicts")
note(ax, f"partial r = {T_P:.2f} [{T_P_CI[0]:.2f}, {T_P_CI[1]:.2f}], n = {N_T}")
ax.set_title("Figure 9. Receivers: Combine braking against NFL separation, size held fixed", fontsize=11)
fig.tight_layout()
plt.show()

With size fixed, r = 0.01 (95% bootstrap CI -0.30 to 0.29, n = 59). Raw,
with top speed or draft class also fixed, or with every variable turned into ranks (an
outlier is visible in Figure 9), it stays between -0.01 and 0.07.

On its own this null is weak evidence. At n = 59 a single test needs |r| of 0.26
for p < 0.05 and finds a true correlation 80% of the time only above 0.36.
The score also repeats so poorly that its expected ceiling against separation is about
0.40 (up to 0.60 at the top of its repeatability interval). The receivers who
reached 100 separation plays scored about the same at the Combine as those who did not
(difference 0.16, -0.09 to 0.42, in score units), and their scores spread as
widely (SD 0.73 against 0.67 for all receivers), so the test set is not a
narrow slice of the score.

What the section does support is the measurement result above it. As measured here, the
stops in the position drills cannot rank players' change of direction, so they cannot carry
a Combine-to-NFL agility finding. For agility, use the official 3-cone and shuttle where
they exist and judge them at their smaller n, or show that a drill measure repeats within a
Combine year before testing it.

### The Combine year in version 1's numbers

The year effect made me recheck sections 2, 3 and 6 with every value centred within its
position group and its Combine year (draft class).

In [ ]:
YR = combine.draft_year


def agree_year(trk, ref):
    d = off[[trk, ref, "combine_position"]].join(YR).dropna()
    c = d[[trk, ref]] - d.groupby(["combine_position", "draft_year"])[[trk, ref]].transform("mean")
    return np.corrcoef(c[trk], c[ref])[0, 1]


YC_FLY, YC_F10 = agree_year("fly30", "fly_off"), agree_year("first10", "ten_yd_split")
RELY = {}
for m in METRICS40:
    w = pairs.pivot(index="nfl_id", columns="k", values=m).join(POS).join(YR).dropna()
    for p, g in w.groupby("combine_position"):
        gc = g[[0, 1]] - g.groupby("draft_year")[[0, 1]].transform("mean")
        RELY[(m, p)] = np.corrcoef(gc[0], gc[1])[0, 1]
RELY = pd.Series(RELY).unstack().loc[list(METRICS40), POS_ORDER]
BRY = RELY.loc[BURST].agg(["min", "max"])
PCY = []
pcd = best40[["vmax"]].join(OUT.top_speed).join(POS).join(YR).dropna()
for p in POS_ORDER:
    g = pcd[pcd.combine_position == p]
    gc = g[["vmax", "top_speed"]] - g.groupby("draft_year")[["vmax", "top_speed"]].transform("mean")
    lo, hi = boot_r(gc.vmax, gc.top_speed)
    PCY.append({"pos": p, "n": len(g), "v1 r": PC.loc[p, "r"], "year-centred r": np.corrcoef(gc.vmax, gc.top_speed)[0, 1],
                "lo": lo, "hi": hi})
PCY = pd.DataFrame(PCY).set_index("pos")
PCY_CLEAR = [p for p in POS_ORDER if PCY.loc[p, "lo"] > 0]
print(f"tracking vs official, r within group: 10 to 40 yd {AG.iloc[0].r_within:.3f} -> {YC_FLY:.3f} (year centred); "
      f"first 10 yd {AG.loc[AG.index[2], 'r_within']:.3f} -> {YC_F10:.3f}")
print("\nattempt 1 vs attempt 2, year centred (compare section 3):")
print(RELY.rename(index=METRICS40).round(3).to_string())
print("\nchange from section 3 (year centred minus version 1):")
print((RELY - REL[POS_ORDER]).rename(index=METRICS40).round(2).to_string())
print("burst metrics, range by group, year centred:\n" + BRY.round(2).to_string())
print("\npositive control, Combine top speed vs in-game top speed, year centred:")
print(PCY.round(3).to_string())
print(f"groups whose year-centred interval excludes zero: {PCY_CLEAR}")

The speed numbers barely move. The title's 10-to-40 agreement goes from 0.948 to
0.952, and top speed's attempt-to-attempt r changes by at most 0.02. The
first-10-yards agreement rises from 0.28 to 0.31. Some burst rows fall: for
receivers the four burst metrics repeat at -0.18 to 0.25 instead of
0.02 to 0.34, and for tight ends at -0.22 to
0.71 instead of 0.27 to 0.71. So part of what looked
like repeatable burst for receivers and tight ends was the Combine year, which makes section 3's
conclusion stronger, not weaker. In the positive control, defensive backs move most, from 0.24 to 0.29 (0.05 to 0.50), so the groups whose interval excludes zero become wide receivers, defensive backs and the defensive line, against wide receivers and the defensive line in version 1. Sections 1 to 6
keep their version 1 numbers because they answer the question as it was stated, within
position group. This table is the sensitivity check, and I chose it after seeing the year
effect, so read it as a check, not a new finding.

## 8. A per-player table you can join (new in version 2)

The cell below writes `combine_tracking_player_metrics.csv` to `/kaggle/working`: one row
per prospect, keyed by `nfl_id`. It holds values computed in this notebook and no raw
tracking rows. Each tracking metric comes with the attempt-to-attempt r of the player's
position group, with each Combine year centred, and a flag that is true when that r is 0.70
or higher, so you can see which metrics behave like traits before you use them. The metric
values themselves are not centred: centre them within `draft_year` and `combine_position`
before you compare players (checklist item 6).

| column | unit | meaning | section |
|---|---|---|---|
| `nfl_id` | | join key to `players.csv` and the game files | |
| `draft_year`, `combine_position` | | Combine year and position group, from `combine_results.csv` | |
| `fly30` | s | tracking 10-to-40-yard split, best attempt | 2, 3 |
| `full40` | s | tracking 40 time from 2 yd/s, best attempt | 2 |
| `first10` | s | tracking first 10 yards from 2 yd/s, best attempt | 2, 3 |
| `shuttle_trk`, `cone_trk` | s | tracking short shuttle and 3-cone, best timed attempt | 2 |
| `vmax` | yd/s | top speed in the 40, best attempt | 3 |
| `split5_10` | s | 5-to-10-yard split, best attempt | 3 |
| `acc2_8` | s | time from 2 to 8 yd/s, best attempt | 3 |
| `amax` | yd/s² | peak acceleration off the start, best attempt | 3 |
| `brake_score` | SD units | mean peak braking into the stop over 2+ position drills, each centred within drill and Combine year; receivers and defensive backs only | 7 |
| `<metric>_retest_r` | r | attempt 1 vs attempt 2 in the player's group, each Combine year centred (for `brake_score`: reliability of the drill average); blank if not testable | 7, 8 |
| `<metric>_repeats` | bool | `<metric>_retest_r` is 0.70 or higher; blank where the metric or the test is missing | 8 |
| `n_forty`, `n_shuttle`, `n_shuttle_timed`, `n_cone`, `n_cone_timed` | count | tracked attempts, and attempts my turn-finder could time | 2 |
| `n_stop_drills` | count | position drills with a usable stop (section 7 definition) | 7 |
| `n_reg_plays` | count | regular-season tracked plays | 4 |
| `n_outcome_plays` | count | regular-season plays with the group's main outcome (any tracked play for defensive backs) | 4 |
| `in_analysis_set` | bool | tracked 40 and 100+ outcome plays: the analysis set of sections 4 to 6 | 4 |

`full40`, `shuttle_trk` and `cone_trk` are not in section 3, so the cell computes their
attempt-to-attempt r the same way, within group, wherever at least 15 players have two
attempts (two timed attempts for the shuttle and 3-cone). Where fewer do, the r and the flag
are blank; the cell prints which.

In [ ]:
A = pd.DataFrame(index=combine.index)
A["draft_year"], A["combine_position"] = combine.draft_year, POS
for c in ["fly30", "full40", "first10", "vmax", "split5_10", "acc2_8", "amax"]:
    A[c] = best40[c]
A["shuttle_trk"], A["cone_trk"] = off.shuttle_trk, off.cone_trk
A["brake_score"] = COMP.brake_score


def retest_by_group(df, col, min_pairs=15):
    # attempt 1 vs attempt 2 within each position group (attempts in recorded order), each centred within its Combine year
    d = df.dropna(subset=[col]).copy()
    d["k"] = d.groupby("nfl_id").cumcount()
    w = d[d.k < 2].pivot(index="nfl_id", columns="k", values=col).dropna().join(POS).join(YR)
    out = {}
    for p, g in w.groupby("combine_position"):
        if len(g) >= min_pairs:
            gc = g[[0, 1]] - g.groupby("draft_year")[[0, 1]].transform("mean")
            out[p] = np.corrcoef(gc[0], gc[1])[0, 1]
    return out


RETEST = {m: RELY.loc[m, POS_ORDER].to_dict() for m in METRICS40}
RETEST["full40"] = retest_by_group(ev40.sort_values(["nfl_id", "attempt"]), "full40")
RETEST["shuttle_trk"] = retest_by_group(agil["SHORT_SHUTTLE"].rename(columns={"secs": "shuttle_trk"}), "shuttle_trk")
RETEST["cone_trk"] = retest_by_group(agil["THREE_CONE_DRILL"].rename(columns={"secs": "cone_trk"}), "cone_trk")
RETEST["brake_score"] = {p: REP.loc[(p, "brake", "drill and year"), "alpha"] for p in SCORE_GROUPS}
for c in ["fly30", "full40", "first10", "shuttle_trk", "cone_trk", "vmax", "split5_10", "acc2_8", "amax", "brake_score"]:
    r_ = POS.map(RETEST[c]).astype(float)
    A[f"{c}_retest_r"] = r_
    A[f"{c}_repeats"] = (r_ >= 0.70).astype("boolean").where(A[c].notna() & r_.notna())
A["n_forty"] = ev40.groupby("nfl_id").size()
for d_, nm in (("SHORT_SHUTTLE", "shuttle"), ("THREE_CONE_DRILL", "cone")):
    A[f"n_{nm}"] = agil[d_].groupby("nfl_id").size()
    A[f"n_{nm}_timed"] = agil[d_][agil[d_].status == "ok"].groupby("nfl_id").size()
A["n_stop_drills"] = N_DR
A["n_reg_plays"] = n_reg
A["n_outcome_plays"] = pd.Series({i: (n_out[POS[i]].get(i, 0) if POS[i] in n_out else n_reg.get(i, 0))
                                  for i in A.index})
cnt = [c for c in A.columns if c.startswith("n_")]
A[cnt] = A[cnt].fillna(0).astype(int)
A["in_analysis_set"] = A.index.isin(list(has40)) & (A.n_outcome_plays >= MIN_PLAYS)
assert len(A) == len(combine) and A.index.is_unique
assert int(A.in_analysis_set.sum()) == N_ANALYSIS, (int(A.in_analysis_set.sum()), N_ANALYSIS)
CSV_PATH = os.path.join(WORK, "combine_tracking_player_metrics.csv")
A.rename_axis("nfl_id").reset_index().to_csv(CSV_PATH, index=False, float_format="%.4f")
print(f"wrote {CSV_PATH}: {A.shape[0]} rows x {A.shape[1] + 1} columns")
print("\nretest r by group, Combine year centred (blank = fewer than 15 pairs, or not built for that group):")
print(pd.DataFrame(RETEST).T.reindex(columns=POS_ORDER).round(2).to_string())
print("\nplayers with a value, by group:")
print(A.groupby("combine_position")[["fly30", "vmax", "shuttle_trk", "cone_trk", "brake_score"]].count()
      .reindex(POS_ORDER).to_string())
print(f"\nrows in the analysis set: {int(A.in_analysis_set.sum())} (section 4: {N_ANALYSIS})")

### Use this table

In your own notebook, add this notebook as an input and read the CSV from its folder under
`/kaggle/input`. The five lines below join it to `players.csv` and keep the players whose
top speed repeats in their group and who have enough NFL plays.

In [ ]:
m = pd.read_csv(CSV_PATH)                       # in your notebook: the CSV's path under /kaggle/input
df = pd.read_csv(find_file("players.csv")).merge(m, on="nfl_id", how="inner")
keep = df[df.vmax_repeats.eq(True) & df.in_analysis_set]
print(len(df), "prospects joined;", len(keep), "with a top speed that repeats and 100+ outcome plays")
print(keep.groupby("combine_position").size().reindex(POS_ORDER).to_dict())

## 9. Does Combine speed show up in NFL playing time?

Sections 5 to 7 asked what a player does once he is on the field. Before that, a coach has to
put him there. For playing time the competition files give career totals, with no season split
and no team snap counts, and section 1 showed those totals are mostly exposure: a 2023 pick has
had three seasons, a 2025 pick one. So for this section I use
my public dataset, [BDB 2027 Public Enrichment](https://www.kaggle.com/datasets/dariushafshar/bdb-2027-public-enrichment).
It adds each player's season-by-season snaps, his team's snap counts and a games-started proxy
for the 2023 to 2025 rookie classes, built from public nflverse data and keyed on the competition's `nfl_id`.
It contains no competition data.

**The question.** Within each position group, do faster prospects play more as rookies? If
they do, is that only because faster players are drafted earlier and teams play their early
picks?

**I wrote these definitions down before computing any playing-time number:**

* **Speed:** tracking top speed in the 40, which repeats at r = 0.89 or more
  (section 3). The 10-to-40 split is a check.
* **Playing time:** his rookie-season snaps on his unit (offence for receivers, tight ends and
  offensive linemen; defence for defensive backs and defensive linemen), as a share of a full
  17-game season of his team's snaps. A player who never played counts as 0, so nobody drops
  out for not playing (one prospect has no public id and is left out). Starts (games on the field for his team's first snap) and his share over
  every season so far are checks.
* **Held fixed:** height, weight and draft class. Then, separately, draft position as well:
  the log of the overall pick, with undrafted players at pick 260 and an undrafted flag.
* **The test:** one per group, chosen in advance: speed against playing time with size and
  class held fixed, judged by the section 5 bars. Then the same with draft position held fixed
  too. Intervals are bootstrap 95% (4,000 resamples of players). "All groups" is the average
  of the five within-group values, weighted by players.

In [ ]:
RNG9 = np.random.default_rng(2027)       # section 9's own random stream; earlier sections are untouched
SEAS9 = pd.read_csv(find_file("rookie_class_seasons_2023_2025.csv"))
RC9 = pd.read_csv(find_file("rookie_classes_2023_2025.csv"))
RC9 = RC9[RC9.nfl_id.notna()].astype({"nfl_id": int}).set_index("nfl_id")
assert RC9.index.is_unique

# does the join hold? aggregate counts only
J9 = players.set_index("nfl_id")[["draft_year", "draft_overall_pick"]].join(
    RC9[["entry_year", "draft_number", "reg_off_snaps", "reg_def_snaps", "reg_st_snaps", "reg_games_first_snap"]],
    how="left")
found9 = J9.entry_year.notna()
JOIN9 = {"found": int(found9.sum()), "prospects": len(J9),
         "same_class": int((J9.entry_year == J9.draft_year)[found9].sum()),
         "drafted": int((found9 & J9.draft_overall_pick.notna()).sum()),
         "same_pick": int((J9.draft_overall_pick == J9.draft_number)[found9].sum()),
         "undrafted": int((found9 & J9.draft_overall_pick.isna()).sum()),
         "undrafted_both": int((found9 & J9.draft_overall_pick.isna() & J9.draft_number.isna()).sum())}
print("competition prospects in the dataset: {found} of {prospects}; same draft class: {same_class}; "
      "same overall pick: {same_pick} of {drafted} drafted; undrafted in both: {undrafted_both} of {undrafted}"
      .format(**JOIN9))
SNAPCHK9 = []
for unit_, comp_c, pack_c in (("offence", "career_offensive_snaps", "reg_off_snaps"),
                              ("defence", "career_defensive_snaps", "reg_def_snaps"),
                              ("special teams", "career_special_teams_snaps", "reg_st_snaps")):
    q9 = pd.DataFrame({"comp": career[comp_c].reindex(J9.index), "pack": J9[pack_c]})[found9].dropna()
    q9 = q9[q9.comp > 0]
    SNAPCHK9.append({"unit": unit_, "players with any": len(q9),
                     "within 1% or 1 snap": ((q9.pack - q9.comp).abs() <= np.maximum(1, 0.01 * q9.comp)).mean(),
                     "r": np.corrcoef(q9.comp, q9.pack)[0, 1], "sum ratio": q9.pack.sum() / q9.comp.sum()})
SNAPCHK9 = pd.DataFrame(SNAPCHK9).set_index("unit")
gs9 = pd.DataFrame({"comp": career.career_games_started.reindex(J9.index), "pack": J9.reg_games_first_snap})[found9].dropna()
STARTS1G9 = ((gs9.pack - gs9.comp).abs() <= 1).mean()
print("\ncareer regular-season snaps, dataset vs the competition's own career totals:")
print(SNAPCHK9.round(4).to_string())
print(f"games on the field for the first snap vs the competition's career games started: within 1 game for "
      f"{STARTS1G9:.1%} of {len(gs9)}")

# the outcomes, as fixed in advance: share of a full 17-game season on his unit; 0 if he never played
UNIT9 = {"WR": "off", "TE": "off", "OL": "off", "DB": "def", "DL": "def"}
s9 = SEAS9[SEAS9.nfl_id.isin(combine.index)].copy()
s9["nfl_id"] = s9.nfl_id.astype(int)
off9 = s9.nfl_id.map(POS).map(UNIT9).eq("off")
s9["u_snaps"] = np.where(off9, s9.off_snaps, s9.def_snaps)
s9["u_team"] = np.where(off9, s9.team_off_snaps, s9.team_def_snaps)
assert (s9.u_team[s9.games_with_snap > 0] > 0).all()
s9["share"] = np.where(s9.games_with_snap > 0, s9.u_snaps / (s9.u_team / s9.games_with_snap.clip(lower=1) * 17), 0.0)
s9["starts"] = s9.games_first_snap / 17
s9["pg_share"] = (s9.u_snaps / s9.u_team).where(s9.u_snaps > 0)        # the usual per-game share, players who played
rk9 = s9[s9.season == s9.entry_year].set_index("nfl_id")
assert rk9.index.is_unique and (rk9.season_number == 1).all()

F9 = pd.DataFrame(index=combine.index)
F9["pos"], F9["year"] = POS, combine.draft_year
F9["vmax"], F9["fly_neg"], F9["forty_neg"] = best40.vmax, -best40.fly30, -combine.forty
F9["height"], F9["weight"] = combine.combine_height, combine.combine_weight
pick9 = players.set_index("nfl_id").draft_overall_pick.reindex(F9.index)
F9["undrafted"] = pick9.isna().astype(float)
F9["log_pick"] = np.log(pick9.fillna(260))
F9["round"] = players.set_index("nfl_id").draft_round.reindex(F9.index).fillna(8)    # 8 = undrafted
for c9 in ("share", "starts", "pg_share"):
    F9[c9] = rk9[c9]
F9["played"] = (rk9.u_snaps > 0).astype(float)
F9["career"] = s9.groupby("nfl_id").share.mean()
F9["in_pack"] = F9.index.isin(rk9.index)
assert (s9.groupby("nfl_id").size().reindex(F9.index[F9.in_pack]) == 2026 - F9.year[F9.in_pack]).all()
print(f"\nrookie share: max {F9.share.max():.3f}, above 1: {int((F9.share > 1).sum())}")

SIZE9 = ["height", "weight"]


def design9(f, draft=None):
    # size + class dummies; draft = None, "log" (log pick + undrafted flag) or "round" (round dummies, post hoc)
    cols = [f[SIZE9].to_numpy(float), pd.get_dummies(f.year, drop_first=True).to_numpy(float)]
    if draft == "log":
        cols.append(f[["log_pick", "undrafted"]].to_numpy(float))
    elif draft == "round":
        cols.append(pd.get_dummies(f["round"], drop_first=True).to_numpy(float))
    return np.column_stack(cols)


def presid9(x, y, Z, idx=None):
    # partial r of x and y given Z; with idx (B x n resample indices), one value per resample
    X = np.c_[np.ones(len(x)), Z]
    if idx is None:
        idx = np.arange(len(x))[None, :]
    Xb, xb, yb = X[idx], x[idx], y[idx]
    P = np.linalg.pinv(np.einsum("bni,bnj->bij", Xb, Xb))
    rx = xb - np.einsum("bni,bi->bn", Xb, np.einsum("bij,bj->bi", P, np.einsum("bni,bn->bi", Xb, xb)))
    ry = yb - np.einsum("bni,bi->bn", Xb, np.einsum("bij,bj->bi", P, np.einsum("bni,bn->bi", Xb, yb)))
    return (rx * ry).sum(1) / np.sqrt((rx ** 2).sum(1) * (ry ** 2).sum(1))


def frame9(p, x_col, y_col, sub=None):
    f = F9[(F9.pos == p) & F9.in_pack].dropna(subset=[x_col, y_col] + SIZE9)
    return f[sub(f)] if sub is not None else f


IDX9, BOOTS9 = {}, {}


def resamples9(f, B=4000):
    # one set of resamples per group and player set, shared by every test on those players (so T and D pair up)
    key = (f.pos.iloc[0], tuple(f.index))
    if key not in IDX9:
        IDX9[key] = RNG9.integers(0, len(f), (B, len(f)))
    return IDX9[key]


def test9(x_col="vmax", y_col="share", draft=None, rank=False, sub=None, keep=None):
    # within each group: partial r and its bootstrap interval; "All" = the five values weighted by players
    est, boots, ns = {}, {}, {}
    for p in POS_ORDER:
        f = frame9(p, x_col, y_col, sub)
        x, y, Z = f[x_col].to_numpy(float), f[y_col].to_numpy(float), design9(f, draft)
        if rank:                       # ranks for speed, outcome, height, weight and (if used) log pick
            x, y = stats.rankdata(x), stats.rankdata(y)
            for j in [0, 1] + ([Z.shape[1] - 2] if draft == "log" else []):
                Z[:, j] = stats.rankdata(Z[:, j])
        est[p], ns[p] = presid9(x, y, Z)[0], len(f)
        boots[p] = presid9(x, y, Z, resamples9(f))
    tot = sum(ns.values())
    if keep:
        BOOTS9[keep] = sum(boots[p] * ns[p] for p in POS_ORDER) / tot
    res = {p: (est[p], *np.quantile(boots[p], [0.025, 0.975]), ns[p]) for p in POS_ORDER}
    res["All"] = (sum(est[p] * ns[p] for p in POS_ORDER) / tot,
                  *np.quantile(sum(boots[p] * ns[p] for p in POS_ORDER) / tot, [0.025, 0.975]), tot)
    return pd.DataFrame(res, index=["r", "lo", "hi", "n"]).T


raw9 = {}
for p in POS_ORDER:
    f = frame9(p, "vmax", "share")
    raw9[p] = (np.corrcoef(f.vmax, f.share)[0, 1], *boot_r(f.vmax, f.share, rng=RNG9), len(f))
R9 = {"raw": pd.DataFrame(raw9, index=["r", "lo", "hi", "n"]).T, "T": test9(keep="T"), "D": test9(draft="log", keep="D")}
MAIN9 = pd.concat(R9, axis=1)
K9 = {"T": 4, "D": 6}                     # covariates: height, weight, two class dummies (+ log pick, undrafted flag)
for p in POS_ORDER:
    n_ = int(R9["T"].loc[p, "n"])
    for e_ in ("T", "D"):
        MAIN9.loc[p, ("bar " + e_, "one test")] = r_crit(n_, k=K9[e_])
        MAIN9.loc[p, ("bar " + e_, "Bonferroni, 5")] = r_crit(n_, 0.05 / 5, k=K9[e_])
    MAIN9.loc[p, ("bar T", "80% power")] = r_min(n_ - K9["T"])            # Fisher z with k covariates: n - 3 - k
N9 = int(R9["T"].loc["All", "n"])
K9_ALL = 5 * (K9["T"] + 1) - 1           # five intercepts and five sets of covariate slopes, less one
MAIN9.loc["All", ("bar T", "one test")] = r_crit(N9, k=K9_ALL)
MAIN9.loc["All", ("bar T", "80% power")] = r_min(N9 - K9_ALL)
print("\nCombine top speed vs rookie-season snap share on his unit, within group (bootstrap 95% CI)")
print("T = height, weight and draft class held fixed (the test chosen in advance); D = T plus draft position")
print(MAIN9.round(3).to_string())
DROP9 = R9["T"].loc["All", "r"] - R9["D"].loc["All", "r"]
DROP9_CI = np.quantile(BOOTS9["T"] - BOOTS9["D"], [0.025, 0.975])
LEFT9_HI = np.quantile(BOOTS9["D"] / BOOTS9["T"], 0.975)
print(f"all groups, T minus D on the same resamples: {DROP9:.3f} [{DROP9_CI[0]:.3f}, {DROP9_CI[1]:.3f}]; "
      f"D / T at the top of its interval: {LEFT9_HI:.2f}")
zD9 = np.arctanh(R9["D"].loc[POS_ORDER, "r"].astype(float))
wD9 = R9["D"].loc[POS_ORDER, "n"].astype(float) - 3 - K9["D"]
Q9 = float((wD9 * (zD9 - (wD9 * zD9).sum() / wD9.sum()) ** 2).sum())
Q9_P = stats.chi2.sf(Q9, len(POS_ORDER) - 1)
print(f"do the five D values differ (post hoc, Cochran's Q on Fisher z)? Q = {Q9:.1f} on {len(POS_ORDER) - 1} df, p = {Q9_P:.3f}")

# variants planned in advance, then one chosen after seeing the result (marked)
VAR9 = pd.concat({
    "T, ranks": test9(rank=True), "D, ranks": test9(draft="log", rank=True),
    "T, 10-to-40 split": test9("fly_neg"), "D, 10-to-40 split": test9("fly_neg", draft="log"),
    "T, starts": test9(y_col="starts"), "D, starts": test9(y_col="starts", draft="log"),
    "T, career share": test9(y_col="career"), "D, career share": test9(y_col="career", draft="log"),
    "T, per game played (snaps > 0)": test9(y_col="pg_share"),
    "D, per game played (snaps > 0)": test9(y_col="pg_share", draft="log"),
    "D, draft round as categories (post hoc)": test9(draft="round"),
    "T, official 40 time (post hoc)": test9("forty_neg"), "D, official 40 time (post hoc)": test9("forty_neg", draft="log"),
    "T, full-season share, players who played (post hoc)": test9(sub=lambda f: f.pg_share.notna()),
    "T, got a snap at all (post hoc)": test9(y_col="played"),
}, names=["variant", "pos"])
VA9 = VAR9.xs("All", level="pos")
print("\nvariants, r by group")
print(VAR9.r.unstack().reindex(columns=POS_ORDER + ["All"]).round(2).to_string())
print("\nvariants, all groups (95% CI)")
print(VAR9.xs("All", level="pos").round(3).to_string())
PLANNED9 = [v for v in VAR9.index.get_level_values(0).unique() if "per game" not in v and "post hoc" not in v]
VT9 = VA9.loc[[v for v in PLANNED9 if v.startswith("T")]]
VD9 = VA9.loc[[v for v in PLANNED9 if v.startswith("D")]]
print(f"planned variants, all groups: T {VT9.r.min():.2f} to {VT9.r.max():.2f} (lowest lower limit {VT9.lo.min():.2f}); "
      f"D {VD9.r.min():.2f} to {VD9.r.max():.2f} (intervals {VD9.lo.max():.2f} or lower to {VD9.hi.min():.2f} or higher)")

# the two halves of the path, speed -> draft position -> playing time, on the same players as T
has_v9 = lambda f: f.vmax.notna()
PATH9 = pd.concat({"speed vs log pick": test9("vmax", "log_pick"),
                   "log pick vs share": test9("log_pick", "share", sub=has_v9)}, names=["path", "pos"])
print("\nthe path, size and class held fixed (95% CI)")
print(PATH9.r.unstack().reindex(columns=POS_ORDER + ["All"]).round(2).to_string())
P1_9, P2_9 = PATH9.loc[("speed vs log pick", "All"), "r"], PATH9.loc[("log pick vs share", "All"), "r"]
print(f"if all of speed's link ran through draft position: {P1_9:.2f} x {P2_9:.2f} = {P1_9 * P2_9:.2f} "
      f"(observed T {R9['T'].loc['All', 'r']:.2f})")

# selection: who has no snaps, who has no tracked 40
pk9 = F9[F9.in_pack]
ZERO9 = (pk9.share == 0).mean()
print(f"\nprospects with no rookie snap on their unit: {ZERO9:.0%} "
      f"({', '.join(f'{p} {v:.0%}' for p, v in (pk9.share == 0).groupby(pk9.pos).mean().reindex(POS_ORDER).items())})")
hv9 = pk9.vmax.notna()
dev9 = pk9.share - pk9.groupby("pos").share.transform("mean")
SEL9 = {"without": int((~hv9).sum()), "diff": dev9[~hv9].mean() - dev9[hv9].mean(),
        "se": np.sqrt(dev9[~hv9].var() / (~hv9).sum() + dev9[hv9].var() / hv9.sum()),
        "rd12_without": (pk9["round"] <= 2)[~hv9].mean(), "rd12_with": (pk9["round"] <= 2)[hv9].mean(),
        "udfa_without": pk9.undrafted[~hv9].mean(), "udfa_with": pk9.undrafted[hv9].mean()}
print(f"no tracked 40 ({SEL9['without']}) minus tracked 40, rookie share, group-centred: {SEL9['diff']:.3f} "
      f"[{SEL9['diff'] - 1.96 * SEL9['se']:.3f}, {SEL9['diff'] + 1.96 * SEL9['se']:.3f}]; rounds 1-2: "
      f"{SEL9['rd12_without']:.0%} vs {SEL9['rd12_with']:.0%}; undrafted {SEL9['udfa_without']:.0%} vs {SEL9['udfa_with']:.0%}")

# the defensive line, checked after seeing its result
fdl9 = frame9("DL", "vmax", "share")
xdl9, ydl9, Zdl9 = fdl9.vmax.to_numpy(float), fdl9.share.to_numpy(float), design9(fdl9, "log")
LOO9 = np.array([presid9(np.delete(xdl9, i), np.delete(ydl9, i), np.delete(Zdl9, i, 0))[0] for i in range(len(xdl9))])
DL_WV9 = np.corrcoef(fdl9.weight, fdl9.vmax)[0, 1]
EDGE9 = players.set_index("nfl_id").nfl_position.reindex(fdl9.index).isin(["DE", "OLB"]).astype(float).to_numpy()
DLROB9 = {}
for lab_, Zx_ in (("weight squared added", np.c_[Zdl9, (fdl9.weight.to_numpy(float) - fdl9.weight.mean()) ** 2]),
                  ("edge-rusher flag added", np.c_[Zdl9, EDGE9])):
    DLROB9[lab_] = (presid9(xdl9, ydl9, Zx_)[0], *np.quantile(presid9(xdl9, ydl9, Zx_, resamples9(fdl9)), [0.025, 0.975]))
DLROB9 = pd.DataFrame(DLROB9, index=["r", "lo", "hi"]).T
print(f"\ndefensive line (post hoc): edge rushers (DE, OLB) {EDGE9.mean():.0%} of the group; D with:")
print(DLROB9.round(3).to_string())
TOP9 = fdl9.iloc[int(LOO9.argmin())]
TOP9_POS = players.set_index("nfl_id").nfl_position[TOP9.name]
TOP9_RANK = int((fdl9.vmax > TOP9.vmax).sum()) + 1
print(f"weight vs top speed r {DL_WV9:.2f}; D leaving out one player at a time: "
      f"{LOO9.min():.2f} to {LOO9.max():.2f}")
print(f"the player whose removal moves it most: NFL position {TOP9_POS}, {TOP9.weight:.0f} lb, top speed "
      f"{TOP9_RANK} of {len(fdl9)} in the group, rookie snap share {TOP9.share:.2f}")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14.5, 5.2), gridspec_kw={"width_ratios": [1.15, 1]})
ax = axs[0]
rows9 = POS_ORDER + ["All"]
yy9 = np.arange(len(rows9))[::-1] * 1.0
for off_, (e_, col_, lab_) in zip((0.24, 0.0, -0.24), (("raw", MUTED, "raw (nothing held fixed)"),
                                                       ("T", BLUE, "size and draft class held fixed (the test)"),
                                                       ("D", ORANGE, "... and draft position held fixed"))):
    t_ = R9[e_].reindex(rows9)
    ok_ = t_.r.notna()
    ax.hlines(yy9[ok_] + off_, t_.lo[ok_], t_.hi[ok_], color=col_, lw=2)
    ax.plot(t_.r[ok_], yy9[ok_] + off_, "o", color=col_, ms=6.5, mec=SURFACE, label=lab_)
for y_, p in zip(yy9, rows9):
    b_ = MAIN9.loc[p, ("bar T", "one test")]
    ax.plot([-b_, b_], [y_ + 0.45, y_ + 0.45], "|", color=INK2, ms=8, mew=1.5)
ax.axvline(0, color=MUTED, lw=0.8)
ax.axhline(yy9[-1] + 0.6, color=GRID, lw=1)
ax.set_yticks(yy9)
ax.set_yticklabels([f"{POS_NAME[p]} (n {int(R9['T'].loc[p, 'n'])})" for p in POS_ORDER] + [f"All groups (n {N9})"])
ax.set_xlim(-0.6, 0.8); ax.set_ylim(-0.6, yy9[0] + 0.7)
ax.set_xlabel("r with rookie-season snap share (95% bootstrap CI); ticks = one-test bar")
ax.legend(loc="upper center", bbox_to_anchor=(0.45, -0.13), ncol=2)
ax.grid(axis="y", visible=False)
ax.set_title("Speed and playing time, within group")

ax = axs[1]
hf9 = F9[F9.in_pack & F9.vmax.notna()].copy()
hf9["fast"] = hf9.vmax > hf9.groupby(["pos", "year"]).vmax.transform("median")
HALF9 = hf9.groupby(["round", "fast"]).share.agg(["mean", "size"]).unstack()
code9 = ((hf9["round"].astype(int) - 1) * 2 + hf9.fast.astype(int)).to_numpy()
sh9, fa9 = hf9.share.to_numpy(float), hf9.fast.to_numpy()


def gaps9(ix):
    # faster minus slower half: overall, and within draft round (rounds weighted by players)
    s_ = np.bincount(code9[ix], weights=sh9[ix], minlength=16).reshape(8, 2)
    c_ = np.bincount(code9[ix], minlength=16).reshape(8, 2)
    ok_ = (c_ > 0).all(1)
    m_ = s_[ok_] / c_[ok_]
    return sh9[ix][fa9[ix]].mean() - sh9[ix][~fa9[ix]].mean(), np.average(m_[:, 1] - m_[:, 0], weights=c_[ok_].sum(1))


GAP_ALL9, GAP_ROUND9 = gaps9(np.arange(len(hf9)))
GAPB9 = np.array([gaps9(RNG9.integers(0, len(hf9), len(hf9))) for _ in range(2000)])
GAP_ALL9_CI, GAP_ROUND9_CI = np.quantile(GAPB9[:, 0], [0.025, 0.975]), np.quantile(GAPB9[:, 1], [0.025, 0.975])
xr9 = HALF9.index.to_numpy(float)
for fast_, col_, lab_ in ((True, BLUE, "faster half of his group and class"), (False, MUTED, "slower half")):
    ax.plot(xr9, HALF9[("mean", fast_)], "o-", color=col_, lw=2, ms=6, mec=SURFACE, label=lab_)
ax.set_xticks(xr9); ax.set_xticklabels([f"{int(r_)}" if r_ < 8 else "undr." for r_ in xr9])
ax.set_xlabel("draft round"); ax.set_ylabel("rookie-season snap share on his unit (mean)")
ax.set_ylim(0, None)
ax.legend(loc="upper right")
ax.set_title("At the same round, fast and slow play about the same")
fig.suptitle("Figure 10. Combine top speed and rookie playing time: most of the link goes with draft position", x=0.01,
             ha="left", fontsize=12, fontweight="bold", color=INK)
fig.tight_layout()
plt.show()
print("mean rookie share by draft round (8 = undrafted), faster vs slower half (players):")
print(HALF9.round(2).to_string())
print(f"faster minus slower half: {GAP_ALL9:.3f} [{GAP_ALL9_CI[0]:.3f}, {GAP_ALL9_CI[1]:.3f}] overall; {GAP_ROUND9:.3f} "
      f"[{GAP_ROUND9_CI[0]:.3f}, {GAP_ROUND9_CI[1]:.3f}] within the same round (weighted by players; 2,000 resamples)")

**The join holds.** 509 of 510 prospects are in the dataset (the other one is not on any
nflverse roster). All 509 are in the same draft class as in the competition
files, and the overall pick matches for 383 of 383 drafted players. Its
regular-season snap totals match the competition's own career totals within 1% (or one snap) for
97.7% of the 213 players with offensive snaps and 95.3% of the
193 with defensive snaps, and the games-started proxy is within one game for
99.6%.

**Speed shows up in playing time.** With size and class held fixed, faster prospects played more
as rookies: r = 0.25 (0.16 to 0.34) across all five groups, n =
417. By group, it is 0.40 for the defensive line (0.20 to 0.59),
0.32 for the offensive line (0.12 to 0.50) and 0.22 for defensive
backs (0.04 to 0.39). The first two clear the Bonferroni bar for five tests;
the defensive backs clear only the one-test bar. For wide receivers it is 0.18 and the
interval includes zero (-0.02 to 0.37). For tight ends (-0.07, n =
34) a test would need a true r near 0.49 to find it 80% of the time, so that
row rules out only a strong positive link (its interval tops out at 0.25).

**Most of it cannot be told apart from draft position.** Hold draft position fixed as well, and
the five-group value falls to 0.04 (-0.06 to 0.14). Measured on the
same resamples, the drop is 0.21 (0.13 to 0.28), so at least half of the
link goes with draft position. For wide receivers, tight ends, defensive backs and offensive
linemen the value is between -0.13 and 0.05, and every one of those
intervals includes zero. Faster prospects were drafted earlier (r = -0.30 with the log of
the pick, size and class fixed), and draft position predicts rookie playing time much more
strongly than speed does (r = -0.71, against at most 0.40 for speed in any group).
If all of speed's link went with draft position, those two numbers would predict about
0.21; the observed value is 0.25.

Figure 10 adds a check without a model. The faster half of each group and class played
0.08 more of a season's snaps than the slower half (0.03 to
0.14); within the same draft round the gap is -0.01 (-0.06 to
0.03). The halves are not adjusted for size, so this view is cruder than the
numbers above.

The planned checks agree. With ranks, the 10-to-40 split, starts or career share in place of
the main choices, the five-group value is 0.21 to 0.25 with size and class fixed
(every interval above zero) and -0.03 to 0.04 with draft position fixed (every
interval includes zero). Two checks I added afterwards agree too. Draft round as categories,
instead of the log of the pick, gives 0.06 (-0.05 to 0.17). The
official 40 time, instead of the tracking, gives 0.28 with size and class fixed and
0.01 with draft position fixed as well.

**The defensive line is the exception, and I do not trust it yet.** With draft position fixed it
stays at 0.35 (0.12 to 0.57), which clears the Bonferroni bar for five
tests (0.27). The five groups' values with draft position fixed also differ by
more than chance alone would explain (Q = 12.9 on 4 degrees of freedom, p = 0.012, a test I added afterwards). But it
fails one of my own planned checks: with ranks it falls to 0.18 (-0.03 to
0.39). I looked at why only after seeing this. Within the defensive line, weight and
top speed correlate at -0.83, so "speed with weight held fixed" is a narrower slice of the
spread, and the raw value is only 0.13. Weight may also stand in for role, since
56% of the group are edge rushers. Adding weight squared gives 0.36
(0.13 to 0.57), and adding an edge-rusher flag gives 0.31
(0.05 to 0.53). One player moves it most, an edge rusher with
the fastest top speed of the 93, who played 85% of a season's defensive snaps
as a rookie. Leaving out one player at a time gives 0.26 to 0.40. So it is
a lead: a defensive lineman who is fast for his weight may play more than his draft slot
predicts. Another rookie class could test it.

**What holding draft position fixed means.** It is not a nuisance variable. Faster players are
drafted earlier. Part of that may be teams paying for speed, which would put draft position on
the path from speed to playing time. Part may be that speed comes with other things teams value,
which makes draft position a stand-in for those. Holding it fixed answers a narrower question:
given where a team drafted him, does his speed tell you anything more? It does not measure what
speed itself is worth. It can even push the answer below zero, because a faster player taken at
the same pick was probably taken there partly for his speed, so he may have less of everything
else. The defensive backs' value with draft position fixed (-0.13) fits that pattern, but I
did not predict that sign in advance. This data also cannot say why teams play their early picks:
because those players are better, or because the team paid for them. Staw and Hoang (1995) asked
that sunk-cost question about the NBA draft.

**Selection.** Of all the prospects, 25% never played a snap on their unit as rookies.
They count as 0 rather than dropping out. Keeping only the players who played (n =
317), the same full-season share gives 0.19 and the usual per-game share
gives 0.16, against 0.25 for everyone. Whether a player got a snap at all gives
0.21 on its own. So part of speed's link is whether a player gets on the field. (I planned
the per-game share and added the other two afterwards.) The 92 prospects without a
tracked 40 played about as much as those with one (difference 0.02 of a season's snaps,
-0.05 to 0.09). More of them went in the first two rounds (24%
against 18%) and more went undrafted (30% against
24%), so the test set is a little light on both ends of the draft.

## 10. Test your own metric

Sections 2 to 9 ran the same checks on every Combine metric I looked at. This section runs them on
yours. You write your metric in the cell near the top of the notebook, under the figure. The next cell
holds the scorecard's code and ends with a check against sections 7 and 9; you do not need to read it.
The cell after it prints your scorecard. To try another metric without rerunning everything, edit and
run the cell at the top, then run that last cell again. The rest of the notebook has to have run once
first, because the scorecard uses its data and functions.

`my_metric` gets one drill attempt at a time and returns one number. `MY_BEST` turns a player's
attempts into one value: for a time, his best is the minimum. You can also set `MY_DRILLS = None`
and write the metric from the per-player table of section 8; a comment in the cell shows how.

The scorecard answers four questions with the rules used above:

1. **Does it repeat?** Attempt 1 against attempt 2 of the same drill, as in section 3. A group that
   ran two or more of your drills gets one drill against another instead, as in section 7. Each
   Combine year is centred. The flag is section 8's r = 0.70, applied to the same r with height and
   weight held fixed, because that is the part question 4 tests. (A score built mostly from weight
   would otherwise look repeatable just because weight is.) Top speed is shown for comparison.
2. **How many players can it be tested on?** Section 4's count (100+ plays with the group's main
   outcome), and the players tested against the outcome you chose. For snap share that is section
   9's rule: everyone in my public dataset, [BDB 2027 Public Enrichment](https://www.kaggle.com/datasets/dariushafshar/bdb-2027-public-enrichment),
   counts, with 0 for a player who never played. It adds each player's season-by-season NFL snaps,
   which the competition files give only as career totals.
3. **What correlation would it need?** Section 5's one-test bar and the true r a test finds 80% of
   the time, at the tested n and counting the variables held fixed. A Bonferroni bar splits
   p < 0.05 over five groups times `MY_TRIES`.
4. **Does it say anything beyond size and draft position?** Section 9's test. Height, weight and
   draft class are held fixed, then draft position as well (and, for defensive-line get-off,
   alignment, as in section 6). Intervals come from 4,000 bootstrap resamples of players, and a
   version on ranks checks whether a few players drive the result. A value counts only if its
   interval excludes zero, it clears the Bonferroni bar and the rank version agrees.

The example is the tracking's first 10 yards from section 2, the classic burst number, against
rookie snap share.

In [ ]:
# section 10's machinery: the rules of sections 3 to 9 as one function, scorecard(). You do not need to read it.
# It ends with a check against sections 7 and 9. The cell after this one prints your scorecard.
FR10 = pl.assign(sec=(pl.t - pl.groupby("event_id").t.transform("first")).dt.total_seconds())
T40 = (ev40[["nfl_id", "attempt", "fly30", "full40", "first10", "vmax", "split5_10", "acc2_8", "amax"]]
       .join(combine, on="nfl_id")
       .join(A[["shuttle_trk", "cone_trk", "brake_score"] + [c for c in A if c.startswith("n_")]], on="nfl_id"))
MIN10, BAR10, BREP10 = 20, 0.70, 2000   # no estimate below 20 players; section 8's repeat flag; resamples in Q1
OUT10 = {"share": "rookie-season snap share (section 9)", "top_speed": "in-game top speed (section 5)",
         "main": "each group's main outcome (section 4)"}


def attempts10(metric, drills):
    # one value per drill attempt; an attempt where the metric fails counts as missing (and is counted)
    errs = []
    if drills is None:                   # the per-player table route: one row per 40 attempt
        v = pd.to_numeric(pd.Series(metric(T40), index=T40.index), errors="coerce")
        V = T40[["nfl_id", "attempt"]].assign(drill_name="FORTY_YARD_DASH", value=v)
    else:
        unknown = sorted(set(drills) - set(FR10.drill_name))
        if unknown:
            raise ValueError(f"no attempts for {unknown}; drill names: {sorted(FR10.drill_name.unique())}")
        rows = []
        for _, f in FR10[FR10.drill_name.isin(drills)].groupby("event_id", sort=False):
            try:
                v = np.asarray(metric(f), dtype=float).item()
            except Exception as e:
                v = np.nan
                errs.append(f"{type(e).__name__}: {e}")
            rows.append((f.nfl_id.iloc[0], f.drill_name.iloc[0], f.attempt.iloc[0], v))
        V = pd.DataFrame(rows, columns=["nfl_id", "drill_name", "attempt", "value"])
    V["value"] = V.value.astype(float).where(np.isfinite(V.value.astype(float)))
    return V.assign(pos=V.nfl_id.map(POS), year=V.nfl_id.map(YR)), errs


def free10(v):
    # the part of a player-level series (index nfl_id) that height, weight and Combine year do not explain
    d_ = pd.DataFrame({"v": v}).join(combine[["combine_height", "combine_weight", "draft_year"]]).dropna()
    X_ = np.c_[np.ones(len(d_)), d_[["combine_height", "combine_weight"]].to_numpy(float),
               pd.get_dummies(d_.draft_year, drop_first=True).to_numpy(float)]
    return d_.v - X_ @ np.linalg.lstsq(X_, d_.v.to_numpy(float), rcond=None)[0]


def repeat10(V, rng, B=BREP10):
    # Q1 per group. With 2+ drills that 20+ of the group ran: one drill against another (each player's first
    # attempt with a value), mean r over pairs of drills, as in section 7. Otherwise attempt 1 against attempt
    # 2 of the same drill, as in section 3. Values are centred within drill and Combine year. The flag uses the
    # same r with height and weight held fixed, the part of the metric that question 4 tests.
    res = {}
    for p in POS_ORDER:
        v = V[V.pos == p].sort_values(["nfl_id", "drill_name", "attempt"])
        first = v.dropna(subset=["value"]).groupby(["nfl_id", "drill_name"]).head(1)
        z = first.groupby(["drill_name", "year"]).value.transform(lambda s_: (s_ - s_.mean()) / s_.std())
        W = first.assign(z=z).pivot(index="nfl_id", columns="drill_name", values="z")
        W = W.loc[:, W.notna().sum() >= MIN10]
        mean_r = lambda M: np.nanmean([r_ for r_, n_ in pair_rs(np.asarray(M, float)) if n_ >= MIN10] or [np.nan])
        if W.shape[1] >= 2 and np.isfinite(mean_r(W)):
            M = W.to_numpy(float)
            ci = np.nanquantile([mean_r(M[rng.integers(0, len(M), len(M))]) for _ in range(B)], [0.025, 0.975]) \
                if B else (np.nan, np.nan)
            res[p] = ("drill to drill", int((W.notna().sum(1) >= 2).sum()), mean_r(M), *ci,
                      mean_r(W.apply(free10).reindex(W.index)))
            continue
        a = v.groupby(["nfl_id", "drill_name"]).head(2)
        a = a.assign(k=a.groupby(["nfl_id", "drill_name"]).cumcount())
        w = a.pivot(index=["nfl_id", "drill_name"], columns="k", values="value").reindex(columns=[0, 1]).dropna()
        w = w.join(a.groupby(["nfl_id", "drill_name"]).year.first())
        c = (w[[0, 1]] - w.groupby(["drill_name", "year"])[[0, 1]].transform("mean")).groupby(level="nfl_id").head(1)
        c = c.droplevel("drill_name")                      # one pair per player
        if len(c) < MIN10 or (c[0] - c[1]).abs().max() < 1e-12:
            res[p] = ("not testable" if len(c) < MIN10 else "same every attempt", len(c), np.nan, np.nan, np.nan, np.nan)
            continue
        cf = pd.concat([free10(c[0]), free10(c[1])], axis=1, keys=[0, 1]).dropna()
        res[p] = ("attempt 1 vs 2", len(c), np.corrcoef(c[0], c[1])[0, 1],
                  *(boot_r(c[0], c[1], B=B, rng=rng) if B else (np.nan, np.nan)), np.corrcoef(cf[0], cf[1])[0, 1])
    return pd.DataFrame(res, index=["check", "players", "r", "lo", "hi", "r_free"]).T.astype(
        {"r": float, "lo": float, "hi": float, "r_free": float})


def players10(V, best):
    # one value per player: his best (or mean) attempt in each drill; with several drills, each drill is
    # standardised within group and Combine year, then averaged
    per = V.dropna(subset=["value"]).groupby(["nfl_id", "drill_name", "pos", "year"]).value.agg(best).reset_index()
    if per.drill_name.nunique() > 1:
        per["value"] = per.groupby(["pos", "drill_name", "year"]).value.transform(lambda s_: (s_ - s_.mean()) / s_.std())
    return per.groupby("nfl_id").value.mean()


def frame10(x, outcome, p):
    # the players a group can be tested on: section 9's rule for snap share (in the dataset; 0 if he never
    # played), section 4's for the others (100+ plays with the outcome); height and weight known
    f = F9[F9.pos == p].assign(x=x, interior=dl.interior)
    if outcome == "share":
        f = f[f.in_pack].assign(y=lambda d_: d_.share)
    else:
        f = f.assign(y=OUT[OUTCOMES[p][0] if outcome == "main" else "top_speed"])
    extra = ["interior"] if outcome == "main" and p == "DL" else []     # where he lines up (section 6)
    return f.dropna(subset=["x", "y"] + SIZE9 + extra), extra


def partials10(f, extra, rng, B):
    # Q4 for one group: size and class fixed (T); draft position fixed too (D); D on ranks. Same resamples for all.
    fr = f.assign(**f[["x", "y", "height", "weight", "log_pick"] + extra].rank())
    idx = rng.integers(0, len(f), (B, len(f))) if B else None
    out = {}
    for e_, g, draft in (("T", f, None), ("D", f, "log"), ("D, ranks", fr, "log")):
        X_ = np.c_[np.ones(len(g)), design9(g, draft), g[extra].to_numpy(float)]
        x, y = g.x.to_numpy(float), g.y.to_numpy(float)
        left = np.std(x - X_ @ np.linalg.lstsq(X_, x, rcond=None)[0]) / np.std(x)   # spread of the metric still free
        ok = left > 1e-6                     # a metric the covariates explain fully has nothing left to test
        out[e_] = (presid9(x, y, X_[:, 1:])[0] if ok else np.nan,
                   presid9(x, y, X_[:, 1:], idx) if ok and B else np.full(1, np.nan), X_.shape[1] - 1, left)
    return out


def scorecard(metric, drills, best, outcome, tries=1, label="my metric", show=True, B=4000, seed=2027):
    assert best in ("min", "max", "mean", "median") and outcome in OUT10 and tries >= 1, (best, outcome, tries)
    rng = np.random.default_rng(seed)        # its own stream, so the same metric always gets the same card
    V, errs = attempts10(metric, drills)
    rep, x = repeat10(V, rng, BREP10 if B else 0), players10(V, best)
    cov, Q4, boots = {}, {}, {}
    for p in POS_ORDER:
        ids = POS.index[POS == p]
        has = x.reindex(ids).notna()
        f, extra = frame10(x, outcome, p)
        cov[p] = {"prospects": len(ids), "with your metric": int(has.sum()),
                  "... and 100+ main-outcome plays (s. 4)": int((has & (A.n_outcome_plays.reindex(ids) >= MIN_PLAYS)).sum()),
                  "tested here": len(f)}
        if len(f) < MIN10 or f.x.nunique() < 3:
            continue
        pr = partials10(f, extra, rng, B)
        n_, k_ = len(f), pr["D"][2]
        Q4[p] = {"n": n_, "k": k_, "one test": r_crit(n_, k=k_), "Bonferroni": r_crit(n_, 0.05 / (5 * tries), k=k_),
                 "80% power": r_min(n_ - k_), "left": pr["D"][3]}
        for e_ in ("T", "D", "D, ranks"):
            Q4[p].update({e_: pr[e_][0], e_ + " lo": np.nanquantile(pr[e_][1], 0.025),
                          e_ + " hi": np.nanquantile(pr[e_][1], 0.975)})
        boots[p] = {e_: pr[e_][1] for e_ in ("T", "D", "D, ranks")}
    Q4 = pd.DataFrame(Q4, dtype=float).T
    pool = pool10(Q4)
    if outcome != "main" and pool:           # one outcome for every group: pool the groups as section 9 does
        q, w_ = Q4.loc[pool], Q4.n[pool] / Q4.n[pool].sum()
        N_, K_ = int(q.n.sum()), int((q.k + 1).sum() - 1)
        all_ = {"n": N_, "k": K_, "one test": r_crit(N_, k=K_), "Bonferroni": r_crit(N_, 0.05 / tries, k=K_),
                "80% power": r_min(N_ - K_)}
        for e_ in ("T", "D", "D, ranks"):
            b_ = sum(boots[p][e_] * w_[p] for p in pool)
            all_.update({e_: (q[e_] * w_).sum(), e_ + " lo": np.nanquantile(b_, 0.025), e_ + " hi": np.nanquantile(b_, 0.975)})
        Q4.loc["All"] = all_
    Q4["verdict"] = [word10(Q4.loc[p]) for p in Q4.index]
    card = {"label": label, "drills": drills, "best": best, "outcome": outcome, "tries": tries, "n_att": len(V),
            "n_val": int(V.value.notna().sum()), "errs": errs, "B": B, "rep": rep, "x": x, "cov": pd.DataFrame(cov).T, "Q4": Q4}
    card["verdict"] = verdict10(card)
    if show:
        show10(card)
    return card


def word10(q):
    # a value counts only if its interval excludes zero, it clears the Bonferroni bar (five groups x MY_TRIES; for
    # the pooled value, MY_TRIES) and the rank version agrees in sign with an interval that also excludes zero
    if not np.isfinite(q["D lo"]):
        return "no estimate"
    out = q["D lo"] > 0 or q["D hi"] < 0
    ranks = np.sign(q["D, ranks lo"]) == np.sign(q["D, ranks hi"]) == np.sign(q.D)
    if out and abs(q.D) > q.Bonferroni:
        w = "clears every bar" if ranks else "clears the bar, not with ranks"
    elif out and abs(q.D) > q["one test"]:
        w = "one-test bar only"
    else:
        w = "not told from noise"
    return w + (" (small n)" if q["80% power"] >= 0.45 else "")


def pool10(Q4):
    return [p for p in Q4.index if p != "All" and np.isfinite(Q4.loc[p, "T"]) and np.isfinite(Q4.loc[p, "D"])]


def names10(ps):
    ps = list(ps)
    return ps[0] if len(ps) == 1 else ", ".join(ps[:-1]) + " and " + ps[-1]


def verdict10(c):
    rep, Q4, o = c["rep"], c["Q4"], OUT10[c["outcome"]].split(" (")[0]
    yes = [p for p in POS_ORDER if rep.loc[p, "r_free"] >= BAR10]
    no = [p for p in POS_ORDER if rep.loc[p, "r_free"] < BAR10]
    if yes:
        lines = [f"With height and weight held fixed, it repeats like a trait (r of {BAR10:.2f} or more) for "
                 f"{names10(yes)}" + (f", not for {names10(no)}." if no else ".")]
    elif no:
        top = rep.r_free.idxmax()
        lines = [f"With height and weight held fixed, it does not repeat like a trait (r of {BAR10:.2f} or more) in "
                 f"any group tested; the closest is {top} at {rep.loc[top, 'r_free']:.2f}."]
    else:
        lines = ["Its repeat cannot be tested here (too few players, or it does not change between attempts)."]
    untested = [p for p in POS_ORDER if not np.isfinite(rep.loc[p, "r_free"])]
    if untested and (yes or no):
        lines[0] = lines[0][:-1] + f" (not testable for {names10(untested)})."
    if not len(Q4):
        return lines + [f"No group has {MIN10} players with a value and the outcome, so there is nothing to test."]
    if "All" in Q4.index:
        a = Q4.loc["All"]
        pooled = "" if len(pool10(Q4)) == 5 else f" (pooled over {names10(pool10(Q4))} only)"
        lines.append(f"Beyond size and draft position, its link to {o} " + (
            a.verdict.split(" (")[0] if a.verdict.startswith("clears") else "cannot be told from noise")
            + f": r = {a.D:.2f} ({a['D lo']:.2f} to {a['D hi']:.2f}), n = {int(a.n)}{pooled}. With only size and "
            f"class fixed it is {a['T']:.2f} ({a['T lo']:.2f} to {a['T hi']:.2f}).")
    elif c["outcome"] == "main":
        lines.append("The main outcome differs by group, so there is no pooled value.")
    grp = [p for p in POS_ORDER if p in Q4.index]
    good = [p for p in grp if Q4.loc[p, "verdict"].startswith("clears every bar")]
    weak = [p for p in grp if Q4.loc[p, "verdict"].startswith("clears the bar, not with ranks")]
    if good:
        lines.append("Groups where it clears every bar: " + "; ".join(f"{p} ({Q4.loc[p, 'D']:.2f})" for p in good)
                     + ". Test them on another draft class before building on them.")
    if weak:
        lines.append("Clears the Bonferroni bar but not with ranks, so a few players may drive it: "
                     + "; ".join(f"{p} ({Q4.loc[p, 'D']:.2f})" for p in weak) + ".")
    if not good and not weak:
        lines.append("No group clears its Bonferroni bar.")
    small = [p for p in grp if Q4.loc[p, "80% power"] >= 0.45]
    if small:
        lines.append("Too few players to rule out a moderate link: " + "; ".join(
            f"{p} (n {int(Q4.loc[p, 'n'])}: only a true r of {Q4.loc[p, '80% power']:.2f} or more is found 80% of the time)"
            for p in small) + ".")
    thin = [p for p in grp if Q4.loc[p, "left"] < 0.30]
    if thin:
        lines.append(f"In {names10(thin)}, size, class and draft position explain most of the metric (SD left below "
                     "0.30), so the test there sees only a narrow slice of it.")
    gone = [p for p in POS_ORDER if p not in grp]
    if gone:
        lines.append(f"Not tested (fewer than {MIN10} players with a value and the outcome, or no spread): {names10(gone)}.")
    if no:
        lines.append("A metric that repeats poorly can correlate only weakly with anything, so a null where it does "
                     "not repeat says little about the skill behind it.")
    if c["tries"] == 1:
        lines.append("These bars assume this is the only metric and outcome you tried. If not, set MY_TRIES and rerun.")
    return lines


def show10(c):
    f2_ = lambda v: f"{v:.2f}" if np.isfinite(v) else ""
    ci_ = lambda q, e_: [f"{lo:.2f} to {hi:.2f}" if np.isfinite(lo) else "" for lo, hi in zip(q[e_ + " lo"], q[e_ + " hi"])]
    src = "the per-player table, one row per 40 attempt" if c["drills"] is None else ", ".join(c["drills"])
    print(f"SCORECARD: {c['label']}\n  attempts: {src} ({c['n_val']} of {c['n_att']} with a value); "
          f"per player: {c['best']}; outcome: {OUT10[c['outcome']]}; metric-and-outcome pairs tried: {c['tries']}")
    if c["errs"]:
        print(f"  the metric raised an error on {len(c['errs'])} attempts (first: {c['errs'][0]}); they count as missing")
    rep = c["rep"]
    print(f"\n1. Does it repeat? Each Combine year centred; 95% bootstrap CI ({BREP10:,} resamples of players).\n"
          "   'size fixed' = the same r with height and weight held fixed; the flag uses it. Top speed is section 7's.")
    print(pd.DataFrame({"check": rep.check, "players": rep.players, "r": rep.r.map(f2_),
                        "95% CI": [f"{a_:.2f} to {b_:.2f}" if np.isfinite(a_) else "" for a_, b_ in zip(rep.lo, rep.hi)],
                        "size fixed": rep.r_free.map(f2_), "top speed": RELY.loc["vmax", POS_ORDER].map(f2_),
                        f"r >= {BAR10:.2f}": [("yes" if v >= BAR10 else "no") if np.isfinite(v) else "" for v in rep.r_free]
                        }).to_string())
    if (rep.check == "drill to drill").any():
        print("   drill to drill: one drill against another. Your value averages the drills, so it repeats a little "
              "better than this.")
    print("\n2. How many players can it be tested on?")
    print(c["cov"].to_string())
    Q4 = c["Q4"]
    if not len(Q4):
        print(f"\nno group has {MIN10} players to test, so questions 3 and 4 have no answer")
    else:
        print("\n3. What correlation would it need? |r| at the tested n, with size, class and draft position held fixed"
              + (" (and alignment for DL get-off)" if c["outcome"] == "main" else "") + ".\n"
              f"   Bonferroni splits p < 0.05 over 5 groups x {c['tries']} (over {c['tries']} for the pooled value).")
        print(pd.DataFrame({"n": Q4.n.astype(int), "one test": Q4["one test"].map(f2_), "Bonferroni": Q4.Bonferroni.map(f2_),
                            "found 80% of the time": Q4["80% power"].map(f2_)}).to_string())
        print("\n4. Does it say anything beyond size and draft position? Partial r, 95% bootstrap CI "
              f"({c['B']:,} resamples of players).\n"
              "   r > 0: a higher value goes with more of the outcome (a faster time is a lower one"
              + ("; more get-off time or pressure allowed is worse" if c["outcome"] == "main" else "") + ").\n"
              "   left = the metric's SD once size, class and draft position are fixed, as a share of its SD.")
        print(pd.DataFrame({"n": Q4.n.astype(int), "size, class": Q4["T"].map(f2_), "95% CI": ci_(Q4, "T"),
                            "+ draft position": Q4.D.map(f2_), "95% CI ": ci_(Q4, "D"), "ranks": Q4["D, ranks"].map(f2_),
                            "95% CI  ": ci_(Q4, "D, ranks"), "left": Q4.left.map(f2_), "verdict": Q4.verdict}).to_string())
    print("\nVERDICT")
    for l_ in c["verdict"]:
        print("  " + l_)


# the shipped example, kept apart from the reader's cell so the text after this section always describes its card
def example10(f):
    sec, x, s = f.sec.to_numpy(), f.x.to_numpy(), f.s.to_numpy()
    return xcross(sec, x, 20.5) - xcross(sec, s, 2.0)     # 2 yd/s to the 10-yard mark, as in section 2


EX10 = scorecard(example10, ["FORTY_YARD_DASH"], "min", "share", show=False)

# a check on the machinery: fed top speed (from the frames and from the table) it must give section 9's values,
# and fed section 7's braking it must give section 7's drill-to-drill values
CHK10 = {"frames": scorecard(lambda f: f.s.max(), ["FORTY_YARD_DASH"], "max", "share", show=False, B=0),
         "table": scorecard(lambda t: t.vmax, None, "max", "share", show=False, B=0),
         "brake": scorecard(lambda f: (stop_metrics(f.sec.to_numpy(), f.s.to_numpy()) or {"brake": np.nan})["brake"],
                            DRILLS["WR"] + DRILLS["DB"], "mean", "main", show=False, B=0)}
CHK10_GAP = max(
    *[abs(CHK10[k_]["Q4"].loc[p, e_] - R9[e_].loc[p, "r"]) for k_ in ("frames", "table")
      for p in POS_ORDER + ["All"] for e_ in ("T", "D")],
    *[abs(CHK10[k_]["rep"].loc[p, "r"] - RELY.loc["vmax", p]) for k_ in ("frames", "table") for p in POS_ORDER],
    *[abs(CHK10["brake"]["rep"].loc[p, "r"] - REP.loc[(p, "brake", "drill and year"), "mean_r"]) for p in SCORE_GROUPS],
    (CHK10["frames"]["x"] - best40.vmax).abs().max())
CHK10_N = CHK10["frames"]["x"].notna().sum() == best40.vmax.notna().sum()
print(f"check: fed top speed, from the frames and from the table, this scorecard gives "
      f"{CHK10['frames']['Q4'].loc['All', 'T']:.2f} and {CHK10['frames']['Q4'].loc['All', 'D']:.2f} for all groups, "
      f"as section 9 does; fed braking, it gives section 7's drill-to-drill r ({CHK10['brake']['rep'].loc['WR', 'r']:.2f} "
      f"for receivers). Largest difference from sections 7 and 9 in any group: {CHK10_GAP:.0e}; same players: {CHK10_N}")

In [ ]:
# your scorecard, for the settings and my_metric in the cell near the top. To try another metric without
# rerunning everything, edit and run that cell, then run this one again.
CARD10 = scorecard(my_metric, MY_DRILLS, MY_BEST, MY_OUTCOME, MY_TRIES, MY_LABEL)

**The example fails at the first question.** A player's first 10 yards in one attempt predict
his next attempt at r = 0.07 for wide receivers and 0.13 for defensive backs.
Even the linemen stop short of the bar (0.64 on the defensive line, 0.57 on the
offensive line), and holding height and weight fixed lifts no group over it. These are the
year-centred values of section 7's recheck, computed again from `my_metric`.

Against rookie snap share (n = 417), a faster start goes with more playing time when size and
class are held fixed: r = -0.13 (-0.24 to -0.03). It is negative because a
faster start is a lower time. With draft position held fixed as well it is -0.05
(-0.15 to 0.06), and no group clears its Bonferroni bar. That is the
all-groups pattern section 9 found for top speed, here for a metric that does not even repeat. The
card also says what it cannot see. For tight ends (n = 34), with draft position held
fixed too, a test finds only a true r of 0.51 or more 80% of the time.

The scorecard's code cell ends with a check on the machinery. Fed top speed, from the frames and
from the table, it gives section 9's values in every group. Fed section 7's braking, it gives
section 7's drill-to-drill values. All agree to nine decimal places.

**Before you trust a card.** Every metric and outcome you try is another chance for noise to clear
a bar, so set `MY_TRIES` honestly: at 20 tries the pooled bar for snap share rises from
0.10 to 0.15. A group that clears every bar is still a lead to test on another
draft class, which is what section 9 says about the defensive line. And a metric that repeats
poorly can correlate only weakly with anything, so a null for it says little about the skill it
was meant to measure.

## 11. The snaps the tracking cannot see

The data page, as I read it on 2026-10-10, says `player_play` holds "every snap involving the rookie
cohort". My public dataset, [BDB 2027 Public Enrichment](https://www.kaggle.com/datasets/dariushafshar/bdb-2027-public-enrichment), counts each player's regular-season
snaps from public play participation, by unit (offence, defence and special teams) and by season, which the
competition files do not. Its career totals match the competition's own within 1% (or one snap) for
97.7% of the players with snaps on offence, 95.3% on defence and 97.5% on
special teams (section 9's check, on career totals). So the two can be compared for every player-season.

**Two questions.**

1. **How much of these players' NFL play is in the play files?** For every player-season from 2023 to 2025, I count
   his regular-season rows in `player_play` against his public snaps on offence or defence, and add his
   special-teams snaps.
2. **Does speed show up in the snaps the files cannot see?** Section 9 found that top speed's link to rookie snaps
   on offence or defence goes with draft position. Covering a kick or a punt starts with a sprint downfield, so
   special teams is where I expected speed to matter beyond the draft.

**I wrote down these definitions, the one test and my predictions before relating any special-teams number to
speed, size or draft position:**

* **Outcome:** rookie-season special-teams snaps as a share of a full 17-game season of his team's special-teams
  plays, built the way section 9 builds snap share; 0 if he never played.
* **Everything else is section 9's:** tracking top speed; height, weight and draft class held fixed (T), then draft
  position as well (D); bootstrap intervals from 4,000 resamples of players, the same resamples as section 9, so
  the two sections' values pair up.
* **The one test:** D for receivers, tight ends and defensive backs together (each group's value weighted by its
  players), the groups that cover kicks and punts. I expected the linemen to play special teams mostly on
  field-goal and extra-point units, where speed is not the job, so they are a contrast, not a test.
* **My predictions:** D = +0.20 for the three coverage groups, and above T, because faster prospects are
  drafted earlier and I expected early picks to spend fewer of their snaps on special teams. About zero for the
  linemen.
* **Planned checks:** ranks; the 10-to-40 split and the official 40 instead of top speed; draft round as
  categories; the special-teams snap count, any special-teams snap at all, the share among players who had one, and
  the mean over his seasons so far; and leaving out kick and punt returners.

In [ ]:
# section 11, part 1: what the competition's play files hold (aggregate counts only)
pp11 = pd.read_csv(find_file("player_play.csv"), low_memory=False,
                   usecols=["game_id", "play_id", "nfl_id", "route_ran", "pass_result", "play_description"])
pp11 = pp11.merge(games[["game_id", "season", "season_type"]], on="game_id", how="left")
pp11["pos"] = pp11.nfl_id.map(POS)
KICK11 = int(pp11.play_description.fillna("").str.lower()
             .str.contains(r"\bpunt|\bkicks\b|kickoff|kicks off|field goal|extra point|onside|fair catch", regex=True).sum())
rec11 = pp11.pos.isin(["WR", "TE"])
N_REC11 = int(rec11.sum())
ROUTE11 = float((pp11.route_ran.notna() & pp11.pass_result.notna())[rec11].mean())
PASS11 = pp11.groupby("pos").pass_result.apply(lambda v: v.notna().mean()).reindex(POS_ORDER)
# the game tracking (loaded in the Loading cell as one row per player-play): does it hold the same plays?
on11 = plays[["game_id", "play_id"]].merge(pp11[["game_id", "play_id"]].drop_duplicates().assign(hit=1),
                                           on=["game_id", "play_id"], how="left").hit.notna()
TRK11 = {"on": int(on11.sum()), "n": len(on11)}
print(f"player_play rows whose description is a punt, kickoff, field goal or extra point: {KICK11} of {len(pp11):,}")
print(f"game tracking player-plays on a play that player_play holds: {TRK11['on']:,} of {TRK11['n']:,}")
print(f"receiver and tight-end rows with a route and a pass result: {ROUTE11:.1%} of {N_REC11:,}")
print("share of each group's rows on pass plays: " + ", ".join(f"{p} {v:.0%}" for p, v in PASS11.items()))

# every regular-season player-season 2023-2025 of the prospects in my dataset: rows in player_play against public snaps
reg11 = pp11[pp11.season_type == "REG"].groupby(["nfl_id", "season"]).size().rename("tracked")
gt11 = plays[plays.season_type == "REG"].groupby(["nfl_id", "season"]).size().rename("tracking")
cv11 = (s9.set_index(["nfl_id", "season"])[["off_snaps", "def_snaps", "st_snaps"]].join(reg11).join(gt11)
        .fillna({"tracked": 0, "tracking": 0}))
cv11["scrim"] = cv11.off_snaps + cv11.def_snaps
COV11 = cv11.groupby(cv11.index.get_level_values("nfl_id").map(POS))[["tracked", "tracking", "scrim", "st_snaps"]].sum()
COV11 = COV11.reindex(POS_ORDER).rename_axis("group")
COV11.loc["All"] = COV11.sum()
COV11["all snaps"] = COV11.scrim + COV11.st_snaps
COV11["tracked / scrimmage"] = COV11.tracked / COV11.scrim
COV11["game tracking / scrimmage"] = COV11.tracking / COV11.scrim
COV11["special teams / all"] = COV11.st_snaps / COV11["all snaps"]
COV11["tracked / all"] = COV11.tracked / COV11["all snaps"]
REG11 = int((pp11.season_type == "REG").sum())
print(f"\nregular season 2023-2025 (scrim = public snaps on offence or defence; tracked = rows in player_play; "
      f"tracking = player-plays in the game tracking; the table holds {int(COV11.loc['All', 'tracked']):,} of the "
      f"{REG11:,} regular-season rows)")
print(COV11.astype({c_: int for c_ in ("tracked", "tracking", "scrim", "st_snaps", "all snaps")}).round(3).to_string())

# part 2: the test written down in advance. Rookie special-teams share, built the way section 9 builds share
assert (s9.team_st_snaps[s9.games_with_snap > 0] > 0).all()
s9["st_share"] = np.where(s9.games_with_snap > 0,
                          s9.st_snaps / (s9.team_st_snaps / s9.games_with_snap.clip(lower=1) * 17), 0.0)
rk11 = s9[s9.season == s9.entry_year].set_index("nfl_id")
assert rk11.index.is_unique and rk11.index.equals(rk9.index)
F9["st_share"], F9["st_snaps"] = rk11.st_share, rk11.st_snaps
F9["st_any"] = (rk11.st_snaps > 0).astype(float)
F9["st_career"] = s9.groupby("nfl_id").st_share.mean()
F9["returner"] = (rk11.punt_returns.fillna(0) + rk11.kickoff_returns.fillna(0) > 0).reindex(F9.index, fill_value=False)
PRED11 = {"D, coverage groups": 0.20, "T, coverage groups": 0.12, "D, defensive backs": 0.25,
          "D, wide receivers": 0.15, "D, linemen": 0.00,
          "D special teams minus D scrimmage, coverage groups": 0.15}       # written down before computing
print(f"\nrookie special-teams share: max {F9.st_share.max():.3f}; my predictions, fixed in advance: {PRED11}")

RNG11 = np.random.default_rng(2027)     # section 11's own stream, for player sets that section 9 did not resample
IDX11 = {}


def resamples11(f, B=4000):
    # section 9's resamples when section 9 used the same players, so the two sections pair up; else section 11's own
    key = (f.pos.iloc[0], tuple(f.index))
    if key in IDX9:
        return IDX9[key]
    if key not in IDX11:
        IDX11[key] = RNG11.integers(0, len(f), (B, len(f)))
    return IDX11[key]


def unit11(x_col="vmax", y_col="st_share", draft="log", rank=False, sub=None, raw=False):
    # per group: (estimate, values on the resamples, n), with section 9's frames and covariates
    out = {}
    for p in POS_ORDER:
        f = frame9(p, x_col, y_col, sub)
        x, y = f[x_col].to_numpy(float), f[y_col].to_numpy(float)
        Z = np.empty((len(f), 0)) if raw else design9(f, draft)
        if rank:                       # ranks for speed, outcome, height, weight and (if used) log pick
            x, y = stats.rankdata(x), stats.rankdata(y)
            for j in [0, 1] + ([Z.shape[1] - 2] if draft == "log" else []):
                Z[:, j] = stats.rankdata(Z[:, j])
        out[p] = (presid9(x, y, Z)[0], presid9(x, y, Z, resamples11(f)), len(f))
    return out


POOLS11 = {"Coverage groups (WR, TE, DB)": ["WR", "TE", "DB"], "Linemen (DL, OL)": ["DL", "OL"], "All five": POS_ORDER}
COVG11, LINE11, ALL11 = POOLS11


def pool11(u, groups):
    # the groups' values weighted by players, as section 9 pools them
    n = sum(u[p][2] for p in groups)
    return sum(u[p][0] * u[p][2] for p in groups) / n, sum(u[p][1] * u[p][2] for p in groups) / n, n


def table11(u):
    rows = {p: (u[p][0], *np.quantile(u[p][1], [0.025, 0.975]), u[p][2]) for p in POS_ORDER}
    for lab, gs in POOLS11.items():
        e_, b_, n_ = pool11(u, gs)
        rows[lab] = (e_, *np.quantile(b_, [0.025, 0.975]), n_)
    return pd.DataFrame(rows, index=["r", "lo", "hi", "n"]).T


U11 = {"raw": unit11(raw=True), "T": unit11(draft=None), "D": unit11(), "D, scrimmage (section 9)": unit11(y_col="share")}
R11 = {k_: table11(v_) for k_, v_ in U11.items()}
MAIN11 = pd.concat(R11, axis=1)
GAP11 = max(abs(R11["D, scrimmage (section 9)"].loc[p, "r"] - R9["D"].loc[p, "r"]) for p in POS_ORDER)
GAP11 = max(GAP11, abs(R11["D, scrimmage (section 9)"].loc[ALL11, "r"] - R9["D"].loc["All", "r"]))
for p in POS_ORDER:
    n_ = int(R11["D"].loc[p, "n"])
    MAIN11.loc[p, ("bar D", "one test")] = r_crit(n_, k=K9["D"])
    MAIN11.loc[p, ("bar D", "Bonferroni, 5")] = r_crit(n_, 0.05 / 5, k=K9["D"])
    MAIN11.loc[p, ("bar D", "80% power")] = r_min(n_ - K9["D"])
    MAIN11.loc[p, ("bar T", "Bonferroni, 5")] = r_crit(n_, 0.05 / 5, k=K9["T"])
for lab, gs in POOLS11.items():
    n_, k_ = int(R11["D"].loc[lab, "n"]), len(gs) * (K9["D"] + 1) - 1      # intercepts and slopes per group, less one
    MAIN11.loc[lab, ("bar D", "one test")] = r_crit(n_, k=k_)
    MAIN11.loc[lab, ("bar D", "80% power")] = r_min(n_ - k_)
print("\nCombine top speed vs rookie special-teams share, within group (bootstrap 95% CI)")
print("T = height, weight and draft class held fixed; D = T plus draft position (the test: D, coverage groups)")
print(MAIN11.round(3).to_string())
print(f"section 9's D rebuilt here on the same resamples: largest difference {GAP11:.0e}")
eD11, bD11, _ = pool11(U11["D"], POOLS11[COVG11])
eT11, bT11, _ = pool11(U11["T"], POOLS11[COVG11])
eS11, bS11, _ = pool11(U11["D, scrimmage (section 9)"], POOLS11[COVG11])
DT11 = (eD11 - eT11, *np.quantile(bD11 - bT11, [0.025, 0.975]))
DIFF11 = (eD11 - eS11, *np.quantile(bD11 - bS11, [0.025, 0.975]))
print(f"coverage groups, same resamples: D minus T {DT11[0]:.3f} [{DT11[1]:.3f}, {DT11[2]:.3f}]; "
      f"D special teams minus D scrimmage {DIFF11[0]:.3f} [{DIFF11[1]:.3f}, {DIFF11[2]:.3f}]")
d11_ = R11["D"]
PVR11 = pd.DataFrame({
    "D, coverage groups (the test)": (PRED11["D, coverage groups"], *d11_.loc[COVG11, ["r", "lo", "hi"]]),
    "T, coverage groups": (PRED11["T, coverage groups"], *R11["T"].loc[COVG11, ["r", "lo", "hi"]]),
    "D, defensive backs": (PRED11["D, defensive backs"], *d11_.loc["DB", ["r", "lo", "hi"]]),
    "D, wide receivers": (PRED11["D, wide receivers"], *d11_.loc["WR", ["r", "lo", "hi"]]),
    "D, linemen": (PRED11["D, linemen"], *d11_.loc[LINE11, ["r", "lo", "hi"]]),
    "D special teams minus D scrimmage, coverage groups": (PRED11["D special teams minus D scrimmage, coverage groups"],
                                                           *DIFF11)}, index=["predicted", "r", "lo", "hi"]).T
print("\nmy predictions, written down in advance, against the results (95% CI)")
print(PVR11.round(3).to_string())
eW11, bW11, nW11 = pool11(U11["D"], ["WR", "DB"])          # added afterwards: the test without the 34 tight ends
WRDB11 = (eW11, *np.quantile(bW11, [0.025, 0.975]), nW11)
print(f"added afterwards: D for receivers and defensive backs only {WRDB11[0]:.3f} [{WRDB11[1]:.3f}, {WRDB11[2]:.3f}], "
      f"n {WRDB11[3]}")

# part 3: the checks planned in advance (variants of the test, not separate tests)
VAR11 = pd.concat({k_: table11(v_) for k_, v_ in {
    "T, ranks": unit11(draft=None, rank=True), "D, ranks": unit11(rank=True),
    "D, 10-to-40 split": unit11("fly_neg"), "D, official 40 time": unit11("forty_neg"),
    "D, draft round as categories": unit11(draft="round"),
    "D, special-teams snaps (count)": unit11(y_col="st_snaps"),
    "D, any special-teams snap": unit11(y_col="st_any"),
    "D, share among players with a special-teams snap": unit11(sub=lambda f: f.st_snaps > 0),
    "D, mean share over his seasons so far": unit11(y_col="st_career"),
    "D, returners left out": unit11(sub=lambda f: ~f.returner.astype(bool)),
}.items()}, names=["variant", "row"])
VC11 = VAR11.xs(COVG11, level="row")
VD11 = VC11.loc[[v_ for v_ in VC11.index if v_.startswith("D")]]       # the D variants, which the text summarises
print("\nplanned checks, r by group and pool")
print(VAR11.r.unstack().reindex(columns=POS_ORDER + list(POOLS11)).round(2).to_string())
print("\nplanned checks, coverage groups (95% CI)")
print(VC11.round(3).to_string())

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14.5, 5.6), gridspec_kw={"width_ratios": [1, 1.15]})
ax = axs[0]
rows11 = POS_ORDER + ["All"]
yy11 = np.arange(len(rows11))[::-1] * 1.0
yy11[-1] -= 0.35
left11 = np.zeros(len(rows11))
for lab_, v_, col_ in (("in player_play", COV11.tracked / COV11["all snaps"], BLUE),
                       ("offence or defence, not in player_play", (COV11.scrim - COV11.tracked).clip(lower=0) / COV11["all snaps"], "#86b6ef"),
                       ("special teams (not in the play files)", COV11["special teams / all"], ORANGE)):
    v_ = v_.reindex(rows11).to_numpy(float)
    ax.barh(yy11, v_, left=left11, color=col_, height=0.62, edgecolor=SURFACE, linewidth=1.5, label=lab_)
    left11 += v_
for y_, p in zip(yy11, rows11):
    ax.text(0.015, y_, f"{COV11.loc[p, 'tracked / all']:.0%}", va="center", ha="left", color="white", fontsize=10,
            fontweight="bold")
ax.set_yticks(yy11)
ax.set_yticklabels([POS_NAME[p] for p in POS_ORDER] + ["All five groups"])
ax.set_xlim(0, 1)
ax.xaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))
ax.set_xlabel("share of regular-season snaps, 2023-2025 (public snap counts)")
ax.legend(loc="upper center", bbox_to_anchor=(0.45, -0.14), ncol=2)
ax.grid(axis="y", visible=False)
ax.set_title("What the play files hold")

ax = axs[1]
rows11b = POS_ORDER + list(POOLS11)
yb11 = np.r_[np.arange(len(POS_ORDER))[::-1] + len(POOLS11) + 0.7, np.arange(len(POOLS11))[::-1] * 1.0]
for off_, (e_, col_, lab_) in zip((0.22, 0.0, -0.22), (("raw", MUTED, "raw"),
                                                       ("T", BLUE, "size and draft class held fixed"),
                                                       ("D", ORANGE, "... and draft position (the test: coverage groups)"))):
    t_ = R11[e_].reindex(rows11b)
    ax.hlines(yb11 + off_, t_.lo, t_.hi, color=col_, lw=2)
    ax.plot(t_.r, yb11 + off_, "o", color=col_, ms=6, mec=SURFACE, label=lab_)
yc11 = yb11[rows11b.index(COVG11)] - 0.22
ax.plot(PRED11["D, coverage groups"], yc11, "D", mfc=SURFACE, mec=INK, mew=1.6, ms=8, label="my prediction for the test")
ax.axvline(0, color=MUTED, lw=0.8)
ax.axhline(len(POOLS11) - 0.15, color=GRID, lw=1)
ax.set_yticks(yb11)
ax.set_yticklabels([f"{POS_NAME[p]} (n {int(R11['D'].loc[p, 'n'])})" for p in POS_ORDER]
                   + [f"{lab.split(' (')[0]} (n {int(R11['D'].loc[lab, 'n'])})" for lab in POOLS11])
ax.set_xlim(-0.75, 0.75)
ax.set_xlabel("r with rookie special-teams share (95% bootstrap CI)")
ax.legend(loc="upper center", bbox_to_anchor=(0.42, -0.14), ncol=2)
ax.grid(axis="y", visible=False)
ax.set_title("Top speed and special-teams snaps, within group")
fig.suptitle("Figure 11. The snaps the tracking cannot see, and top speed against the special-teams part", x=0.01,
             ha="left", fontsize=12, fontweight="bold", color=INK)
fig.tight_layout()
plt.show()

**`player_play` holds 72% of these players' regular-season snaps.** For defensive backs and
linemen it holds 99.7% or more of their snaps on offence or defence, on pass and run plays alike. The
rest is missing in two ways:

* **No special teams.** No `player_play` row is a punt, kickoff, field goal or extra point, and all but
  209 of the game tracking's 314,944 player-plays sit on plays that `player_play` holds. Yet
  special teams are 17% of these players' snaps: 25% for defensive backs, 23% for
  tight ends and 21% for the defensive line. The competition files keep only a career total of
  special-teams snaps, in `player_career_successes.csv`.
* **Receivers and tight ends only on routes.** All 53,252 of their `player_play` rows have a route and a
  pass result. Those rows cover 58% of a receiver's snaps on offence and 40% of a tight
  end's, so their run blocking, and a tight end's snaps in pass protection, are not there. The game tracking holds
  a few more of their plays (61% and 42% of those snaps).

So for a receiver a count of `player_play` rows is a count of routes, and for a lineman it is a count of snaps. That
is fine for "plays with the outcome", as section 4 uses it, but it is not playing time. For playing time, use snap
counts: the competition gives career totals, and my dataset,
[BDB 2027 Public Enrichment](https://www.kaggle.com/datasets/dariushafshar/bdb-2027-public-enrichment), gives them by season and unit.

**Beyond draft position, top speed does not show up in special-teams snaps, and my prediction failed.** Every
prediction I wrote down, next to what came out (95% bootstrap intervals):

| | my prediction | result |
|---|---|---|
| **The test: D for receivers, tight ends and defensive backs** (n = 229) | +0.20 | **0.01** (-0.13 to 0.13) |
| T for the same three groups | +0.12 | 0.09 (-0.04 to 0.21) |
| D above T | yes | no: D minus T is -0.08 (-0.15 to -0.02) |
| D, defensive backs | +0.25 | 0.036 (-0.182 to 0.248) |
| D, wide receivers | +0.15 | 0.09 (-0.13 to 0.27) |
| D, tight ends (n = 34) | none: too few players | -0.31 (-0.62 to 0.05) |
| D, linemen (the contrast) | about zero | 0.03 (-0.10 to 0.18) |
| D minus section 9's D for snaps on offence or defence, the three groups | +0.15 | 0.09 (-0.12 to 0.28) |

The test's interval rules out my prediction. For scale, at this n a single test needs |r| of 0.14 for
p < 0.05, and it finds only a true r of 0.19 or more 80% of the time. Holding draft position fixed lowers
the value instead of raising it. Most of the gap between the interval's top and my prediction comes from the tight ends, the group I
called too small to read. For receivers and defensive backs alone, a check I added afterwards, D =
0.062 (-0.089 to 0.197): without the tight ends, the interval stops just short of
my prediction. The last row of the table cannot be told from zero, or from my prediction.

Over all five groups the pattern is section 9's. With size and class fixed, faster prospects played more special
teams (0.12, 0.03 to 0.21); with draft position fixed as well it is
0.02 (-0.08 to 0.11). No group's value clears its Bonferroni bar, with or without
draft position held fixed. The defensive line, section 9's lead, reads 0.17 (-0.01 to
0.35): the same sign as there, with an interval that includes zero.

Every planned check agrees. Ranks, the 10-to-40 split, the official 40, draft round as categories, the snap count,
any special-teams snap at all, the share among players who had one, the mean over his seasons so far, and leaving
out the 32 returners all give D between -0.08 and 0.04 for the three coverage groups,
and every interval includes zero.

**What this means.** In these three draft classes, the special-teams snaps the tracking cannot see show no sign of
a speed link that draft position misses, as section 9 found for snaps on offence or defence outside the defensive
line. Two limits. The interval still allows a small link, up to 0.13. And special-teams snaps depend on
the roster: a rookie who starts on offence or defence plays fewer of them, which holding draft position fixed
handles only in part.

## What this audit does not claim

* **Not that burst or agility is irrelevant.** Section 6 says 54 linemen cannot
  separate burst from body size and alignment; it says nothing about a larger sample, other
  confounders or a better outcome.
* **Not that the official times are the truth.** They are the reference the scouting world
  uses. Where the tracking and the official first 10 yards disagree, I cannot say which is
  closer to the player's real start, only that the tracking's own start does not repeat
  well except for linemen.
* **Not that my agility timings are the best possible.** They cover the attempts my
  turn-finder could time, and the direction of any bias is unknown.
* **Not that the noise floors are exact.** The one-test and Bonferroni bars assume roughly
  normal data, and all three count only the pairs listed here; any search wider than that
  needs a higher bar.
* **Not that change of direction cannot be measured from the tracking.** Section 7 tests one
  definition of a stop and four measures of it, in the drills where the stop is clear. A
  better definition, other drills, or body orientation (the Combine file has no `o` column)
  might repeat better. Its separation test is one test at n = 59, so it rules out only
  a large link.
* **Not that the draft prices speed correctly, or that speed does not matter to coaches.**
  Section 9 finds that, in three rookie classes, speed's link to rookie playing time cannot be
  told apart from draft position outside the defensive line. It does not say why teams draft
  fast players early, whether they should, or what happens after a player's first seasons.
* **Not that a metric which passes the section 10 scorecard is a finding.** Its bars are for one
  metric chosen in advance, tested on three draft classes. It cannot know how many metrics you
  tried unless you tell it, and a group that clears every bar is a lead to test on another class.
* **Not that the play files are wrong, or that speed is useless on special teams.** Section 11
  shows what `player_play` leaves out (special teams, and receivers' and tight ends' snaps without
  a route), not that its rows are wrong. Its special-teams test is one test chosen in advance, on
  three draft classes, and its interval still leaves room for a small link.

## A checklist before trusting a Combine-to-NFL correlation

1. Regular season only; tracking frames with `entity_type == "PLAYER"`; distance from
   `x` and `y`, not `dis`.
2. Build on what repeats: top speed and the 10 to 40 yard split. Check attempt-to-attempt
   r for anything else, within position, before using it.
3. Compare within position, never pooled (top speed: 0.87 pooled against
   0.14 to 0.62 within position).
4. Hold size and alignment fixed. If a link disappears when they are fixed, it cannot be
   told apart from them.
5. Judge every pair at its own n, state how many you tried, and clear the shuffled-outcome
   bar for that search.
6. Centre Combine tracking metrics within each Combine year as well as each position. The
   same drill can read 3.3 SDs differently from one year to the next, and a shared
   year looks like a shared trait.
7. Before calling a Combine measure a predictor of NFL playing time, hold draft position fixed
   and say what that means: draft position is partly on the path from the measure to the snaps
   and partly a stand-in for everything else teams knew.
8. Know what the play files hold: no special-teams play, and receivers and tight ends only on
   routes. A count of `player_play` rows is a count of routes for a receiver and of snaps for a
   lineman. For playing time, use snap counts (section 11).

Section 10 runs most of this checklist on a metric you supply.

Data: NFL Big Data Bowl 2027 (Combine and game tracking, NFL Next Gen Stats), used under
the competition rules. No rows from the competition files are redistributed here; the output
files hold per-player values computed in this notebook (section 8) and the numbers this text quotes
(`headline.json`).

The playing-time data in sections 9 to 11: my public dataset [BDB 2027 Public Enrichment](https://www.kaggle.com/datasets/dariushafshar/bdb-2027-public-enrichment),
built from nflverse data (maintained by Tan Ho and Sebastian Carl, with Ben Baldwin, Lee Sharpe
and the nflverse contributors; CC BY 4.0) and FTN Data via nflverse (play participation, CC BY-SA
4.0). It adds each player's season-by-season snaps by unit, special teams included, keyed on
`nfl_id`, and it holds no competition data.

In [ ]:
F10 = AG.index[2]


def join_words(xs):
    xs = list(xs)
    return xs[0] if len(xs) == 1 else ", ".join(xs[:-1]) + " and " + xs[-1]


clause = ("none of them involves a burst or agility metric" if N_CLEAR_BURST == 0
          else f"{N_CLEAR_BURST} of them involve a burst or agility metric, all for "
               f"{join_words(GROUP_NOUN[p] for p in BURST_GROUPS)}")
HEADLINE = {
    "fly_rw": f"{AG.iloc[0].r_within:.3f}", "fly_rw2": f"{AG.iloc[0].r_within:.2f}", "fly_n": f"{int(AG.iloc[0].n)}",
    "fly_pos_lo": f"{AG.iloc[0][POS_ORDER].min():.2f}", "fly_pos_hi": f"{AG.iloc[0][POS_ORDER].max():.2f}",
    "f10_rw": f"{AG.loc[F10, 'r_within']:.2f}", "f10_rw2": f"{AG.loc[F10, 'r_within']:.2f}",
    "f10_pos_lo": f"{AG.loc[F10, POS_ORDER].min():.2f}", "f10_pos_hi": f"{AG.loc[F10, POS_ORDER].max():.2f}",
    "f10_zero_groups": join_words(GROUP_NOUN[p] for p in F10_ZERO),
    "wr_ten_sd": f"{AG.loc[F10, 'sdref_WR']:.3f}", "wr_f10_diff_sd": f"{AG.loc[F10, 'sddiff_WR']:.3f}",
    "f10_rule_lo": f"{ONS.r_first10.min():.2f}", "f10_rule_hi": f"{ONS.r_first10.max():.2f}",
    "f10_pooled": f"{AG.loc[F10, 'r_pooled']:.2f}",
    "shuttle_rw": f"{AG.iloc[3].r_within:.2f}", "cone_rw": f"{AG.iloc[4].r_within:.2f}",
    "shuttle_timed": f"{(agil['SHORT_SHUTTLE'].status == 'ok').mean():.0%}",
    "cone_timed": f"{(agil['THREE_CONE_DRILL'].status == 'ok').mean():.0%}",
    "x_start_med": f"{ev40.x_start.median():.2f}", "reach_505": f"{ev40.fly30.notna().mean():.1%}",
    "n_retest": f"{N_RETEST}",
    "vmax_rel_min": f"{REL.loc['vmax', POS_ORDER].min():.2f}", "vmax_rel_max": f"{REL.loc['vmax', POS_ORDER].max():.2f}",
    "amax_rel_wr": f"{REL.loc['amax', 'WR']:.2f}", "amax_rel_db": f"{REL.loc['amax', 'DB']:.2f}",
    "amax_rel_dl": f"{REL.loc['amax', 'DL']:.2f}",
    "dl_burst_min": f"{BR.loc['min', 'DL']:.2f}", "dl_burst_max": f"{BR.loc['max', 'DL']:.2f}",
    "ol_burst_min": f"{BR.loc['min', 'OL']:.2f}", "ol_burst_max": f"{BR.loc['max', 'OL']:.2f}",
    "wr_burst_min": f"{BR.loc['min', 'WR']:.2f}", "wr_burst_max": f"{BR.loc['max', 'WR']:.2f}",
    "db_burst_min": f"{BR.loc['min', 'DB']:.2f}", "db_burst_max": f"{BR.loc['max', 'DB']:.2f}",
    "noise_wr": f"{SPREAD.loc['WR', 'noise_sd']:.2f}", "noise_ol": f"{SPREAD.loc['OL', 'noise_sd']:.2f}",
    "between_wr": f"{SPREAD.loc['WR', 'between_sd']:.2f}", "between_ol": f"{SPREAD.loc['OL', 'between_sd']:.2f}",
    "hw_lo": f"{min(HW):.2f}", "hw_hi": f"{max(HW):.2f}",
    "relpair_min": f"{int(RELN.min().min())}", "relpair_max": f"{int(RELN.max().max())}",
    "n_prospects": f"{len(combine)}", "n_analysis": f"{N_ANALYSIS}",
    "n_min_pos": POS_NAME[COV[ANALYSIS_COL].idxmin()].lower(), "n_min_pos_n": f"{COV[ANALYSIS_COL].min()}",
    "n_max_pos": POS_NAME[COV[ANALYSIS_COL].idxmax()].lower(), "n_max_pos_n": f"{COV[ANALYSIS_COL].max()}",
    "share_2023": f"{share.loc[2023]:.0%}", "share_2025": f"{share.loc[2025]:.0%}",
    "n_metrics": f"{N_METRICS}",
    "one_test_lo": f"{NF.one_test.min():.2f}", "one_test_hi": f"{NF.one_test.max():.2f}",
    "power_lo": f"{NF.power80.min():.2f}", "power_hi": f"{NF.power80.max():.2f}",
    "perm_r_lo": f"{NF.best_bar_r.min():.2f}", "perm_r_hi": f"{NF.best_bar_r.max():.2f}",
    "n_clear_pairs": f"{len(CLEARED)}", "n_clear_groups": f"{len(CLEAR_GROUPS)}",
    "clear_burst_clause": clause, "clear_burst_clause_cap": clause[0].upper() + clause[1:],
    "clear_summary": "; ".join(f"{GROUP_NOUN[p]} {int(KINDS.loc[p].sum())}: " + ", ".join(
        f"{int(v)} {k}" for k, v in KINDS.loc[p].items() if v > 0) for p in CLEAR_GROUPS),
    "dl_wcorr_min": f"{min(DL_WCORR.values()):.2f}", "dl_wcorr_max": f"{max(DL_WCORR.values()):.2f}",
    "wr_size": f"{int(KINDS.loc['WR'].get('size', 0))}", "wr_speed": f"{int(KINDS.loc['WR'].get('speed', 0))}",
    "cap_wr_amax": f"{CAP[('WR', 'amax')][0]:.2f}", "cap_wr_amax_hi": f"{CAP[('WR', 'amax')][1]:.2f}",
    "rel_getoff": f"{REL_OUT[('DL', 'get_off')]:.2f}",
    "go_interior": f"{GO_IN:.3f}", "go_edge": f"{GO_EDGE:.3f}", "w_interior": f"{W_IN:.2f}",
    "dl_n": f"{N_DL}",
    "dl_raw_r": f"{row.raw:.2f}", "dl_raw_lo": f"{row.raw_lo:.2f}", "dl_raw_hi": f"{row.raw_hi:.2f}",
    "dl_part_r": f"{row.partial:.2f}", "dl_part_lo": f"{row.part_lo:.2f}", "dl_part_hi": f"{row.part_hi:.2f}",
    "dl_part2_r": f"{PW2:.2f}", "dl_part2_lo": f"{PW2_LO:.2f}", "dl_part2_hi": f"{PW2_HI:.2f}",
    "pa_drop": f"{PA_DROP:.0%}", "pa_nodrop": f"{PA_NODROP:.0%}",
    "cone_wr": f"{AG.iloc[4]['WR']:.2f}", "cone_te": f"{AG.iloc[4]['TE']:.2f}",
    "cone_wr_n": f"{int(AG.iloc[4]['n_WR'])}", "cone_te_n": f"{int(AG.iloc[4]['n_TE'])}", "dl_part_p": f"{row.part_p:.3f}",
    "w_go": f"{W_GO:.2f}", "w_ten": f"{W_TEN:.2f}", "dl_shared": f"{SHARED:.0%}",
    "dl_pairs": f"{DL_PAIRS}", "dl_bonf": f"{DL_BONF:.2f}",
    "trk_part_lo": f"{TRK_PART.min():.2f}", "trk_part_hi": f"{TRK_PART.max():.2f}",
    "forty_raw": f"{FOREST.loc['forty', 'raw']:.2f}", "forty_part": f"{FOREST.loc['forty', 'partial']:.2f}",
    "pc_clear_names": join_words(f"{GROUP_NOUN[p]} (r = {PC.loc[p, 'r']:.2f})" for p in PC_CLEAR),
    "pc_pooled": f"{PC_POOLED:.2f}", "pc_min": f"{PC.r.min():.2f}", "pc_max": f"{PC.r.max():.2f}",
    "pre_share": f"{PRE_SHARE:.0%}", "gt_ratio": f"{GT_RATIO:.1f}", "forty_dis_sum": f"{FORTY_DIS_SUM:.1f}",
    "game_dis_step": f"{GAME_DIS_STEP:.2f}",
}
# version 2: section 7 (change of direction) and the year check
r7 = lambda p, m="brake", c="drill and year": REP.loc[(p, m, c)]
OTHER7 = REP.xs("drill and year", level="centred").loc[SCORE_GROUPS].drop(index="brake", level="metric")
best7 = OTHER7.mean_r.idxmax()
COD_PHRASE = {"stop_speed": "speed at the bottom of the stop", "stop_time": "time from 80% of entry speed to the bottom",
              "exit": "speed gained after the stop"}
alt7 = TR.drop(index=TR.index[1]).r
pchg = (PCY["year-centred r"] - PCY["v1 r"]).abs().idxmax()
pcy_clause = (f"{GROUP_NOUN[pchg]} move most, from {PCY.loc[pchg, 'v1 r']:.2f} to "
              f"{PCY.loc[pchg, 'year-centred r']:.2f} ({PCY.loc[pchg, 'lo']:.2f} to {PCY.loc[pchg, 'hi']:.2f})")
pcy_clause += (f", and the groups whose interval excludes zero are the same as in version 1 "
               f"({join_words(GROUP_NOUN[p] for p in PCY_CLEAR)})" if set(PCY_CLEAR) == set(PC_CLEAR) else
               f", so the groups whose interval excludes zero become {join_words(GROUP_NOUN[p] for p in PCY_CLEAR)}, "
               f"against {join_words(GROUP_NOUN[p] for p in PC_CLEAR)} in version 1")
nz7 = [f"{GROUP_NOUN[p]} with the {'shuttle' if o == 'short_shuttle' else '3-cone'}"
       for (p, o), q in VAL.iterrows() if not q.lo < 0 < q.hi]
HEADLINE.update({
    "wr_n": f"{C7.loc['WR', 'prospects']}", "db_n": f"{C7.loc['DB', 'prospects']}",
    "wr_cone_n": f"{C7.loc['WR', 'official 3-cone']}", "wr_shut_n": f"{C7.loc['WR', 'official shuttle']}",
    "db_cone_n": f"{C7.loc['DB', 'official 3-cone']}", "db_shut_n": f"{C7.loc['DB', 'official shuttle']}",
    "wr_stop_n": f"{C7.loc['WR', 'stop in 2+ position drills']}", "db_stop_n": f"{C7.loc['DB', 'stop in 2+ position drills']}",
    "yg_max": f"{YG.gap_sd.max():.1f}",
    "yg_where": f"{YG_TOP[1].lower().replace('_', ' ')} for {GROUP_NOUN[YG_TOP[0]]}, "
                f"{YG.loc[YG_TOP, 'low']} vs {YG.loc[YG_TOP, 'high']}",
    "cod_wr_r": f"{r7('WR').mean_r:.2f}", "cod_wr_lo": f"{r7('WR').lo:.2f}", "cod_wr_hi": f"{r7('WR').hi:.2f}",
    "cod_wr_pair_lo": f"{r7('WR').pair_r_min:.2f}", "cod_wr_pair_hi": f"{r7('WR').pair_r_max:.2f}",
    "cod_wr_n_lo": f"{int(r7('WR').pair_n_min)}", "cod_wr_n_hi": f"{int(r7('WR').pair_n_max)}",
    "cod_wr_pooled": f"{r7('WR', c='drill only').mean_r:.2f}", "cod_wr_pooled_lo": f"{r7('WR', c='drill only').lo:.2f}",
    "cod_wr_pooled_hi": f"{r7('WR', c='drill only').hi:.2f}",
    "cod_db_r": f"{r7('DB').mean_r:.2f}", "cod_db_lo": f"{r7('DB').lo:.2f}", "cod_db_hi": f"{r7('DB').hi:.2f}",
    "cod_db_pair_lo": f"{r7('DB').pair_r_min:.2f}", "cod_db_pair_hi": f"{r7('DB').pair_r_max:.2f}",
    "cod_dl_r": f"{r7('DL').mean_r:.2f}",
    "cod_other_lo": f"{OTHER7.mean_r.min():.2f}", "cod_other_hi": f"{OTHER7.mean_r.max():.2f}",
    "cod_best_what": f"{COD_PHRASE[best7[1]]} for {GROUP_NOUN[best7[0]]}", "cod_best": f"{OTHER7.loc[best7, 'mean_r']:.2f}",
    "cod_best_lo": f"{OTHER7.loc[best7, 'lo']:.2f}", "cod_best_hi": f"{OTHER7.loc[best7, 'hi']:.2f}",
    "bench_lo": f"{BENCH.r.min():.2f}", "bench_hi": f"{BENCH.r.max():.2f}",
    "val_wr_shut": f"{VAL.loc[('WR', 'short_shuttle'), 'r']:.2f}", "val_wr_shut_n": f"{int(VAL.loc[('WR', 'short_shuttle'), 'n'])}",
    "val_db_shut": f"{VAL.loc[('DB', 'short_shuttle'), 'r']:.2f}", "val_db_shut_n": f"{int(VAL.loc[('DB', 'short_shuttle'), 'n'])}",
    "val_clause": ("all four intervals, the 3-cone included, contain zero" if VAL_ZERO
                   else "the interval excludes zero only for " + join_words(nz7)),
    "dl_mix_w": f"{MIX['DL']:.2f}",
    "t_n": f"{N_T}", "t_p": f"{T_P:.2f}", "t_p_lo": f"{T_P_CI[0]:.2f}", "t_p_hi": f"{T_P_CI[1]:.2f}",
    "t_alt_lo": f"{alt7.min():.2f}", "t_alt_hi": f"{alt7.max():.2f}", "t_crit": f"{T_CRIT:.2f}", "t_pow": f"{T_POW:.2f}",
    "t_ceil": f"{T_CEIL:.2f}", "t_ceil_hi": f"{T_CEIL_HI:.2f}",
    "sel_d": f"{SEL_D:.2f}", "sel_lo": f"{SEL_CI[0]:.2f}", "sel_hi": f"{SEL_CI[1]:.2f}",
    "sel_sd_in": f"{SEL_SD[0]:.2f}", "sel_sd_all": f"{SEL_SD[1]:.2f}",
    "yc_fly": f"{YC_FLY:.3f}", "yc_f10": f"{YC_F10:.2f}",
    "yc_vmax_change": f"{(RELY.loc['vmax'] - REL.loc['vmax', POS_ORDER]).abs().max():.2f}",
    "yc_wr_burst_lo": f"{BRY.loc['min', 'WR']:.2f}", "yc_wr_burst_hi": f"{BRY.loc['max', 'WR']:.2f}",
    "yc_te_burst_lo": f"{BRY.loc['min', 'TE']:.2f}", "yc_te_burst_hi": f"{BRY.loc['max', 'TE']:.2f}",
    "te_burst_min": f"{BR.loc['min', 'TE']:.2f}", "te_burst_max": f"{BR.loc['max', 'TE']:.2f}",
    "pcy_clause": pcy_clause,
})
# version 3: section 9 (speed and NFL playing time, with the public dataset)
t9, d9, f2 = R9["T"], R9["D"], (lambda v: f"{v:.2f}")
FOUR9 = ["WR", "TE", "DB", "OL"]
rk_dl9 = VAR9.loc[("D, ranks", "DL")]
CLAIMS9 = {   # each worded claim in section 9's text, checked against this run
    "every found prospect in the same class": JOIN9["same_class"] == JOIN9["found"],
    "T, all groups: interval excludes zero": t9.loc["All", "lo"] > 0,
    "T: DL and OL clear the Bonferroni bar": all(t9.loc[p, "r"] > MAIN9.loc[p, ("bar T", "Bonferroni, 5")] for p in ("DL", "OL")),
    "T: DL, OL, DB intervals exclude zero": all(t9.loc[p, "lo"] > 0 for p in ("DL", "OL", "DB")),
    "T: DB clears one test, not Bonferroni": MAIN9.loc["DB", ("bar T", "one test")] < t9.loc["DB", "r"]
                                             < MAIN9.loc["DB", ("bar T", "Bonferroni, 5")],
    "T: WR interval includes zero": t9.loc["WR", "lo"] < 0 < t9.loc["WR", "hi"],
    "D, all groups: interval includes zero": d9.loc["All", "lo"] < 0 < d9.loc["All", "hi"],
    "D: WR, TE, DB, OL intervals include zero": all(d9.loc[p, "lo"] < 0 < d9.loc[p, "hi"] for p in FOUR9),
    "D: DL clears the Bonferroni bar": d9.loc["DL", "r"] > MAIN9.loc["DL", ("bar D", "Bonferroni, 5")] > 0 < d9.loc["DL", "lo"],
    "D: DL with ranks includes zero": rk_dl9.lo < 0 < rk_dl9.hi,
    "draft position predicts playing time more strongly than speed in any group": abs(P2_9) > t9.r.abs().max(),
    "within a round the halves play about the same": GAP_ROUND9_CI[0] < 0 < GAP_ROUND9_CI[1] and GAP_ALL9_CI[0] > 0,
    "the influential DL player is an edge rusher": TOP9_POS in ("OLB", "DE"),
    "D: DB below zero": d9.loc["DB", "r"] < 0,
    "faster prospects drafted earlier": PATH9.loc[("speed vs log pick", "All"), "hi"] < 0,
    "at least half of the link goes with draft position": DROP9_CI[0] > 0 and LEFT9_HI < 0.5,
    "planned T variants all above zero": (VT9.lo > 0).all(),
    "planned D variants all include zero": ((VD9.lo < 0) & (VD9.hi > 0)).all(),
    "official 40: T above zero, D includes zero": VA9.loc["T, official 40 time (post hoc)", "lo"] > 0
        and VA9.loc["D, official 40 time (post hoc)", "lo"] < 0 < VA9.loc["D, official 40 time (post hoc)", "hi"],
    "the five D values differ (p < 0.05)": Q9_P < 0.05,
    "TE: interval includes zero": t9.loc["TE", "lo"] < 0 < t9.loc["TE", "hi"],
    "DL D stays above zero with weight squared or the edge flag": (DLROB9.lo > 0).all(),
    "selection: played-only and per-game values below T, any-snap above zero":
        VA9.loc["T, full-season share, players who played (post hoc)", "r"] < t9.loc["All", "r"]
        and VA9.loc["T, per game played (snaps > 0)", "r"] < t9.loc["All", "r"]
        and VA9.loc["T, got a snap at all (post hoc)", "lo"] > 0,
    "no tracked 40: more undrafted": SEL9["udfa_without"] > SEL9["udfa_with"],
    "D with round categories includes zero": VAR9.loc[("D, draft round as categories (post hoc)", "All"), "lo"] < 0,
    "no tracked 40: difference includes zero": abs(SEL9["diff"]) < 1.96 * SEL9["se"],
    "no tracked 40: slightly more early picks": SEL9["rd12_without"] > SEL9["rd12_with"],
}
print("section 9 claims:", {k_: bool(v_) for k_, v_ in CLAIMS9.items()})
assert all(CLAIMS9.values()), [k_ for k_, v_ in CLAIMS9.items() if not v_]
HEADLINE.update({
    "j_found": f"{JOIN9['found']}", "j_n": f"{JOIN9['prospects']}", "j_class": f"{JOIN9['same_class']}",
    "j_pick": f"{JOIN9['same_pick']}", "j_drafted": f"{JOIN9['drafted']}",
    "snap_off_1pct": f"{SNAPCHK9.loc['offence', 'within 1% or 1 snap']:.1%}",
    "snap_off_n": f"{SNAPCHK9.loc['offence', 'players with any']}",
    "snap_def_1pct": f"{SNAPCHK9.loc['defence', 'within 1% or 1 snap']:.1%}",
    "snap_def_n": f"{SNAPCHK9.loc['defence', 'players with any']}", "starts_1g": f"{STARTS1G9:.1%}",
    "s9_n": f"{N9}", "s9_n_te": f"{int(t9.loc['TE', 'n'])}", "s9_n_dl": f"{int(t9.loc['DL', 'n'])}",
    "s9_pow_te": f2(MAIN9.loc["TE", ("bar T", "80% power")]),
    **{f"s9_t_{p.lower()}{sfx}": f2(t9.loc[p, col]) for p in POS_ORDER + ["All"]
       for sfx, col in (("", "r"), ("_lo", "lo"), ("_hi", "hi"))},
    "s9_d_all": f2(d9.loc["All", "r"]), "s9_d_all_lo": f2(d9.loc["All", "lo"]), "s9_d_all_hi": f2(d9.loc["All", "hi"]),
    "s9_d_four_lo": f2(d9.loc[FOUR9, "r"].min()), "s9_d_four_hi": f2(d9.loc[FOUR9, "r"].max()),
    "s9_d_dl": f2(d9.loc["DL", "r"]), "s9_d_dl_lo": f2(d9.loc["DL", "lo"]), "s9_d_dl_hi": f2(d9.loc["DL", "hi"]),
    "s9_d_db": f2(d9.loc["DB", "r"]), "s9_bonf_d_dl": f2(MAIN9.loc["DL", ("bar D", "Bonferroni, 5")]),
    "s9_raw_dl": f2(R9["raw"].loc["DL", "r"]), "s9_t_max": f2(t9.r.abs().max()),
    "s9_path1": f2(P1_9), "s9_path2": f2(P2_9), "s9_prod": f2(P1_9 * P2_9),
    "s9_vt_lo": f2(VT9.r.min()), "s9_vt_hi": f2(VT9.r.max()), "s9_vd_lo": f2(VD9.r.min()), "s9_vd_hi": f2(VD9.r.max()),
    "s9_drop": f2(DROP9), "s9_drop_lo": f2(DROP9_CI[0]), "s9_drop_hi": f2(DROP9_CI[1]),
    "s9_q": f"{Q9:.1f}", "s9_q_p": f"{Q9_P:.3f}",
    "s9_o40_t": f2(VA9.loc["T, official 40 time (post hoc)", "r"]), "s9_o40_d": f2(VA9.loc["D, official 40 time (post hoc)", "r"]),
    "s9_dl_edge": f"{EDGE9.mean():.0%}",
    "s9_dl_w2": f2(DLROB9.loc["weight squared added", "r"]), "s9_dl_w2_lo": f2(DLROB9.loc["weight squared added", "lo"]),
    "s9_dl_w2_hi": f2(DLROB9.loc["weight squared added", "hi"]),
    "s9_dl_edge_d": f2(DLROB9.loc["edge-rusher flag added", "r"]), "s9_dl_edge_lo": f2(DLROB9.loc["edge-rusher flag added", "lo"]),
    "s9_dl_edge_hi": f2(DLROB9.loc["edge-rusher flag added", "hi"]),
    "s9_cond_share": f2(VA9.loc["T, full-season share, players who played (post hoc)", "r"]),
    "s9_any": f2(VA9.loc["T, got a snap at all (post hoc)", "r"]),
    "s9_udfa_without": f"{SEL9['udfa_without']:.0%}", "s9_udfa_with": f"{SEL9['udfa_with']:.0%}",
    "s9_gap_all_lo": f2(GAP_ALL9_CI[0]), "s9_gap_all_hi": f2(GAP_ALL9_CI[1]),
    "s9_gap_round_lo": f2(GAP_ROUND9_CI[0]), "s9_gap_round_hi": f2(GAP_ROUND9_CI[1]),
    "s9_dl_rank": f2(rk_dl9.r), "s9_dl_rank_lo": f2(rk_dl9.lo), "s9_dl_rank_hi": f2(rk_dl9.hi),
    "s9_dl_wv": f2(DL_WV9), "s9_dl_loo_lo": f2(LOO9.min()), "s9_dl_loo_hi": f2(LOO9.max()),
    "s9_dl_top_rank": "the fastest top speed" if TOP9_RANK == 1 else
                      f"the {TOP9_RANK}{'nd' if TOP9_RANK == 2 else 'rd' if TOP9_RANK == 3 else 'th'} fastest top speed", "s9_dl_top_share": f"{TOP9.share:.0%}",
    "s9_dround": f2(VAR9.loc[("D, draft round as categories (post hoc)", "All"), "r"]),
    "s9_dround_lo": f2(VAR9.loc[("D, draft round as categories (post hoc)", "All"), "lo"]),
    "s9_dround_hi": f2(VAR9.loc[("D, draft round as categories (post hoc)", "All"), "hi"]),
    "s9_gap_all": f2(GAP_ALL9), "s9_gap_round": f2(GAP_ROUND9),
    "s9_zero": f"{ZERO9:.0%}", "s9_cond_n": f"{int(VAR9.loc[('T, per game played (snaps > 0)', 'All'), 'n'])}",
    "s9_cond_t": f2(VAR9.loc[("T, per game played (snaps > 0)", "All"), "r"]),
    "s9_sel_n": f"{SEL9['without']}", "s9_sel_d": f2(SEL9["diff"]),
    "s9_sel_lo": f2(SEL9["diff"] - 1.96 * SEL9["se"]), "s9_sel_hi": f2(SEL9["diff"] + 1.96 * SEL9["se"]),
    "s9_rd12_without": f"{SEL9['rd12_without']:.0%}", "s9_rd12_with": f"{SEL9['rd12_with']:.0%}",
})
# version 4: section 10. EX10 and CHK10 are computed in the scorecard cell, before the reader's cell runs.
rep10, q10 = EX10["rep"], EX10["Q4"]
CLAIMS10 = {   # each worded claim in section 10's text, checked against this run
    "example repeat r equals section 7's year-centred first 10 yards":
        (rep10.r - RELY.loc["first10", POS_ORDER]).abs().max() < 1e-9,
    "example player values equal section 8's first10 column":
        (EX10["x"] - best40.first10).abs().max() < 1e-9 and EX10["x"].notna().sum() == best40.first10.notna().sum(),
    "example repeats below the bar in every group, with and without size fixed":
        (rep10.r < BAR10).all() and (rep10.r_free < BAR10).all(),
    "the linemen repeat best": set(rep10.r.astype(float).nlargest(2).index) == {"DL", "OL"},
    "size and class fixed: all-groups interval below zero": q10.loc["All", "T hi"] < 0,
    "draft position fixed: all-groups interval includes zero": q10.loc["All", "D lo"] < 0 < q10.loc["All", "D hi"],
    "no group clears its Bonferroni bar": not any(q10.loc[p_, "verdict"].startswith("clears") for p_ in POS_ORDER),
    "tight ends flagged as small n": "small n" in q10.loc["TE", "verdict"],
    "fed top speed and braking, the machinery matches sections 7 and 9 to nine decimals": CHK10_GAP < 1e-9 and CHK10_N,
}
print("section 10 claims:", {k_: bool(v_) for k_, v_ in CLAIMS10.items()})
assert all(CLAIMS10.values()), [k_ for k_, v_ in CLAIMS10.items() if not v_]
HEADLINE.update({
    **{f"s10_rep_{p_.lower()}": f2(rep10.loc[p_, "r"]) for p_ in ("WR", "DB", "DL", "OL")},
    "s10_n": f"{int(q10.loc['All', 'n'])}", "s10_n_te": f"{int(q10.loc['TE', 'n'])}",
    "s10_pow_te": f2(q10.loc["TE", "80% power"]),
    **{f"s10_{e_.lower()}_all{sfx}": f2(q10.loc["All", e_ + col]) for e_ in ("T", "D")
       for sfx, col in (("", ""), ("_lo", " lo"), ("_hi", " hi"))},
    "s10_bar1": f2(q10.loc["All", "one test"]),
    "s10_bar20": f2(r_crit(int(q10.loc["All", "n"]), 0.05 / 20, k=int(q10.loc["All", "k"]))),
})
# version 5: the figure under the title must show section 2's values, and section 11's worded claims must hold
FIG0_GAP = max(abs(FIG0_R["fly"] - AG.iloc[0].r_within), abs(FIG0_R["f10"] - AG.loc[F10, "r_within"]))
assert FIG0_GAP < 1e-12 and FIG0_N == (int(AG.iloc[0].n), int(AG.loc[F10, "n"])), (FIG0_GAP, FIG0_N)
print(f"the figure under the title shows section 2's values: largest difference {FIG0_GAP:.0e}, same players")
d11, t11 = R11["D"], R11["T"]
THREE11 = ["DB", "DL", "OL"]
big11 = d11.loc[POS_ORDER, "r"].abs().idxmax()
CLAIMS11 = {   # each worded claim in section 11's text, checked against this run
    "no player_play row is a kick": KICK11 == 0,
    "the game tracking's player-plays sit on player_play's plays, all but a few": TRK11["on"] / TRK11["n"] > 0.99,
    "every receiver and tight-end row has a route and a pass result": ROUTE11 == 1.0,
    "defensive backs and linemen appear on pass and run plays": all(0.5 < PASS11[p_] < 0.7 for p_ in THREE11),
    "defensive backs and linemen: 99.5% or more of their scrimmage snaps": (COV11.loc[THREE11, "tracked / scrimmage"] >= 0.995).all(),
    "receivers and tight ends: well under": (COV11.loc[["WR", "TE"], "tracked / scrimmage"] < 0.7).all(),
    "the table covers every regular-season row": int(COV11.loc["All", "tracked"]) == REG11,
    "the test: coverage-group D interval includes zero": d11.loc[COVG11, "lo"] < 0 < d11.loc[COVG11, "hi"],
    "the prediction lies above that interval": PRED11["D, coverage groups"] > d11.loc[COVG11, "hi"],
    "draft position lowers it (D minus T below zero)": DT11[2] < 0,
    "linemen: D interval includes zero": d11.loc[LINE11, "lo"] < 0 < d11.loc[LINE11, "hi"],
    "all five: T interval above zero, D includes zero": t11.loc[ALL11, "lo"] > 0 and d11.loc[ALL11, "lo"] < 0 < d11.loc[ALL11, "hi"],
    "no group's D clears its Bonferroni bar": all(abs(d11.loc[p_, "r"]) < MAIN11.loc[p_, ("bar D", "Bonferroni, 5")]
                                                for p_ in POS_ORDER),
    "no group's T clears its Bonferroni bar": all(abs(t11.loc[p_, "r"]) < MAIN11.loc[p_, ("bar T", "Bonferroni, 5")]
                                                for p_ in POS_ORDER),
    "added afterwards: without the tight ends the interval includes zero and stops short of the prediction":
        WRDB11[1] < 0 < WRDB11[2] < PRED11["D, coverage groups"],
    "most of the margin below the prediction comes from the tight ends":
        PRED11["D, coverage groups"] - WRDB11[2] < (PRED11["D, coverage groups"] - d11.loc[COVG11, "hi"]) / 2,
    "special teams minus scrimmage: its interval also holds the predicted value":
        DIFF11[1] < PRED11["D special teams minus D scrimmage, coverage groups"] < DIFF11[2],
    "the game tracking holds a few more receiver and tight-end plays than player_play":
        all(0 < COV11.loc[p_, "game tracking / scrimmage"] - COV11.loc[p_, "tracked / scrimmage"] < 0.05 for p_ in ("WR", "TE")),
    "the largest D in size is the tight ends'": big11 == "TE",
    "defensive line: same sign as section 9, interval includes zero":
        d11.loc["DL", "r"] > 0 and R9["D"].loc["DL", "r"] > 0 and d11.loc["DL", "lo"] < 0,
    "special teams minus scrimmage D includes zero": DIFF11[1] < 0 < DIFF11[2],
    "every planned check's coverage-group D interval includes zero": ((VD11.lo < 0) & (VD11.hi > 0)).all(),
    "the rank version of T includes zero too": VC11.loc["T, ranks", "lo"] < 0 < VC11.loc["T, ranks", "hi"],
    "section 9's D rebuilt on the same resamples": GAP11 < 1e-12,
}
print("section 11 claims:", {k_: bool(v_) for k_, v_ in CLAIMS11.items()})
assert all(CLAIMS11.values()), [k_ for k_, v_ in CLAIMS11.items() if not v_]
pc11 = lambda v: f"{v:.0%}"
HEADLINE.update({
    "s11_trk_all": pc11(COV11.loc["All", "tracked / all"]),
    "s11_cov_min3": f"{COV11.loc[THREE11, 'tracked / scrimmage'].min():.1%}",
    "s11_cov_wr": pc11(COV11.loc["WR", "tracked / scrimmage"]), "s11_cov_te": pc11(COV11.loc["TE", "tracked / scrimmage"]),
    "s11_st_all": pc11(COV11.loc["All", "special teams / all"]), "s11_st_db": pc11(COV11.loc["DB", "special teams / all"]),
    "s11_st_te": pc11(COV11.loc["TE", "special teams / all"]), "s11_st_dl": pc11(COV11.loc["DL", "special teams / all"]),
    "s11_trk_off": f"{TRK11['n'] - TRK11['on']:,}", "s11_trk_n": f"{TRK11['n']:,}", "s11_rec_rows": f"{N_REC11:,}",
    "s11_pred_d": f"{PRED11['D, coverage groups']:+.2f}",
    "s11_n_cov": f"{int(d11.loc[COVG11, 'n'])}", "s11_pow_cov": f2(MAIN11.loc[COVG11, ("bar D", "80% power")]),
    **{f"s11_{e_}_{lab_}{sfx}": f2(R11[e_.upper()].loc[row_, col]) for e_ in ("t", "d")
       for lab_, row_ in (("cov", COVG11), ("line", LINE11), ("all", ALL11))
       for sfx, col in (("", "r"), ("_lo", "lo"), ("_hi", "hi"))},
    "s11_dt": f2(DT11[0]), "s11_dt_lo": f2(DT11[1]), "s11_dt_hi": f2(DT11[2]),
    "s11_diff": f2(DIFF11[0]), "s11_diff_lo": f2(DIFF11[1]), "s11_diff_hi": f2(DIFF11[2]),
    "s11_d_te": f2(d11.loc["TE", "r"]), "s11_n_te": f"{int(d11.loc['TE', 'n'])}",
    "s11_bonf_te": f2(MAIN11.loc["TE", ("bar D", "Bonferroni, 5")]),
    "s11_d_dl": f2(d11.loc["DL", "r"]), "s11_d_dl_lo": f2(d11.loc["DL", "lo"]), "s11_d_dl_hi": f2(d11.loc["DL", "hi"]),
    "s11_ret_n": f"{int(d11.loc[COVG11, 'n'] - VC11.loc['D, returners left out', 'n'])}",
    "s11_pred_t": f"{PRED11['T, coverage groups']:+.2f}", "s11_pred_db": f"{PRED11['D, defensive backs']:+.2f}",
    "s11_pred_wr": f"{PRED11['D, wide receivers']:+.2f}",
    "s11_pred_diff": f"{PRED11['D special teams minus D scrimmage, coverage groups']:+.2f}",
    **{f"s11_d_{p_.lower()}{sfx}": f2(d11.loc[p_, col]) for p_ in ("DB", "WR") for sfx, col in (("", "r"), ("_lo", "lo"), ("_hi", "hi"))},
    "s11_d_te_lo": f2(d11.loc["TE", "lo"]), "s11_d_te_hi": f2(d11.loc["TE", "hi"]),
    "s11_bar_cov": f2(MAIN11.loc[COVG11, ("bar D", "one test")]),
    **{f"s11_d_db{sfx}3": f"{d11.loc['DB', col]:.3f}" for sfx, col in (("", "r"), ("_lo", "lo"), ("_hi", "hi"))},
    "s11_d_wrdb3": f"{WRDB11[0]:.3f}", "s11_d_wrdb_lo3": f"{WRDB11[1]:.3f}", "s11_d_wrdb_hi3": f"{WRDB11[2]:.3f}",
    "s11_trk_wr": pc11(COV11.loc["WR", "game tracking / scrimmage"]),
    "s11_trk_te": pc11(COV11.loc["TE", "game tracking / scrimmage"]),
    "s11_var_lo": f2(VD11.r.min()), "s11_var_hi": f2(VD11.r.max()),
    "s11_snap_st_1pct": f"{SNAPCHK9.loc['special teams', 'within 1% or 1 snap']:.1%}",
})
json.dump(HEADLINE, open(os.path.join(WORK, "headline.json"), "w"), indent=1)
for k_, v_ in HEADLINE.items():
    print(f"{k_:22s} {v_}")
FROZEN = {'fly_rw': '0.948', 'fly_rw2': '0.95', 'fly_n': '418', 'fly_pos_lo': '0.85', 'fly_pos_hi': '0.99', 'f10_rw': '0.28', 'f10_rw2': '0.28', 'f10_pos_lo': '-0.12', 'f10_pos_hi': '0.49', 'f10_zero_groups': 'wide receivers, tight ends and defensive backs', 'wr_ten_sd': '0.026', 'wr_f10_diff_sd': '0.096', 'f10_rule_lo': '0.19', 'f10_rule_hi': '0.28', 'f10_pooled': '0.62', 'shuttle_rw': '0.65', 'cone_rw': '0.66', 'shuttle_timed': '79%', 'cone_timed': '76%', 'x_start_med': '10.67', 'reach_505': '98.9%', 'n_retest': '375', 'vmax_rel_min': '0.89', 'vmax_rel_max': '0.98', 'amax_rel_wr': '0.02', 'amax_rel_db': '0.26', 'amax_rel_dl': '0.65', 'dl_burst_min': '0.65', 'dl_burst_max': '0.92', 'ol_burst_min': '0.51', 'ol_burst_max': '0.84', 'wr_burst_min': '0.02', 'wr_burst_max': '0.34', 'db_burst_min': '0.20', 'db_burst_max': '0.63', 'noise_wr': '0.12', 'noise_ol': '0.13', 'between_wr': '0.10', 'between_ol': '0.29', 'hw_lo': '0.01', 'hw_hi': '0.33', 'relpair_min': '31', 'relpair_max': '93', 'n_prospects': '510', 'n_analysis': '243', 'n_min_pos': 'tight ends', 'n_min_pos_n': '18', 'n_max_pos': 'defensive backs', 'n_max_pos_n': '62', 'share_2023': '70%', 'share_2025': '50%', 'n_metrics': '18', 'one_test_lo': '0.25', 'one_test_hi': '0.47', 'power_lo': '0.35', 'power_hi': '0.62', 'perm_r_lo': '0.36', 'perm_r_hi': '0.68', 'n_clear_pairs': '28', 'n_clear_groups': '3', 'clear_burst_clause': '8 of them involve a burst or agility metric, all for the defensive line', 'clear_burst_clause_cap': '8 of them involve a burst or agility metric, all for the defensive line', 'clear_summary': 'wide receivers 8: 5 size, 3 speed; defensive backs 1: 1 size; the defensive line 19: 8 burst or agility, 3 jump or strength, 2 size, 6 speed', 'dl_wcorr_min': '0.37', 'dl_wcorr_max': '0.84', 'wr_size': '5', 'wr_speed': '3', 'cap_wr_amax': '0.18', 'cap_wr_amax_hi': '0.57', 'rel_getoff': '0.96', 'go_interior': '1.025', 'go_edge': '0.908', 'w_interior': '0.85', 'dl_n': '54', 'dl_raw_r': '0.67', 'dl_raw_lo': '0.50', 'dl_raw_hi': '0.81', 'dl_part_r': '0.35', 'dl_part_lo': '0.09', 'dl_part_hi': '0.56', 'dl_part2_r': '0.34', 'dl_part2_lo': '0.09', 'dl_part2_hi': '0.54', 'pa_drop': '100%', 'pa_nodrop': '0%', 'cone_wr': '0.09', 'cone_te': '-0.04', 'cone_wr_n': '15', 'cone_te_n': '14', 'dl_part_p': '0.011', 'w_go': '0.63', 'w_ten': '0.83', 'dl_shared': '84%', 'dl_pairs': '36', 'dl_bonf': '0.43', 'trk_part_lo': '0.04', 'trk_part_hi': '0.22', 'forty_raw': '0.68', 'forty_part': '0.36', 'pc_clear_names': 'wide receivers (r = 0.59) and the defensive line (r = 0.62)', 'pc_pooled': '0.87', 'pc_min': '0.14', 'pc_max': '0.62', 'pre_share': '16%', 'gt_ratio': '2.6', 'forty_dis_sum': '36.7', 'game_dis_step': '1.00', 'wr_n': '107', 'db_n': '122', 'wr_cone_n': '33', 'wr_shut_n': '38', 'db_cone_n': '25', 'db_shut_n': '33', 'wr_stop_n': '103', 'db_stop_n': '118', 'yg_max': '3.3', 'yg_where': 'comeback route right for wide receivers, 2023 vs 2025', 'cod_wr_r': '0.07', 'cod_wr_lo': '-0.07', 'cod_wr_hi': '0.20', 'cod_wr_pair_lo': '0.02', 'cod_wr_pair_hi': '0.17', 'cod_wr_n_lo': '92', 'cod_wr_n_hi': '98', 'cod_wr_pooled': '0.23', 'cod_wr_pooled_lo': '0.12', 'cod_wr_pooled_hi': '0.35', 'cod_db_r': '0.00', 'cod_db_lo': '-0.12', 'cod_db_hi': '0.13', 'cod_db_pair_lo': '-0.04', 'cod_db_pair_hi': '0.03', 'cod_dl_r': '0.04', 'cod_other_lo': '-0.01', 'cod_other_hi': '0.17', 'cod_best_what': 'speed gained after the stop for defensive backs', 'cod_best': '0.17', 'cod_best_lo': '0.02', 'cod_best_hi': '0.34', 'bench_lo': '0.62', 'bench_hi': '0.83', 'val_wr_shut': '-0.15', 'val_wr_shut_n': '36', 'val_db_shut': '-0.03', 'val_db_shut_n': '33', 'val_clause': 'all four intervals, the 3-cone included, contain zero', 'dl_mix_w': '-0.58', 't_n': '59', 't_p': '0.01', 't_p_lo': '-0.30', 't_p_hi': '0.29', 't_alt_lo': '-0.01', 't_alt_hi': '0.07', 't_crit': '0.26', 't_pow': '0.36', 't_ceil': '0.40', 't_ceil_hi': '0.60', 'sel_d': '0.16', 'sel_lo': '-0.09', 'sel_hi': '0.42', 'sel_sd_in': '0.73', 'sel_sd_all': '0.67', 'yc_fly': '0.952', 'yc_f10': '0.31', 'yc_vmax_change': '0.02', 'yc_wr_burst_lo': '-0.18', 'yc_wr_burst_hi': '0.25', 'yc_te_burst_lo': '-0.22', 'yc_te_burst_hi': '0.71', 'te_burst_min': '0.27', 'te_burst_max': '0.71', 'pcy_clause': 'defensive backs move most, from 0.24 to 0.29 (0.05 to 0.50), so the groups whose interval excludes zero become wide receivers, defensive backs and the defensive line, against wide receivers and the defensive line in version 1', 'j_found': '509', 'j_n': '510', 'j_class': '509', 'j_pick': '383', 'j_drafted': '383', 'snap_off_1pct': '97.7%', 'snap_off_n': '213', 'snap_def_1pct': '95.3%', 'snap_def_n': '193', 'starts_1g': '99.6%', 's9_n': '417', 's9_n_te': '34', 's9_n_dl': '93', 's9_pow_te': '0.49', 's9_t_wr': '0.18', 's9_t_wr_lo': '-0.02', 's9_t_wr_hi': '0.37', 's9_t_te': '-0.07', 's9_t_te_lo': '-0.39', 's9_t_te_hi': '0.25', 's9_t_db': '0.22', 's9_t_db_lo': '0.04', 's9_t_db_hi': '0.39', 's9_t_dl': '0.40', 's9_t_dl_lo': '0.20', 's9_t_dl_hi': '0.59', 's9_t_ol': '0.32', 's9_t_ol_lo': '0.12', 's9_t_ol_hi': '0.50', 's9_t_all': '0.25', 's9_t_all_lo': '0.16', 's9_t_all_hi': '0.34', 's9_d_all': '0.04', 's9_d_all_lo': '-0.06', 's9_d_all_hi': '0.14', 's9_d_four_lo': '-0.13', 's9_d_four_hi': '0.05', 's9_d_dl': '0.35', 's9_d_dl_lo': '0.12', 's9_d_dl_hi': '0.57', 's9_d_db': '-0.13', 's9_bonf_d_dl': '0.27', 's9_raw_dl': '0.13', 's9_t_max': '0.40', 's9_path1': '-0.30', 's9_path2': '-0.71', 's9_prod': '0.21', 's9_vt_lo': '0.21', 's9_vt_hi': '0.25', 's9_vd_lo': '-0.03', 's9_vd_hi': '0.04', 's9_drop': '0.21', 's9_drop_lo': '0.13', 's9_drop_hi': '0.28', 's9_q': '12.9', 's9_q_p': '0.012', 's9_o40_t': '0.28', 's9_o40_d': '0.01', 's9_dl_edge': '56%', 's9_dl_w2': '0.36', 's9_dl_w2_lo': '0.13', 's9_dl_w2_hi': '0.57', 's9_dl_edge_d': '0.31', 's9_dl_edge_lo': '0.05', 's9_dl_edge_hi': '0.53', 's9_cond_share': '0.19', 's9_any': '0.21', 's9_udfa_without': '30%', 's9_udfa_with': '24%', 's9_gap_all_lo': '0.03', 's9_gap_all_hi': '0.14', 's9_gap_round_lo': '-0.06', 's9_gap_round_hi': '0.03', 's9_dl_rank': '0.18', 's9_dl_rank_lo': '-0.03', 's9_dl_rank_hi': '0.39', 's9_dl_wv': '-0.83', 's9_dl_loo_lo': '0.26', 's9_dl_loo_hi': '0.40', 's9_dl_top_rank': 'the fastest top speed', 's9_dl_top_share': '85%', 's9_dround': '0.06', 's9_dround_lo': '-0.05', 's9_dround_hi': '0.17', 's9_gap_all': '0.08', 's9_gap_round': '-0.01', 's9_zero': '25%', 's9_cond_n': '317', 's9_cond_t': '0.16', 's9_sel_n': '92', 's9_sel_d': '0.02', 's9_sel_lo': '-0.05', 's9_sel_hi': '0.09', 's9_rd12_without': '24%', 's9_rd12_with': '18%', 's10_rep_wr': '0.07', 's10_rep_db': '0.13', 's10_rep_dl': '0.64', 's10_rep_ol': '0.57', 's10_n': '417', 's10_n_te': '34', 's10_pow_te': '0.51', 's10_t_all': '-0.13', 's10_t_all_lo': '-0.24', 's10_t_all_hi': '-0.03', 's10_d_all': '-0.05', 's10_d_all_lo': '-0.15', 's10_d_all_hi': '0.06', 's10_bar1': '0.10', 's10_bar20': '0.15', 's11_trk_all': '72%', 's11_cov_min3': '99.7%', 's11_cov_wr': '58%', 's11_cov_te': '40%', 's11_st_all': '17%', 's11_st_db': '25%', 's11_st_te': '23%', 's11_st_dl': '21%', 's11_trk_off': '209', 's11_trk_n': '314,944', 's11_rec_rows': '53,252', 's11_pred_d': '+0.20', 's11_n_cov': '229', 's11_pow_cov': '0.19', 's11_t_cov': '0.09', 's11_t_cov_lo': '-0.04', 's11_t_cov_hi': '0.21', 's11_t_line': '0.16', 's11_t_line_lo': '0.01', 's11_t_line_hi': '0.30', 's11_t_all': '0.12', 's11_t_all_lo': '0.03', 's11_t_all_hi': '0.21', 's11_d_cov': '0.01', 's11_d_cov_lo': '-0.13', 's11_d_cov_hi': '0.13', 's11_d_line': '0.03', 's11_d_line_lo': '-0.10', 's11_d_line_hi': '0.18', 's11_d_all': '0.02', 's11_d_all_lo': '-0.08', 's11_d_all_hi': '0.11', 's11_dt': '-0.08', 's11_dt_lo': '-0.15', 's11_dt_hi': '-0.02', 's11_diff': '0.09', 's11_diff_lo': '-0.12', 's11_diff_hi': '0.28', 's11_d_te': '-0.31', 's11_n_te': '34', 's11_bonf_te': '0.48', 's11_d_dl': '0.17', 's11_d_dl_lo': '-0.01', 's11_d_dl_hi': '0.35', 's11_ret_n': '32', 's11_pred_t': '+0.12', 's11_pred_db': '+0.25', 's11_pred_wr': '+0.15', 's11_pred_diff': '+0.15', 's11_d_db': '0.04', 's11_d_db_lo': '-0.18', 's11_d_db_hi': '0.25', 's11_d_wr': '0.09', 's11_d_wr_lo': '-0.13', 's11_d_wr_hi': '0.27', 's11_d_te_lo': '-0.62', 's11_d_te_hi': '0.05', 's11_bar_cov': '0.14', 's11_d_db3': '0.036', 's11_d_db_lo3': '-0.182', 's11_d_db_hi3': '0.248', 's11_d_wrdb3': '0.062', 's11_d_wrdb_lo3': '-0.089', 's11_d_wrdb_hi3': '0.197', 's11_trk_wr': '61%', 's11_trk_te': '42%', 's11_var_lo': '-0.08', 's11_var_hi': '0.04', 's11_snap_st_1pct': '97.5%'}   # the values written into the text above; None in a draft
if FROZEN is not None:
    bad = {k_: (FROZEN[k_], HEADLINE.get(k_)) for k_ in FROZEN if FROZEN[k_] != HEADLINE.get(k_)}
    print(f"\ntext vs this run: {len(FROZEN) - len(bad)}/{len(FROZEN)} values identical")
    assert not bad, bad
print(f"\ntotal runtime {time.time() - T0:.0f}s")